## 1. Setup and configuration


In [ ]:
import copy
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
import warnings
import textwrap
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, ScalarFormatter
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import brentq, minimize
from scipy.stats import gamma as gamma_dist
from scipy.stats import invgauss

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=RuntimeWarning)

MASTER_SEED = 2026
RUN_MODE = os.environ.get("ADT_RUN_MODE", "paper").strip().lower()
if RUN_MODE not in {"audit", "paper"}:
    raise ValueError("ADT_RUN_MODE must be 'audit' or 'paper'.")

AF_METHOD = 1
EA_RELATIVE_DEVIATION = float(
    os.environ.get("ADT_EA_RELATIVE_DEVIATION", "0.10")
)
if not (0.0 < EA_RELATIVE_DEVIATION < 1.0):
    raise ValueError("ADT_EA_RELATIVE_DEVIATION must lie in (0, 1).")

np.random.seed(MASTER_SEED)
random.seed(MASTER_SEED)

UNCERTAINTY_SEVERITY = float(os.environ.get("ADT_UNCERTAINTY_SEVERITY", "1.0"))
if UNCERTAINTY_SEVERITY < 0:
    raise ValueError("ADT_UNCERTAINTY_SEVERITY must be nonnegative.")

def _filename_number_tag(value):
    value = float(value)
    if np.isclose(value, round(value)):
        return str(int(round(value)))
    return f"{value:g}".replace(".", "p")


_DEFAULT_MISSION_YEARS = float(os.environ.get("ADT_MISSION_YEARS", "15.0"))
_DEFAULT_TMAX_ALLOW = float(os.environ.get("ADT_TMAX_ALLOW", "40000.0"))
DEFAULT_OUTPUT = (
    "adt_RRE_optimality_outputs_v11_"
    f"Ea{_filename_number_tag(100.0 * EA_RELATIVE_DEVIATION)}pct_"
    f"{_filename_number_tag(_DEFAULT_MISSION_YEARS)}yr_"
    f"AFtime_{_filename_number_tag(_DEFAULT_TMAX_ALLOW)}h_{RUN_MODE}"
)
OUT_DIR = Path(os.environ.get("ADT_OUTPUT_DIR", DEFAULT_OUTPUT))
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("default")

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8.6,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.2,
    "axes.titleweight": "regular",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.75,
    "ytick.major.width": 0.75,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "legend.fontsize": 7.5,
    "legend.frameon": False,
    "lines.linewidth": 1.35,
    "lines.markersize": 4.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

PALETTE = {
    "navy": "#2B4C7E",
    "blue": "#4C78A8",
    "teal": "#2A9D8F",
    "green": "#4C956C",
    "amber": "#D19A3E",
    "orange": "#D97742",
    "red": "#A63D40",
    "wine": "#7D3C59",
    "purple": "#7A6FA8",
    "slate": "#66788A",
    "gray": "#7C858D",
    "light_gray": "#D9DEE3",
    "very_light_gray": "#EEF1F4",
    "black": "#202428",
}

METHOD_ORDER = [
    "Proposed RRE-optimality",
    "V-optimality",
    "Cost-matched V-optimality",
    "D-optimality",
    "A-optimality",
    "M-inspired AF-slope precision",
    "Conventional equal-allocation",
    "Literature Tmin plan",
]

METHOD_STYLE = {
    "Proposed RRE-optimality": {
        "color": PALETTE["red"], "marker": "o", "linestyle": "-", "label": r"RRE-optimality"
    },
    "V-optimality": {
        "color": PALETTE["navy"], "marker": "s", "linestyle": "--", "label": "V-optimality"
    },
    "Cost-matched V-optimality": {
        "color": PALETTE["teal"], "marker": "^", "linestyle": "-.", "label": "Cost-matched V"
    },
    "D-optimality": {
        "color": PALETTE["purple"], "marker": "D", "linestyle": ":", "label": "D-optimality"
    },
    "A-optimality": {
        "color": PALETTE["amber"], "marker": "P", "linestyle": (0, (3, 1, 1, 1)), "label": "A-optimality"
    },
    "M-inspired AF-slope precision": {
        "color": PALETTE["slate"], "marker": "X", "linestyle": (0, (5, 2)), "label": "M-inspired"
    },
    "Conventional equal-allocation": {
        "color": PALETTE["green"], "marker": "h", "linestyle": (0, (4, 2)),
        "label": "Equal allocation"
    },
    "Literature Tmin plan": {
        "color": PALETTE["gray"], "marker": "v", "linestyle": (0, (1, 1)),
        "label": r"Literature $T_{\min}$"
    },
}

TEMPERATURE_COLORS = {
    65.0: PALETTE["blue"],
    85.0: PALETTE["teal"],
    100.0: PALETTE["red"],
}


def env_int(name, default):
    return int(os.environ.get(name, str(default)))


def env_float(name, default):
    return float(os.environ.get(name, str(default)))


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def stable_seed(*parts):
    text = "|".join(str(v) for v in parts)
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:8], 16)


def _safe_tight_layout(fig):
    try:
        fig.tight_layout(pad=0.8)
    except Exception:
        pass


def save_figure(fig, name):
    _safe_tight_layout(fig)
    common = dict(bbox_inches="tight", facecolor="white", edgecolor="white")
    fig.savefig(OUT_DIR / f"{name}.pdf", **common)
    fig.savefig(OUT_DIR / f"{name}.png", dpi=600, **common)
    try:
        fig.savefig(
            OUT_DIR / f"{name}.tiff",
            dpi=600,
            pil_kwargs={"compression": "tiff_lzw"},
            **common,
        )
    except Exception:
        fig.savefig(OUT_DIR / f"{name}.tiff", dpi=600, **common)
    plt.close(fig)


def panel_label(ax, label, x=-0.14, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=9.0, fontweight="bold",
        color=PALETTE["black"],
        clip_on=False,
    )


def style_axis(ax, xlabel=None, ylabel=None, title=None, grid="y"):
    if xlabel:
        ax.set_xlabel(xlabel, labelpad=4)
    if ylabel:
        ax.set_ylabel(ylabel, labelpad=4)
    if title:
        ax.set_title(title, pad=5)
    ax.tick_params(which="both", direction="in", width=0.75)
    ax.spines["left"].set_color(PALETTE["black"])
    ax.spines["bottom"].set_color(PALETTE["black"])
    if grid == "y":
        ax.grid(axis="y", color=PALETTE["light_gray"], linewidth=0.55, alpha=0.65)
    elif grid == "x":
        ax.grid(axis="x", color=PALETTE["light_gray"], linewidth=0.55, alpha=0.65)
    elif grid == "both":
        ax.grid(True, color=PALETTE["light_gray"], linewidth=0.5, alpha=0.55)
    else:
        ax.grid(False)


def wrapped_label(text, width=18):
    return textwrap.fill(str(text), width=width)


def ordered_methods(values):
    values = list(dict.fromkeys(str(v) for v in values))
    known = [m for m in METHOD_ORDER if m in values]
    unknown = [m for m in values if m not in known]
    return known + unknown


def method_style(method):
    return METHOD_STYLE.get(
        str(method),
        {"color": PALETTE["gray"], "marker": "o", "linestyle": "-", "label": str(method)},
    )


@dataclass
class DesignConfig:
    T0: float = 40.0
    Tmax: float = 100.0
    r: int = 3
    Tmin_lb: int = 40
    Tmin_ub: int = 80
    N_lb: int = 60
    N_ub: int = 60
    m_lb: int = 12
    m_ub: int = 12
    min_units_per_stress: int = 3

    af_method: int = AF_METHOD
    ea_relative_deviation: float = EA_RELATIVE_DEVIATION

    delta_T_min: float = 1.0
    AF_min: float = 1.0
    tmax_allow: float = env_float("ADT_TMAX_ALLOW", 40000.0)
    observe_ratio: float = 0.80
    YD: float = 30.0
    censor_at_failure: bool = True

    c_sample: float = 100.0
    c_measure: float = 2.0
    c_stress_base: float = 0.0025
    comparison_budget: float = env_float("ADT_BUDGET", 12000.0)
    budget_tolerance: float = 1e-8
    matched_cost_lower_ratio: float = 0.0
    matched_cost_tolerance_ratio: float = 0.02

    robust_cvar_alpha: float = 0.80
    robust_cvar_weight: float = 0.25
    mission_years: float = env_float("ADT_MISSION_YEARS", 15.0)
    hours_per_year: float = 365.0 * 24.0
    mission_time_h: float = float("nan")
    primary_target: str = "life_q10"
    fit_mode: str = "full"
    uncertainty_severity: float = UNCERTAINTY_SEVERITY

    candidate_count: int = env_int(
        "ADT_CANDIDATE_COUNT", 220 if RUN_MODE == "audit" else 2200
    )
    shortlist_count: int = env_int(
        "ADT_SHORTLIST_COUNT", 20 if RUN_MODE == "audit" else 120
    )
    rre_screen_reps: int = int(os.environ.get(
        "ADT_RRE_SCREEN_REPS",
        os.environ.get(
            "ADT_VR_SCREEN_REPS",
            os.environ.get("ADT_RTE_SCREEN_REPS", 1 if RUN_MODE == "audit" else 12),
        ),
    ))
    rre_refine_reps: int = int(os.environ.get(
        "ADT_RRE_REFINE_REPS",
        os.environ.get(
            "ADT_VR_REFINE_REPS",
            os.environ.get("ADT_RTE_REFINE_REPS", 5 if RUN_MODE == "audit" else 100),
        ),
    ))
    refine_top_k: int = env_int(
        "ADT_REFINE_TOP_K", 3 if RUN_MODE == "audit" else 8
    )
    confirmation_reps: int = env_int(
        "ADT_CONFIRMATION_REPS", 8 if RUN_MODE == "audit" else 250
    )
    confirmation_top_k: int = env_int(
        "ADT_CONFIRMATION_TOP_K", 3 if RUN_MODE == "audit" else 5
    )
    validation_reps: int = env_int(
        "ADT_VALIDATION_REPS", 6 if RUN_MODE == "audit" else 500
    )
    bootstrap_reps: int = env_int(
        "ADT_BOOTSTRAP_REPS", 100 if RUN_MODE == "audit" else 3000
    )
    severity_diagnostic_reps: int = env_int(
        "ADT_SEVERITY_REPS", 2 if RUN_MODE == "audit" else 80
    )
    minimum_valid_fit_fraction: float = 0.80 if RUN_MODE == "audit" else 0.95

    def __post_init__(self):
        if self.af_method != 1:
            raise ValueError(
                "V11 revised comparison uses the real-data fitted acceleration relation "
                "as the true/reference model; af_method must be 1."
            )
        if not (0.0 < self.ea_relative_deviation < 1.0):
            raise ValueError("ea_relative_deviation must lie in (0,1).")
        if not (self.mission_years > 0):
            raise ValueError("mission_years must be positive.")
        if not np.isfinite(self.mission_time_h):
            self.mission_time_h = float(self.mission_years * self.hours_per_year)
        if not (self.mission_time_h > 0):
            raise ValueError("mission_time_h must be positive.")
        if not (0 < self.robust_cvar_alpha < 1):
            raise ValueError("robust_cvar_alpha must be in (0,1).")
        if not (0 <= self.robust_cvar_weight <= 1):
            raise ValueError("robust_cvar_weight must be in [0,1].")
        if self.r != 3:
            raise ValueError("This implementation is predeclared for three stress groups.")
        if self.N_lb < self.r * self.min_units_per_stress:
            raise ValueError("N_lb is incompatible with the minimum group allocation.")
        if self.Tmin_lb > self.Tmin_ub:
            raise ValueError("Empty Tmin domain.")
        if self.N_lb != self.N_ub or self.m_lb != self.m_ub:
            raise ValueError(
                "V11 primary comparison requires fixed N and m. Use a separate resource-planning analysis "
                "if joint resource optimization is required."
            )
        if not (0 < self.minimum_valid_fit_fraction <= 1):
            raise ValueError("minimum_valid_fit_fraction must be in (0,1].")
        if not (0 < self.matched_cost_tolerance_ratio < 0.25):
            raise ValueError("matched_cost_tolerance_ratio must be in (0,0.25).")


CFG = DesignConfig()


def build_scenarios(severity=1.0):
    z = float(severity)
    delta = float(CFG.ea_relative_deviation) * z
    ea_low = float(EA_DATA_FITTED_EV * (1.0 - delta))
    ea_high = float(EA_DATA_FITTED_EV * (1.0 + delta))
    return [
        {"Scenario": "True IG model", "Category": "True model", "kind": "true_model",
         "process": "ig", "weight": 0.20, "role": "primary"},
        {"Scenario": f"Fitted Ea - {100*CFG.ea_relative_deviation:.0f}%",
         "Category": "Activation energy", "kind": "ea_multiplier",
         "activation_energy_eV": ea_low, "ea_multiplier": 1.0 - delta,
         "process": "ig", "weight": 0.075, "role": "primary"},
        {"Scenario": f"Fitted Ea + {100*CFG.ea_relative_deviation:.0f}%",
         "Category": "Activation energy", "kind": "ea_multiplier",
         "activation_energy_eV": ea_high, "ea_multiplier": 1.0 + delta,
         "process": "ig", "weight": 0.075, "role": "primary"},
        {"Scenario": "High-T slope flattening", "Category": "Mechanism change",
         "kind": "piecewise", "switch_T": 85.0,
         "high_slope_multiplier": 1.0 + z * (0.50 - 1.0),
         "high_eta_multiplier": 1.0 + z * (0.70 - 1.0),
         "process": "ig", "weight": 0.125, "role": "primary"},
        {"Scenario": "High-T slope steepening", "Category": "Mechanism change",
         "kind": "piecewise", "switch_T": 85.0,
         "high_slope_multiplier": 1.0 + z * (1.75 - 1.0),
         "high_eta_multiplier": 1.0 + z * (0.70 - 1.0),
         "process": "ig", "weight": 0.125, "role": "primary"},
        {"Scenario": "Stress-dependent q (-20%)", "Category": "Time scale",
         "kind": "q_shift", "q_shift": -0.20 * z,
         "process": "ig", "weight": 0.10, "role": "primary"},
        {"Scenario": "Stress-dependent q (+20%)", "Category": "Time scale",
         "kind": "q_shift", "q_shift": 0.20 * z,
         "process": "ig", "weight": 0.10, "role": "primary"},
        {"Scenario": "Concave acceleration", "Category": "Acceleration curvature",
         "kind": "quadratic", "c_quad": -0.60 * z,
         "process": "ig", "weight": 0.10, "role": "primary"},
        {"Scenario": "Convex acceleration", "Category": "Acceleration curvature",
         "kind": "quadratic", "c_quad": 0.60 * z,
         "process": "ig", "weight": 0.10, "role": "primary"},
        {"Scenario": "Gamma increments / IG fit", "Category": "Process family",
         "kind": "gamma", "process": "gamma", "weight": 0.0,
         "role": "external_stress_test"},
    ]


SCENARIOS = None


def normalize_weights(values):
    arr = np.asarray(values, dtype=float)
    if np.any(arr < 0) or not np.isfinite(arr).all() or arr.sum() <= 0:
        raise ValueError("Scenario weights must be finite, nonnegative and nonzero.")
    return arr / arr.sum()


SCENARIO_WEIGHTS = None


## 2. Connector observations


In [ ]:
def load_connector_relaxation_data():
    return {
        65.0: {
            "times": np.array([108, 241, 534, 839, 1074, 1350, 1637, 1890, 2178, 2513, 2810], float),
            "Y": np.array([
                [2.12, 2.70, 3.52, 4.25, 5.55, 6.12, 6.75, 7.22, 7.68, 8.46, 9.46],
                [2.29, 3.24, 4.16, 4.86, 5.74, 6.85, 7.12, 7.40, 8.14, 9.25, 10.55],
                [2.40, 3.61, 4.35, 5.09, 5.50, 7.03, 8.24, 8.81, 9.629, 10.27, 11.11],
                [2.31, 3.48, 5.51, 6.20, 7.31, 7.96, 8.57, 9.07, 10.46, 11.48, 12.30],
                [3.14, 4.33, 5.92, 7.22, 8.14, 9.07, 9.44, 10.09, 11.20, 12.77, 13.51],
                [3.59, 5.55, 5.92, 7.68, 8.61, 10.37, 11.11, 12.22, 13.51, 14.16, 15.00],
            ], float),
        },
        85.0: {
            "times": np.array([46, 108, 212, 408, 632, 764, 1011, 1333, 1517, 2586], float),
            "Y": np.array([
                [2.77, 4.62, 5.83, 6.66, 8.05, 10.61, 11.20, 11.98, 13.33, 15.64],
                [3.88, 4.37, 6.29, 7.77, 9.16, 9.90, 10.37, 12.77, 14.72, 16.80],
                [3.18, 4.53, 6.94, 8.14, 8.79, 10.09, 11.11, 14.72, 16.47, 18.66],
                [3.61, 4.37, 6.29, 7.87, 9.35, 11.48, 12.40, 13.70, 15.37, 18.51],
                [3.42, 4.25, 7.31, 8.61, 10.18, 12.03, 13.70, 15.27, 17.22, 19.25],
                [5.27, 5.92, 8.05, 9.81, 12.40, 13.24, 15.83, 17.59, 20.09, 23.51],
            ], float),
        },
        100.0: {
            "times": np.array([46, 108, 212, 344, 446, 626, 729, 972, 1005, 1218], float),
            "Y": np.array([
                [3.88, 5.09, 7.22, 8.79, 10.37, 12.68, 14.81, 16.38, 17.77, 19.53],
                [4.07, 6.20, 7.50, 9.16, 11.20, 13.14, 15.09, 17.22, 18.88, 20.64],
                [4.35, 6.66, 8.79, 10.64, 12.40, 14.72, 16.57, 18.79, 20.92, 22.40],
                [4.62, 7.03, 9.35, 11.11, 12.87, 15.18, 17.22, 19.53, 21.57, 23.14],
                [5.09, 7.31, 9.81, 11.98, 14.07, 16.20, 18.70, 20.92, 22.77, 24.62],
                [5.55, 8.05, 10.55, 12.96, 15.00, 17.77, 20.09, 22.96, 24.72, 27.22],
            ], float),
        },
    }


REAL_DATA = load_connector_relaxation_data()


def plot_connector_data():
    fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.35), sharey=True)
    temperatures = sorted(REAL_DATA)
    for idx, (ax, temperature) in enumerate(zip(axes, temperatures)):
        item = REAL_DATA[temperature]
        accent = TEMPERATURE_COLORS.get(float(temperature), PALETTE["blue"])
        times = np.asarray(item["times"], float)
        paths = np.asarray(item["Y"], float)

        for path in paths:
            ax.plot(
                times, path,
                color=PALETTE["gray"],
                marker="o", markersize=2.2,
                markerfacecolor="white", markeredgewidth=0.55,
                linewidth=0.70, alpha=0.62,
                zorder=1,
            )

        mean_path = paths.mean(axis=0)
        ax.plot(
            times, mean_path,
            color=accent,
            marker="o", markersize=3.2,
            markerfacecolor="white", markeredgewidth=0.9,
            linewidth=1.8,
            zorder=3,
        )
        ax.axhline(
            CFG.YD,
            color=PALETTE["black"],
            linestyle=(0, (4, 2)),
            linewidth=0.9,
            alpha=0.75,
            zorder=0,
        )
        ax.set_ylim(0, max(CFG.YD * 1.05, float(paths.max()) * 1.08))
        style_axis(
            ax,
            "Time (h)",
            "Stress relaxation (%)" if idx == 0 else None,
            rf"{temperature:.0f} $^\circ$C",
            grid="y",
        )
        panel_label(ax, f"({chr(97 + idx)})", x=-0.12, y=1.03)

    legend_handles = [
        Line2D([0], [0], color=PALETTE["gray"], marker="o", markerfacecolor="white",
               markersize=3.5, linewidth=0.8, label="Individual units"),
        Line2D([0], [0], color=PALETTE["navy"], marker="o", markerfacecolor="white",
               markersize=4.0, linewidth=1.8, label="Mean path"),
        Line2D([0], [0], color=PALETTE["black"], linestyle=(0, (4, 2)),
               linewidth=0.9, label="Failure threshold"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.04),
        ncol=3,
        columnspacing=1.5,
        handlelength=2.2,
    )
    fig.subplots_adjust(top=0.82, wspace=0.16)
    save_figure(fig, "Fig01_connector_data")


## 3. IG model and calibration


In [ ]:
BOLTZMANN_EV_PER_K = 8.617333262145e-5
EV_TO_J_PER_MOL = 96485.33212331002


def arrhenius_x(T, T0):
    T = np.asarray(T, dtype=float)
    return 1.0 / (T0 + 273.15) - 1.0 / (T + 273.15)


def standardized_stress(T, T0, Tmax):
    return arrhenius_x(T, T0) / float(arrhenius_x(Tmax, T0))


def activation_energy_to_eV(value, unit="eV"):
    unit_key = str(unit).lower().replace(" ", "")
    if unit_key == "ev":
        answer = float(value)
    elif unit_key == "kj/mol":
        answer = float(value) * 1000.0 / EV_TO_J_PER_MOL
    elif unit_key == "j/mol":
        answer = float(value) / EV_TO_J_PER_MOL
    else:
        raise ValueError("Activation-energy unit must be eV, kJ/mol, or J/mol.")
    if not np.isfinite(answer) or answer <= 0:
        raise ValueError("Activation energy must be positive and finite.")
    return answer


def activation_energy_to_alpha1(value, cfg=CFG, unit="eV"):
    return float(
        activation_energy_to_eV(value, unit)
        / BOLTZMANN_EV_PER_K
        * arrhenius_x(cfg.Tmax, cfg.T0)
    )


def alpha1_to_activation_energy_eV(alpha1, cfg=CFG):
    return float(
        float(alpha1)
        * BOLTZMANN_EV_PER_K
        / float(arrhenius_x(cfg.Tmax, cfg.T0))
    )


def stress_levels(Tmin, cfg=CFG):
    value = int(np.rint(Tmin))
    reciprocal = np.linspace(
        1.0 / (value + 273.15), 1.0 / (cfg.Tmax + 273.15), cfg.r
    )
    return 1.0 / reciprocal - 273.15


def beta_model(T, alpha0, alpha1, cfg=CFG, c_quad=0.0):
    s = standardized_stress(T, cfg.T0, cfg.Tmax)
    return np.exp(np.clip(alpha0 + alpha1 * s + c_quad * s**2, -700, 700))


def ig_logpdf(y, mean, shape):
    y = np.maximum(np.asarray(y, float), 1e-300)
    mean = np.maximum(np.asarray(mean, float), 1e-300)
    shape = np.maximum(np.asarray(shape, float), 1e-300)
    return (
        0.5 * (np.log(shape) - np.log(2 * np.pi) - 3 * np.log(y))
        - shape * (y - mean) ** 2 / (2 * mean**2 * y)
    )


def build_increment_dataset(data):
    Ts, lefts, rights, dys = [], [], [], []
    for temperature, item in sorted(data.items()):
        times = np.r_[0.0, np.asarray(item["times"], float)]
        for path in np.asarray(item["Y"], float):
            increments = np.diff(np.r_[0.0, path])
            valid = increments > 0
            Ts.extend([float(temperature)] * int(valid.sum()))
            lefts.extend(times[:-1][valid])
            rights.extend(times[1:][valid])
            dys.extend(increments[valid])
    return tuple(np.asarray(x, float) for x in (Ts, lefts, rights, dys))


def raw_to_theta(raw):
    raw = np.asarray(raw, float)
    return {
        "alpha0": float(raw[0]), "alpha1": float(raw[1]),
        "eta": float(np.exp(raw[2])), "q": float(np.exp(raw[3])),
    }


def theta_to_raw(theta):
    return np.array([
        theta["alpha0"], theta["alpha1"],
        np.log(theta["eta"]), np.log(theta["q"]),
    ], float)


def neg_loglik_raw(raw, T, left, right, dy, cfg=CFG):
    theta = raw_to_theta(raw)
    A = np.asarray(right, float) ** theta["q"] - np.asarray(left, float) ** theta["q"]
    if np.any(A <= 0) or not np.isfinite(A).all():
        return 1e100
    beta = beta_model(T, theta["alpha0"], theta["alpha1"], cfg)
    value = -np.sum(ig_logpdf(dy, beta * A, theta["eta"] * A**2))
    return float(value) if np.isfinite(value) else 1e100


def regression_start(T, left, right, dy, q0, cfg=CFG):
    A = np.asarray(right, float) ** q0 - np.asarray(left, float) ** q0
    s = standardized_stress(T, cfg.T0, cfg.Tmax)
    X = np.column_stack([np.ones_like(s), s])
    coef, *_ = np.linalg.lstsq(X, np.log(np.maximum(dy / A, 1e-12)), rcond=None)
    return np.array([coef[0], coef[1], np.log(0.6), np.log(q0)], float)


def fit_assumed_ig(data, cfg=CFG, init=None, multistart=False):
    T, left, right, dy = (np.asarray(v, float) for v in data)
    if len(dy) < 8 or np.any(dy <= 0):
        return {"success": False, "valid": False, "nll": np.inf}
    if init is None:
        starts = [regression_start(T, left, right, dy, q0, cfg) for q0 in (0.30, 0.45, 0.70, 1.0)]
    elif multistart:
        starts = [np.asarray(init, float)] + [regression_start(T, left, right, dy, q0, cfg) for q0 in (0.35, 0.70)]
    else:
        starts = [np.asarray(init, float)]
    bounds = [(-20, 10), (-10, 15), (-20, 10), (math.log(0.05), math.log(3.0))]
    candidates = []
    for start in starts:
        result = minimize(
            neg_loglik_raw, start, args=(T, left, right, dy, cfg),
            method="L-BFGS-B", bounds=bounds,
            options={"maxiter": 160, "ftol": 1e-10},
        )
        candidates.append(result)
    result = min(candidates, key=lambda item: item.fun)
    theta = raw_to_theta(result.x)
    interior = all(
        lower + 1e-6 < value < upper - 1e-6
        for value, (lower, upper) in zip(result.x, bounds)
    )
    valid = bool(np.isfinite(result.fun) and all(np.isfinite(list(theta.values()))) and interior)
    return {**theta, "raw": np.asarray(result.x, float), "success": bool(result.success),
            "valid": valid, "nll": float(result.fun), "n_increments": int(len(dy))}


def fit_assumed_ig_with_fallback(data, cfg=CFG, init=None):
    fit = fit_assumed_ig(data, cfg, init=init, multistart=False)
    fallback_used = not bool(fit.get("valid", False))
    if fallback_used:
        fit = fit_assumed_ig(data, cfg, init=init, multistart=True)
    fit["multistart_fallback_used"] = bool(fallback_used)
    return fit


def failure_cdf_ig(t, beta, eta, q, threshold):
    A = max(float(t), 1e-12) ** float(q)
    mean = float(beta) * A
    shape = float(eta) * A**2
    return float(invgauss.sf(float(threshold), mu=mean / shape, scale=shape))


def lifetime_quantile_ig(beta, eta, q, threshold, p):
    if not (0 < p < 1):
        raise ValueError("p must be in (0,1).")
    drift = (float(threshold) / float(beta)) ** (1.0 / float(q))
    lo, hi = max(drift * 1e-5, 1e-9), drift * 10
    root = lambda t: failure_cdf_ig(t, beta, eta, q, threshold) - p
    while root(lo) > 0:
        lo *= 0.1
    while root(hi) < 0:
        hi *= 2
    return float(brentq(root, lo, hi, xtol=1e-8, rtol=1e-10, maxiter=250))


T_REAL, LEFT_REAL, RIGHT_REAL, DY_REAL = build_increment_dataset(REAL_DATA)
REAL_FIT = fit_assumed_ig((T_REAL, LEFT_REAL, RIGHT_REAL, DY_REAL), CFG, init=None)
if not REAL_FIT["valid"]:
    raise RuntimeError("The real-data IG calibration did not produce a valid interior estimate.")

ALPHA1_FITTED = float(REAL_FIT["alpha1"])
EA_DATA_FITTED_EV = alpha1_to_activation_energy_eV(ALPHA1_FITTED, CFG)
EA_DATA_LOWER_EV = EA_DATA_FITTED_EV * (1.0 - CFG.ea_relative_deviation)
EA_DATA_UPPER_EV = EA_DATA_FITTED_EV * (1.0 + CFG.ea_relative_deviation)

ALPHA1_SELECTED = ALPHA1_FITTED
TRUE_CTX = {
    "alpha0": REAL_FIT["alpha0"],
    "alpha1": ALPHA1_SELECTED,
    "alpha1_data_fitted": ALPHA1_FITTED,
    "activation_energy_data_fitted_eV": EA_DATA_FITTED_EV,
    "eta": REAL_FIT["eta"],
    "q": REAL_FIT["q"],
    "D": CFG.YD,
}
TRUE_RAW = theta_to_raw(TRUE_CTX)

_beta_use = math.exp(float(TRUE_CTX["alpha0"]))
B10_TRUE_H = lifetime_quantile_ig(
    _beta_use, TRUE_CTX["eta"], TRUE_CTX["q"], CFG.YD, 0.10
)

SCENARIOS = build_scenarios(CFG.uncertainty_severity)
SCENARIO_WEIGHTS = normalize_weights([s["weight"] for s in SCENARIOS])


def selected_beta_ratio(T, cfg=CFG, ctx=TRUE_CTX):
    return np.exp(
        float(ctx["alpha1_data_fitted"])
        * standardized_stress(T, cfg.T0, cfg.Tmax)
    )


def selected_time_acceleration_factor(T, cfg=CFG, ctx=TRUE_CTX):
    beta_ratio = selected_beta_ratio(T, cfg, ctx)
    return np.asarray(beta_ratio, float) ** (1.0 / float(ctx["q"]))


def selected_acceleration_factor(T, cfg=CFG, ctx=TRUE_CTX):
    return selected_time_acceleration_factor(T, cfg, ctx)


## 4. Design constraints and costs


In [ ]:
def allocate_units(N, proportions, cfg=CFG):
    N = int(N)
    minimum = int(cfg.min_units_per_stress)
    remainder = N - cfg.r * minimum
    if remainder < 0:
        raise ValueError("N is too small for the minimum allocation.")
    p = np.maximum(np.asarray(proportions, float), 0)
    if p.size != cfg.r or p.sum() <= 0:
        p = np.ones(cfg.r, float)
    p /= p.sum()
    raw = remainder * p
    extra = np.floor(raw).astype(int)
    for index in np.argsort(-(raw - extra))[: remainder - int(extra.sum())]:
        extra[index] += 1
    return extra + minimum


def canonicalize_design(design, cfg=CFG):
    Tmin = int(np.rint(float(design["Tmin"])))
    N = int(np.rint(float(design["N"])))
    m = int(np.rint(float(design["m"])))
    if "Ni" in design:
        Ni_input = np.asarray(design["Ni"], int)
        if Ni_input.size != cfg.r or Ni_input.sum() != N or np.any(Ni_input < cfg.min_units_per_stress):
            Ni = allocate_units(N, np.maximum(Ni_input, 0), cfg)
        else:
            Ni = Ni_input.copy()
    else:
        Ni = allocate_units(N, design.get("pi", np.ones(cfg.r)), cfg)
    return {"Tmin": Tmin, "N": N, "m": m, "Ni": Ni}


def design_id(design, cfg=CFG):
    d = canonicalize_design(design, cfg)
    return f"T{d['Tmin']}_N{d['N']}_m{d['m']}_a{'-'.join(map(str, d['Ni']))}"


def planned_schedule(design, cfg=CFG, ctx=TRUE_CTX):
    d = canonicalize_design(design, cfg)
    levels = stress_levels(d["Tmin"], cfg)
    beta0 = math.exp(float(ctx["alpha0"]))
    use_mean_crossing_h = (cfg.YD / beta0) ** (1.0 / float(ctx["q"]))
    af_time = np.asarray(selected_time_acceleration_factor(levels, cfg, ctx), float)
    durations = cfg.observe_ratio * use_mean_crossing_h / np.maximum(af_time, 1e-15)
    schedules = [np.linspace(0, float(duration), d["m"] + 1) for duration in durations]
    return d, levels, durations, schedules


def compute_total_cost(design, cfg=CFG, ctx=TRUE_CTX):
    d, levels, durations, _ = planned_schedule(design, cfg, ctx)
    ranks = np.argsort(np.argsort(levels))
    stress_rates = cfg.c_stress_base * (2.0 ** ranks)
    sample_cost = cfg.c_sample * d["N"]
    measurement_cost = cfg.c_measure * d["N"] * d["m"]
    running_cost = float(np.sum(d["Ni"] * stress_rates * durations))
    return {
        "C_total": float(sample_cost + measurement_cost + running_cost),
        "C_sample": float(sample_cost),
        "C_measure": float(measurement_cost),
        "C_stress_run": running_cost,
        "t_campaign": float(np.max(durations)),
        "t_campaign_parallel_h": float(np.max(durations)),
        "t_sum_group_durations_h": float(np.sum(durations)),
        "total_unit_hours": float(np.dot(d["Ni"], durations)),
        "durations": np.asarray(durations, float),
        "T_levels": np.asarray(levels, float),
    }


def constraint_report(design, cfg=CFG, ctx=TRUE_CTX):
    try:
        d = canonicalize_design(design, cfg)
    except Exception as exc:
        return {"feasible": False, "reason": str(exc), "violation": np.inf}
    violations = {}
    violations["Tmin_domain"] = max(cfg.Tmin_lb - d["Tmin"], 0, d["Tmin"] - cfg.Tmin_ub)
    violations["N_domain"] = max(cfg.N_lb - d["N"], 0, d["N"] - cfg.N_ub)
    violations["m_domain"] = max(cfg.m_lb - d["m"], 0, d["m"] - cfg.m_ub)
    violations["allocation"] = 0.0 if (
        d["Ni"].sum() == d["N"] and np.all(d["Ni"] >= cfg.min_units_per_stress)
    ) else 1.0
    levels = stress_levels(d["Tmin"], cfg)
    violations["stress_gap"] = max(cfg.delta_T_min - float(np.min(np.diff(levels))), 0)
    beta_ratio = float(selected_beta_ratio(d["Tmin"], cfg, ctx))
    af_time = float(selected_time_acceleration_factor(d["Tmin"], cfg, ctx))
    violations["AF"] = max(cfg.AF_min - af_time, 0)
    costs = compute_total_cost(d, cfg, ctx)
    violations["time"] = max(costs["t_campaign"] - cfg.tmax_allow, 0)
    violations["budget"] = max(costs["C_total"] - cfg.comparison_budget - cfg.budget_tolerance, 0)
    total = float(sum(violations.values()))
    return {
        "feasible": total <= 0,
        "violation": total,
        "components": violations,
        "Beta_ratio_Tmin": beta_ratio,
        "AFt_Tmin": af_time,
        "AF_Tmin": af_time,
        **costs,
    }


def is_feasible(design, cfg=CFG, ctx=TRUE_CTX):
    return bool(constraint_report(design, cfg, ctx)["feasible"])


def random_feasible_design(rng, cfg=CFG, ctx=TRUE_CTX, near_budget=False, fixed_Tmin=None):
    for _ in range(40000):
        Tmin = int(fixed_Tmin) if fixed_Tmin is not None else int(rng.integers(cfg.Tmin_lb, cfg.Tmin_ub + 1))
        design = canonicalize_design({
            "Tmin": Tmin,
            "N": int(rng.integers(cfg.N_lb, cfg.N_ub + 1)),
            "m": int(rng.integers(cfg.m_lb, cfg.m_ub + 1)),
            "pi": rng.dirichlet(np.ones(cfg.r)),
        }, cfg)
        if not is_feasible(design, cfg, ctx):
            continue
        if near_budget:
            cost = compute_total_cost(design, cfg, ctx)["C_total"]
            if cost < cfg.matched_cost_lower_ratio * cfg.comparison_budget:
                continue
        return design
    raise RuntimeError(f"Unable to generate a feasible design; fixed_Tmin={fixed_Tmin}.")


def tmin_feasibility_table(cfg=CFG, ctx=TRUE_CTX):
    rows = []
    probe_N = cfg.N_lb
    for Tmin in range(cfg.Tmin_lb, cfg.Tmin_ub + 1):
        probes = []
        for m in range(cfg.m_lb, cfg.m_ub + 1):
            d = canonicalize_design({"Tmin": Tmin, "N": probe_N, "m": m, "pi": np.ones(cfg.r)}, cfg)
            report = constraint_report(d, cfg, ctx)
            probes.append((report["violation"], d, report))
        _, best, report = min(probes, key=lambda item: item[0])
        rows.append({
            "Tmin": Tmin,
            "feasible": report["feasible"],
            "Beta_ratio_Tmin": report["Beta_ratio_Tmin"],
            "AFt_Tmin": report["AFt_Tmin"],
            "minimum_probe_cost": report["C_total"],
            "campaign_time_h": report["t_campaign"],
            "parallel_campaign_h": report["t_campaign_parallel_h"],
            "sum_group_durations_h": report["t_sum_group_durations_h"],
            "total_unit_hours": report["total_unit_hours"],
            "probe_N": best["N"], "probe_m": best["m"],
            "binding_time": report["t_campaign"] >= cfg.tmax_allow - 1e-8,
            "binding_budget": report["C_total"] >= cfg.comparison_budget - 1e-8,
        })
    return pd.DataFrame(rows)


## 5. Scenarios and reliability targets


In [ ]:
def scenario_alpha1(scenario, cfg=CFG, ctx=TRUE_CTX):
    kind = scenario.get("kind", "true_model")
    if kind in {"alpha1_multiplier", "ea_multiplier"}:
        return float(ctx["alpha1"] * scenario.get(
            "alpha1_multiplier",
            scenario.get("ea_multiplier", 1.0),
        ))
    return float(ctx["alpha1"])


def scenario_log_beta(T, scenario, cfg=CFG, ctx=TRUE_CTX):
    T = np.asarray(T, float)
    s = standardized_stress(T, cfg.T0, cfg.Tmax)
    a1 = scenario_alpha1(scenario, cfg, ctx)
    kind = scenario.get("kind", "true_model")
    if kind == "piecewise":
        switch_s = float(standardized_stress(scenario.get("switch_T", 85.0), cfg.T0, cfg.Tmax))
        high = float(scenario.get("high_slope_multiplier", 1.0))
        return ctx["alpha0"] + a1 * np.minimum(s, switch_s) + a1 * high * np.maximum(s - switch_s, 0)
    if kind == "quadratic":
        return ctx["alpha0"] + a1 * s + float(scenario.get("c_quad", 0.0)) * s**2
    return ctx["alpha0"] + a1 * s


def scenario_beta(T, scenario, cfg=CFG, ctx=TRUE_CTX):
    return np.exp(np.clip(scenario_log_beta(T, scenario, cfg, ctx), -700, 700))


def scenario_eta(T, scenario, ctx=TRUE_CTX):
    T = np.asarray(T, float)
    eta = np.full_like(T, float(ctx["eta"]), dtype=float)
    if scenario.get("kind") == "piecewise":
        eta = np.where(
            T > float(scenario.get("switch_T", 85.0)),
            eta * float(scenario.get("high_eta_multiplier", 1.0)), eta,
        )
    return np.maximum(eta, 1e-12)


def scenario_q(T, scenario, cfg=CFG, ctx=TRUE_CTX):
    s = standardized_stress(T, cfg.T0, cfg.Tmax)
    return np.maximum(ctx["q"] * (1.0 + float(scenario.get("q_shift", 0.0)) * s), 0.05)


def gamma_failure_cdf(t, beta, eta, q, threshold):
    A = max(float(t), 1e-12) ** float(q)
    shape = max(float(eta) * A / max(float(beta), 1e-15), 1e-12)
    scale = max(float(beta) ** 2 / max(float(eta), 1e-15), 1e-12)
    return float(gamma_dist.sf(float(threshold), a=shape, scale=scale))


def lifetime_quantile_gamma(beta, eta, q, threshold, p):
    drift = (float(threshold) / float(beta)) ** (1.0 / float(q))
    lo, hi = max(drift * 1e-5, 1e-9), drift * 10
    root = lambda t: gamma_failure_cdf(t, beta, eta, q, threshold) - p
    while root(lo) > 0:
        lo *= 0.1
    while root(hi) < 0:
        hi *= 2
    return float(brentq(root, lo, hi, xtol=1e-8, rtol=1e-10, maxiter=250))


TARGETS = {
    "life_q10": {"label": r"$\log \xi_{0.1}$", "type": "life_quantile", "p": 0.10},
    "life_q50": {"label": r"$\log \xi_{0.5}$", "type": "life_quantile", "p": 0.50},
    "life_q90": {"label": r"$\log \xi_{0.9}$", "type": "life_quantile", "p": 0.90},
    "mission_R": {
        "label": rf"$R({CFG.mission_years:g}\,\mathrm{{yr}})$",
        "type": "mission_reliability",
        "mission_years": float(CFG.mission_years),
    },
}


def target_from_working_theta(theta, target_name, cfg=CFG):
    spec = TARGETS[target_name]
    beta0 = math.exp(float(theta["alpha0"]))
    if spec["type"] == "life_quantile":
        xi = lifetime_quantile_ig(beta0, theta["eta"], theta["q"], cfg.YD, spec["p"])
        return float(np.log(xi))
    failure = failure_cdf_ig(cfg.mission_time_h, beta0, theta["eta"], theta["q"], cfg.YD)
    return float(np.clip(1.0 - failure, 0.0, 1.0))


def target_truth(scenario, target_name, cfg=CFG, ctx=TRUE_CTX):
    spec = TARGETS[target_name]
    T = cfg.T0
    beta0 = float(scenario_beta(T, scenario, cfg, ctx))
    eta0 = float(scenario_eta(T, scenario, ctx))
    q0 = float(scenario_q(T, scenario, cfg, ctx))
    process = scenario.get("process", "ig")
    if spec["type"] == "life_quantile":
        if process == "gamma":
            xi = lifetime_quantile_gamma(beta0, eta0, q0, cfg.YD, spec["p"])
        else:
            xi = lifetime_quantile_ig(beta0, eta0, q0, cfg.YD, spec["p"])
        return float(np.log(xi))
    if process == "gamma":
        failure = gamma_failure_cdf(cfg.mission_time_h, beta0, eta0, q0, cfg.YD)
    else:
        failure = failure_cdf_ig(cfg.mission_time_h, beta0, eta0, q0, cfg.YD)
    return float(np.clip(1.0 - failure, 0.0, 1.0))


def scenario_increment_moments(T, left, right, scenario, cfg=CFG, ctx=TRUE_CTX):
    beta = float(scenario_beta(T, scenario, cfg, ctx))
    eta = float(scenario_eta(T, scenario, ctx))
    q = float(scenario_q(T, scenario, cfg, ctx))
    A = max(float(right) ** q - float(left) ** q, 1e-15)
    mean = beta * A
    variance = beta**3 * A / eta
    return mean, max(variance, 1e-18)


def scenario_failure_cdf(t, T, scenario, cfg=CFG, ctx=TRUE_CTX):
    beta = float(scenario_beta(T, scenario, cfg, ctx))
    eta = float(scenario_eta(T, scenario, ctx))
    q = float(scenario_q(T, scenario, cfg, ctx))
    if scenario.get("process") == "gamma":
        return gamma_failure_cdf(t, beta, eta, q, cfg.YD)
    return failure_cdf_ig(t, beta, eta, q, cfg.YD)


def simulate_design_data(design, scenario, seed, cfg=CFG, ctx=TRUE_CTX):
    d, levels, _, schedules = planned_schedule(design, cfg, ctx)
    rng = rng_from_seed(seed)
    Ts, lefts, rights, dys = [], [], [], []
    for T, units, times in zip(levels, d["Ni"], schedules):
        for _ in range(int(units)):
            cumulative = 0.0
            for left, right in zip(times[:-1], times[1:]):
                mean, variance = scenario_increment_moments(T, left, right, scenario, cfg, ctx)
                if scenario.get("process") == "gamma":
                    shape = max(mean**2 / variance, 1e-12)
                    scale = max(variance / mean, 1e-12)
                    increment = float(rng.gamma(shape, scale))
                else:
                    wald_shape = max(mean**3 / variance, 1e-12)
                    increment = float(rng.wald(max(mean, 1e-12), wald_shape))
                Ts.append(float(T)); lefts.append(float(left)); rights.append(float(right)); dys.append(max(increment, 1e-15))
                cumulative += increment
                if cfg.censor_at_failure and cumulative >= cfg.YD:
                    break
    return tuple(np.asarray(x, float) for x in (Ts, lefts, rights, dys))


## 6. Fisher-information benchmarks


In [ ]:
def power_log_term(t, q):
    t = np.asarray(t, float)
    out = np.zeros_like(t)
    positive = t > 0
    out[positive] = t[positive] ** q * np.log(t[positive])
    return out


def expected_fisher_information(design, cfg=CFG, ctx=TRUE_CTX, survival_weight=True):
    d, levels, _, schedules = planned_schedule(design, cfg, ctx)
    a0, a1, eta, q = ctx["alpha0"], ctx["alpha1"], ctx["eta"], ctx["q"]
    info = np.zeros((4, 4), float)
    for T, units, times in zip(levels, d["Ni"], schedules):
        s = float(standardized_stress(T, cfg.T0, cfg.Tmax))
        beta = float(beta_model(T, a0, a1, cfg))
        left, right = times[:-1], times[1:]
        A = right**q - left**q
        dA = power_log_term(right, q) - power_log_term(left, q)
        if survival_weight:
            survival = np.array([1.0 - failure_cdf_ig(x, beta, eta, q, cfg.YD) for x in left])
        else:
            survival = np.ones_like(A)
        w = float(units) * np.maximum(survival, 1e-6)
        info[0, 0] += np.sum(w * eta / beta * A)
        info[0, 1] += np.sum(w * eta * s / beta * A)
        info[0, 3] += np.sum(w * eta / beta * dA)
        info[1, 1] += np.sum(w * eta * s**2 / beta * A)
        info[1, 3] += np.sum(w * eta * s / beta * dA)
        info[2, 2] += np.sum(w / (2 * eta**2))
        info[2, 3] += np.sum(w * dA / (eta * A))
        info[3, 3] += np.sum(w * dA**2 * (2 / A**2 + eta / (beta * A)))
    info[1, 0] = info[0, 1]
    info[3, 0] = info[0, 3]
    info[3, 1] = info[1, 3]
    info[3, 2] = info[2, 3]
    return 0.5 * (info + info.T)


def target_gradient(theta, target_name, cfg=CFG):
    natural = np.array([theta["alpha0"], theta["alpha1"], theta["eta"], theta["q"]], float)
    gradient = np.zeros(4, float)
    for index in (0, 2, 3):
        step = 1e-5 * max(abs(natural[index]), 1.0)
        if index in (2, 3):
            step = min(step, 0.2 * natural[index])
        upper, lower = natural.copy(), natural.copy()
        upper[index] += step; lower[index] -= step
        theta_upper = {"alpha0": upper[0], "alpha1": upper[1], "eta": upper[2], "q": upper[3]}
        theta_lower = {"alpha0": lower[0], "alpha1": lower[1], "eta": lower[2], "q": lower[3]}
        gradient[index] = (
            target_from_working_theta(theta_upper, target_name, cfg)
            - target_from_working_theta(theta_lower, target_name, cfg)
        ) / (2 * step)
    return gradient


TARGET_GRADIENT_CACHE = {}


def cached_target_gradient(ctx, target_name, cfg=CFG):
    key = (
        target_name, float(cfg.YD), float(cfg.mission_time_h),
        float(ctx["alpha0"]), float(ctx["alpha1"]),
        float(ctx["eta"]), float(ctx["q"]),
    )
    if key not in TARGET_GRADIENT_CACHE:
        TARGET_GRADIENT_CACHE[key] = target_gradient(ctx, target_name, cfg)
    return TARGET_GRADIENT_CACHE[key].copy()


def classical_criteria(design, cfg=CFG, ctx=TRUE_CTX):
    information = expected_fisher_information(design, cfg, ctx, survival_weight=True)
    eig = np.linalg.eigvalsh(information)
    if not np.isfinite(eig).all() or eig.min() <= 0:
        result = {"D": np.inf, "A": np.inf, "M_inspired": np.inf,
                  "FIM_min_eig": float(np.nanmin(eig)), "FIM_condition": np.inf}
        result.update({f"V_{name}": np.inf for name in TARGETS})
        return result
    covariance = np.linalg.inv(information)
    sign, logdet = np.linalg.slogdet(information)
    result = {
        "D": float(-logdet) if sign > 0 else np.inf,
        "A": float(np.trace(covariance)),
        "M_inspired": float(covariance[1, 1]),
        "FIM_min_eig": float(eig.min()),
        "FIM_condition": float(np.linalg.cond(information)),
    }
    for name in TARGETS:
        gradient = cached_target_gradient(ctx, name, cfg)
        result[f"V_{name}"] = float(gradient @ covariance @ gradient)
    return result


## 7. Finite-sample loss and mean-CVaR


In [ ]:
def weighted_cvar(losses, weights, alpha):
    losses = np.asarray(losses, float)
    weights = normalize_weights(weights)
    order = np.argsort(-losses)
    remaining = 1.0 - float(alpha)
    tail_mass = max(remaining, 1e-12)
    total = 0.0
    for index in order:
        take = min(float(weights[index]), remaining)
        total += take * float(losses[index])
        remaining -= take
        if remaining <= 1e-14:
            break
    return float(total / tail_mass)


RRE_CACHE = {}


def evaluate_rre(design, reps, stage, cfg=CFG, ctx=TRUE_CTX, scenarios=SCENARIOS):
    d = canonicalize_design(design, cfg)
    scenario_signature = tuple(
        json.dumps(s, ensure_ascii=False, sort_keys=True) for s in scenarios
    )
    key = (design_id(d, cfg), int(reps), str(stage), scenario_signature)
    if key in RRE_CACHE:
        return copy.deepcopy(RRE_CACHE[key])
    scenario_rows = []
    raw_rows = []
    for scenario_index, scenario in enumerate(scenarios):
        truths = {name: target_truth(scenario, name, cfg, ctx) for name in TARGETS}
        errors = {name: [] for name in TARGETS}
        estimates = {name: [] for name in TARGETS}
        valid_count = 0
        formal_success = 0
        fallback_count = 0
        for rep in range(int(reps)):
            seed = stable_seed(MASTER_SEED, stage, scenario_index, rep)
            data = simulate_design_data(d, scenario, seed, cfg, ctx)
            fit = fit_assumed_ig_with_fallback(data, cfg, init=TRUE_RAW)
            valid = bool(fit.get("valid", False))
            formal_success += int(bool(fit.get("success", False)))
            fallback_count += int(bool(fit.get("multistart_fallback_used", False)))
            if valid:
                try:
                    fitted_targets = {name: target_from_working_theta(fit, name, cfg) for name in TARGETS}
                    valid = bool(np.isfinite(list(fitted_targets.values())).all())
                except Exception:
                    valid = False
            if valid:
                valid_count += 1
            for name in TARGETS:
                if valid:
                    estimate = float(fitted_targets[name])
                    error = estimate - truths[name]
                    sq_error = float(error**2)
                    estimates[name].append(estimate)
                else:
                    estimate = np.nan
                    error = np.nan
                    sq_error = np.nan
                errors[name].append(sq_error)
                raw_rows.append({
                    "design_id": design_id(d, cfg), "stage": stage,
                    "Scenario": scenario["Scenario"], "Category": scenario["Category"],
                    "role": scenario.get("role", "primary"),
                    "scenario_index": scenario_index, "rep": rep, "target": name,
                    "truth_transformed": truths[name], "estimate_transformed": estimate,
                    "error_transformed": error, "squared_error": sq_error,
                    "fit_valid": valid, "optimizer_success": bool(fit.get("success", False)),
                    "multistart_fallback_used": bool(fit.get("multistart_fallback_used", False)),
                })
        for name in TARGETS:
            valid_values = np.asarray(estimates[name], float)
            if valid_values.size:
                bias = float(np.mean(valid_values) - truths[name])
                variance = float(np.var(valid_values, ddof=1)) if valid_values.size > 1 else 0.0
            else:
                bias, variance = np.nan, np.nan
            finite_errors = np.asarray(errors[name], float)
            finite_errors = finite_errors[np.isfinite(finite_errors)]
            scenario_rows.append({
                "design_id": design_id(d, cfg), "Scenario": scenario["Scenario"],
                "Category": scenario["Category"], "scenario_index": scenario_index,
                "role": scenario.get("role", "primary"),
                "scenario_weight": float(scenario.get("weight", 1.0)), "target": name,
                "scenario_MSE": float(np.mean(finite_errors)) if finite_errors.size else np.inf,
                "bias_transformed": bias, "bias2_transformed": float(bias**2) if np.isfinite(bias) else np.nan,
                "variance_transformed": variance,
                "valid_fit_fraction": valid_count / max(int(reps), 1),
                "optimizer_success_fraction": formal_success / max(int(reps), 1),
                "multistart_fallback_fraction": fallback_count / max(int(reps), 1),
            })
    scenario_frame = pd.DataFrame(scenario_rows)
    summary = {"design_id": design_id(d, cfg), "Tmin": d["Tmin"], "N": d["N"],
               "m": d["m"], "Ni": tuple(int(v) for v in d["Ni"]), "reps": int(reps)}
    for name in TARGETS:
        sub = scenario_frame[scenario_frame["target"] == name].sort_values("scenario_index")
        primary = sub[sub["scenario_weight"] > 0].copy()
        weights = normalize_weights(primary["scenario_weight"].to_numpy(float))
        losses = primary["scenario_MSE"].to_numpy(float)
        mean_loss = float(np.dot(weights, losses))
        cvar_loss = weighted_cvar(losses, weights, cfg.robust_cvar_alpha)
        conditional_phi = (
            (1 - cfg.robust_cvar_weight) * mean_loss
            + cfg.robust_cvar_weight * cvar_loss
        )
        min_fit = float(primary["valid_fit_fraction"].min())
        eligible = bool(
            np.isfinite(conditional_phi)
            and min_fit >= cfg.minimum_valid_fit_fraction
        )
        summary[f"RRE_mean_{name}"] = mean_loss
        summary[f"RRE_CVaR_{name}"] = cvar_loss
        summary[f"Phi_RRE_conditional_{name}"] = float(conditional_phi)
        summary[f"Phi_RRE_{name}"] = float(conditional_phi) if eligible else np.inf
        summary[f"RRE_eligible_{name}"] = eligible
        summary[f"min_fit_fraction_{name}"] = min_fit
    result = {"summary": summary, "scenario": scenario_frame, "raw": pd.DataFrame(raw_rows)}
    RRE_CACHE[key] = copy.deepcopy(result)
    return result


## 8. Stochastic design search


In [ ]:
def design_row(design, cfg=CFG, ctx=TRUE_CTX):
    d = canonicalize_design(design, cfg)
    cost = compute_total_cost(d, cfg, ctx)
    classical = classical_criteria(d, cfg, ctx)
    return {
        "design_id": design_id(d, cfg), "Tmin": d["Tmin"], "N": d["N"], "m": d["m"],
        "n1": int(d["Ni"][0]), "n2": int(d["Ni"][1]), "n3": int(d["Ni"][2]),
        "Ni": tuple(int(v) for v in d["Ni"]), "Cost": cost["C_total"],
        "Cost_ratio": cost["C_total"] / cfg.comparison_budget,
        "Campaign_h": cost["t_campaign"],
        "Beta_ratio_Tmin": float(selected_beta_ratio(d["Tmin"], cfg, ctx)),
        "AFt_Tmin": float(selected_time_acceleration_factor(d["Tmin"], cfg, ctx)),
        "AF_Tmin": float(selected_time_acceleration_factor(d["Tmin"], cfg, ctx)),
        **classical,
    }


def row_to_design(row):
    return canonicalize_design({
        "Tmin": int(row["Tmin"]), "N": int(row["N"]), "m": int(row["m"]),
        "Ni": np.array([int(row["n1"]), int(row["n2"]), int(row["n3"])]),
    }, CFG)


def generate_candidate_pool(cfg=CFG, ctx=TRUE_CTX, seed=MASTER_SEED + 11):
    if cfg.N_lb == cfg.N_ub and cfg.m_lb == cfg.m_ub:
        rows = []
        feasible_Tmin = []
        N, m, u = int(cfg.N_lb), int(cfg.m_lb), int(cfg.min_units_per_stress)
        for Tmin in range(cfg.Tmin_lb, cfg.Tmin_ub + 1):
            any_feasible = False
            for n1 in range(u, N - 2 * u + 1):
                for n2 in range(u, N - n1 - u + 1):
                    n3 = N - n1 - n2
                    design = {"Tmin": Tmin, "N": N, "m": m,
                              "Ni": np.array([n1, n2, n3], int)}
                    if is_feasible(design, cfg, ctx):
                        rows.append(design_row(design, cfg, ctx))
                        any_feasible = True
            if any_feasible:
                feasible_Tmin.append(Tmin)
        if not rows:
            raise RuntimeError("The fixed-resource integer design domain is empty.")
        frame = pd.DataFrame(rows).sort_values(
            ["Cost", "Tmin", "n1", "n2", "n3"]
        ).reset_index(drop=True)
        return frame, feasible_Tmin

    rng = rng_from_seed(seed)
    designs = {}
    feasible_Tmin = []
    for Tmin in range(cfg.Tmin_lb, cfg.Tmin_ub + 1):
        try:
            probe = random_feasible_design(rng, cfg, ctx, near_budget=False, fixed_Tmin=Tmin)
            feasible_Tmin.append(Tmin)
            designs[design_id(probe, cfg)] = probe
        except RuntimeError:
            continue
    attempts = 0
    while len(designs) < cfg.candidate_count and attempts < cfg.candidate_count * 1000:
        attempts += 1
        try:
            design = random_feasible_design(rng, cfg, ctx, near_budget=True)
        except RuntimeError:
            break
        designs[design_id(design, cfg)] = design
    target_total = int(math.ceil(cfg.candidate_count * 1.15))
    while len(designs) < target_total and attempts < cfg.candidate_count * 2000:
        attempts += 1
        try:
            design = random_feasible_design(rng, cfg, ctx, near_budget=False)
        except RuntimeError:
            break
        designs[design_id(design, cfg)] = design
    if len(designs) < max(30, cfg.shortlist_count):
        raise RuntimeError("The feasible candidate pool is too small for a fair comparison.")
    frame = pd.DataFrame([design_row(d, cfg, ctx) for d in designs.values()])
    frame = frame.sort_values(["Cost", "Tmin", "N", "m"]).reset_index(drop=True)
    return frame, feasible_Tmin


def matched_cost_pool(candidate_frame, cfg=CFG):
    if cfg.N_lb == cfg.N_ub and cfg.m_lb == cfg.m_ub:
        pool = candidate_frame[
            candidate_frame["Cost"] <= cfg.comparison_budget + cfg.budget_tolerance
        ].copy()
        if len(pool) < 12:
            raise RuntimeError("Too few fixed-resource feasible candidates.")
        return pool.reset_index(drop=True), 0.0
    lower = cfg.matched_cost_lower_ratio * cfg.comparison_budget
    pool = candidate_frame[(candidate_frame["Cost"] >= lower) & (candidate_frame["Cost"] <= cfg.comparison_budget + 1e-8)].copy()
    if len(pool) < max(20, cfg.shortlist_count // 2):
        lower = 0.85 * cfg.comparison_budget
        pool = candidate_frame[(candidate_frame["Cost"] >= lower) & (candidate_frame["Cost"] <= cfg.comparison_budget + 1e-8)].copy()
    if len(pool) < 12:
        raise RuntimeError("Too few matched-cost candidates; increase ADT_CANDIDATE_COUNT.")
    return pool.reset_index(drop=True), lower


def choose_shortlist(pool, cfg=CFG, seed=MASTER_SEED + 29):
    criteria = ["D", "A", "M_inspired"] + [f"V_{name}" for name in TARGETS]
    indices = set()
    per_criterion = max(2, cfg.shortlist_count // (len(criteria) + 2))
    for criterion in criteria:
        indices.update(pool.nsmallest(per_criterion, criterion).index.tolist())
    for _, group in pool.groupby("Tmin"):
        indices.add(int((cfg.comparison_budget - group["Cost"]).abs().idxmin()))
    rng = rng_from_seed(seed)
    remaining = [idx for idx in pool.index if idx not in indices]
    desired = min(cfg.shortlist_count, len(pool))
    if len(indices) < desired and remaining:
        extra = rng.choice(remaining, size=min(desired - len(indices), len(remaining)), replace=False)
        indices.update(int(v) for v in np.atleast_1d(extra))
    return pool.loc[sorted(indices)].copy().reset_index(drop=True)


def local_neighbors(design, rng, cfg=CFG, ctx=TRUE_CTX, count=16):
    base = canonicalize_design(design, cfg)
    answers = {}
    for _ in range(count * 20):
        if len(answers) >= count:
            break
        Tmin = int(np.clip(base["Tmin"] + rng.integers(-3, 4), cfg.Tmin_lb, cfg.Tmin_ub))
        N = int(np.clip(base["N"] + rng.integers(-8, 9), cfg.N_lb, cfg.N_ub))
        m = int(np.clip(base["m"] + rng.integers(-3, 4), cfg.m_lb, cfg.m_ub))
        weights = np.maximum(base["Ni"] + rng.normal(0, 2.0, cfg.r), 0.1)
        candidate = canonicalize_design({"Tmin": Tmin, "N": N, "m": m, "pi": weights}, cfg)
        if is_feasible(candidate, cfg, ctx):
            cost = compute_total_cost(candidate, cfg, ctx)["C_total"]
            if cost >= cfg.matched_cost_lower_ratio * cfg.comparison_budget:
                answers[design_id(candidate, cfg)] = candidate
    return list(answers.values())


def run_three_stage_optimization(cfg=CFG, ctx=TRUE_CTX):
    candidate_frame, feasible_Tmin = generate_candidate_pool(cfg, ctx)
    pool, cost_lower = matched_cost_pool(candidate_frame, cfg)
    shortlist = choose_shortlist(pool, cfg)
    print(
        f"Candidate pool={len(candidate_frame)}, matched-cost pool={len(pool)}, "
        f"initial shortlist={len(shortlist)}", flush=True
    )
    screen_rows, screen_scenarios = [], []
    for order, (_, row) in enumerate(shortlist.iterrows(), start=1):
        result = evaluate_rre(row_to_design(row), cfg.rre_screen_reps, "screen", cfg, ctx)
        screen_rows.append({**row.to_dict(), **result["summary"], "evaluation_order": order})
        screen_scenarios.append(result["scenario"])
        if order % 10 == 0 or order == len(shortlist):
            print(f"RRE screen: {order}/{len(shortlist)}", flush=True)
    screen_frame = pd.DataFrame(screen_rows)

    mandatory_rows = [
        pool.nsmallest(1, "A").iloc[0],
        pool.nsmallest(1, "D").iloc[0],
        pool.nsmallest(1, "M_inspired").iloc[0],
        equal_allocation_baseline(pool, None, cfg),
        literature_tmin_baseline(pool, 65),
    ] + [pool.nsmallest(1, f"V_{target}").iloc[0] for target in TARGETS]
    mandatory_designs = {
        str(row["design_id"]): row_to_design(row) for row in mandatory_rows
    }

    rng = rng_from_seed(MASTER_SEED + 43)
    neighbor_map = dict(mandatory_designs)
    for target in TARGETS:
        for _, row in screen_frame.nsmallest(2, f"Phi_RRE_{target}").iterrows():
            for neighbor in local_neighbors(row_to_design(row), rng, cfg, ctx, count=6):
                neighbor_map[design_id(neighbor, cfg)] = neighbor
    for baseline_design in mandatory_designs.values():
        for neighbor in local_neighbors(baseline_design, rng, cfg, ctx, count=4):
            neighbor_map[design_id(neighbor, cfg)] = neighbor
    existing = set(screen_frame["design_id"])
    for neighbor in neighbor_map.values():
        if design_id(neighbor, cfg) in existing:
            continue
        result = evaluate_rre(neighbor, cfg.rre_screen_reps, "screen", cfg, ctx)
        row = design_row(neighbor, cfg, ctx)
        screen_rows.append({**row, **result["summary"], "evaluation_order": len(screen_rows) + 1})
        screen_scenarios.append(result["scenario"])
    screen_frame = pd.DataFrame(screen_rows).drop_duplicates("design_id").reset_index(drop=True)

    refine_ids = set()
    for target in TARGETS:
        refine_ids.update(
            screen_frame.nsmallest(cfg.refine_top_k, f"Phi_RRE_{target}")["design_id"].tolist()
        )
    refine_ids.update(mandatory_designs.keys())
    refine_rows, refine_scenarios, refine_raw = [], [], []
    for did in sorted(refine_ids):
        row = screen_frame[screen_frame["design_id"] == did].iloc[0]
        result = evaluate_rre(row_to_design(row), cfg.rre_refine_reps, "refine", cfg, ctx)
        refine_rows.append({**{key: row[key] for key in candidate_frame.columns}, **result["summary"]})
        refine_scenarios.append(result["scenario"])
        refine_raw.append(result["raw"])
        print(f"RRE refine: {len(refine_rows)}/{len(refine_ids)} ({did})", flush=True)
    refine_frame = pd.DataFrame(refine_rows)
    if refine_frame.empty:
        raise RuntimeError("No design reached the refinement stage.")

    confirmation_ids = set(mandatory_designs.keys())
    for target in TARGETS:
        confirmation_ids.update(
            refine_frame.nsmallest(cfg.confirmation_top_k, f"Phi_RRE_{target}")["design_id"].tolist()
        )
    confirmation_rows, confirmation_scenarios, confirmation_raw = [], [], []
    for did in sorted(confirmation_ids):
        row = screen_frame[screen_frame["design_id"] == did].iloc[0]
        result = evaluate_rre(
            row_to_design(row), cfg.confirmation_reps, "confirmation", cfg, ctx
        )
        confirmation_rows.append({
            **{key: row[key] for key in candidate_frame.columns}, **result["summary"]
        })
        confirmation_scenarios.append(result["scenario"])
        confirmation_raw.append(result["raw"])
        print(
            f"RRE confirmation: {len(confirmation_rows)}/{len(confirmation_ids)} ({did})",
            flush=True,
        )
    confirmation_frame = pd.DataFrame(confirmation_rows)
    if confirmation_frame.empty:
        raise RuntimeError("No design reached the independent confirmation stage.")

    for audit_round in range(5):
        new_rows = {}
        evaluated_ids = set(confirmation_frame["design_id"])
        for target in TARGETS:
            leader = confirmation_frame.nsmallest(1, f"Phi_RRE_{target}").iloc[0]
            center = float(leader["Cost"])
            tolerance = cfg.matched_cost_tolerance_ratio * center
            band = pool[(pool["Cost"] - center).abs() <= tolerance]
            if band.empty:
                continue
            v_row = band.nsmallest(1, f"V_{target}").iloc[0]
            did = str(v_row["design_id"])
            if did not in evaluated_ids:
                new_rows[did] = v_row
        if not new_rows:
            break
        for did, row in sorted(new_rows.items()):
            result = evaluate_rre(
                row_to_design(row), cfg.confirmation_reps, "confirmation", cfg, ctx
            )
            confirmation_rows.append({
                **{key: row[key] for key in candidate_frame.columns}, **result["summary"]
            })
            confirmation_scenarios.append(result["scenario"])
            confirmation_raw.append(result["raw"])
            print(f"Cost-band confirmation audit round {audit_round + 1}: {did}", flush=True)
        confirmation_frame = pd.DataFrame(confirmation_rows).drop_duplicates(
            "design_id", keep="last"
        ).reset_index(drop=True)
    return {
        "candidates": candidate_frame, "pool": pool, "shortlist": shortlist,
        "screen": screen_frame, "screen_scenarios": pd.concat(screen_scenarios, ignore_index=True),
        "refine": refine_frame, "refine_scenarios": pd.concat(refine_scenarios, ignore_index=True),
        "refine_raw": pd.concat(refine_raw, ignore_index=True),
        "confirmation": confirmation_frame,
        "confirmation_scenarios": pd.concat(confirmation_scenarios, ignore_index=True),
        "confirmation_raw": pd.concat(confirmation_raw, ignore_index=True),
        "feasible_Tmin": feasible_Tmin, "matched_cost_lower": cost_lower,
    }


def literature_tmin_baseline(pool, anchor=65):
    fixed = pool[pool["Tmin"] == int(anchor)]
    if fixed.empty:
        distance = (pool["Tmin"] - int(anchor)).abs()
        fixed = pool[distance == distance.min()]
    return fixed.nsmallest(1, "V_life_q10").iloc[0]


def fixed_tmin_baseline(pool, anchor=65):
    return literature_tmin_baseline(pool, anchor)


def equal_allocation_baseline(pool, Tmin=None, cfg=CFG):
    if Tmin is None:
        Tmin = int(pool["Tmin"].min())
    N = int(cfg.N_lb)
    base = N // cfg.r
    allocation = np.full(cfg.r, base, dtype=int)
    allocation[: N - int(allocation.sum())] += 1
    mask = (
        (pool["Tmin"] == int(Tmin))
        & (pool["N"] == N)
        & (pool["m"] == int(cfg.m_lb))
        & (pool["n1"] == int(allocation[0]))
        & (pool["n2"] == int(allocation[1]))
        & (pool["n3"] == int(allocation[2]))
    )
    exact = pool[mask]
    if exact.empty:
        raise RuntimeError(
            f"The equal-allocation engineering plan at Tmin={int(Tmin)} degC is infeasible "
            "or absent from the exhaustive design pool."
        )
    return exact.iloc[0]


def select_method_designs(optimization, cfg=CFG):
    pool = optimization["pool"]
    confirmation = optimization["confirmation"]
    common = {
        "A-optimality": pool.nsmallest(1, "A").iloc[0],
        "D-optimality": pool.nsmallest(1, "D").iloc[0],
        "M-inspired AF-slope precision": pool.nsmallest(1, "M_inspired").iloc[0],
        "Conventional equal-allocation": equal_allocation_baseline(pool, None, cfg),
        "Literature Tmin plan": literature_tmin_baseline(pool, 65),
    }
    rows = []
    for target in TARGETS:
        proposed = confirmation.nsmallest(1, f"Phi_RRE_{target}").iloc[0]
        proposed_cost = float(proposed["Cost"])
        tolerance = cfg.matched_cost_tolerance_ratio * proposed_cost
        cost_matched = pool[(pool["Cost"] - proposed_cost).abs() <= tolerance]
        if cost_matched.empty:
            raise RuntimeError(f"No cost-matched V comparison is available for {target}.")
        target_methods = dict(common)
        target_methods["V-optimality"] = pool.nsmallest(1, f"V_{target}").iloc[0]
        target_methods["Cost-matched V-optimality"] = cost_matched.nsmallest(
            1, f"V_{target}"
        ).iloc[0]
        target_methods["Proposed RRE-optimality"] = proposed
        for method, row in target_methods.items():
            record = {"target": target, "method": method}
            record.update({key: row[key] for key in [
                "design_id", "Tmin", "N", "m", "n1", "n2", "n3", "Ni", "Cost",
                "Cost_ratio", "Campaign_h", "Beta_ratio_Tmin", "AFt_Tmin", "AF_Tmin",
                "D", "A", "M_inspired",
                f"V_{target}", "FIM_min_eig", "FIM_condition",
            ] if key in row.index})
            record["V_target"] = float(row[f"V_{target}"])
            phi_name = f"Phi_RRE_{target}"
            record["Phi_RRE_confirmation"] = (
                float(row[phi_name])
                if phi_name in row.index and np.isfinite(row[phi_name]) else np.nan
            )
            rows.append(record)
    return pd.DataFrame(rows)


## 9. Independent validation


In [ ]:
def inverse_target(value, target_name):
    if target_name.startswith("life_"):
        return float(np.exp(np.clip(value, -700, 700)))
    return float(np.clip(value, 0.0, 1.0))


def validate_unique_designs(method_designs, cfg=CFG, ctx=TRUE_CTX):
    unique_rows = method_designs.drop_duplicates("design_id")
    raw = []
    for design_order, (_, design_row_item) in enumerate(unique_rows.iterrows(), start=1):
        design = row_to_design(design_row_item)
        did = design_id(design, cfg)
        print(
            f"Independent validation design {design_order}/{len(unique_rows)}: {did}",
            flush=True,
        )
        for scenario_index, scenario in enumerate(SCENARIOS):
            truths = {name: target_truth(scenario, name, cfg, ctx) for name in TARGETS}
            for rep in range(cfg.validation_reps):
                seed = stable_seed(MASTER_SEED, "independent_validation", scenario_index, rep)
                data = simulate_design_data(design, scenario, seed, cfg, ctx)
                fit = fit_assumed_ig_with_fallback(data, cfg, init=TRUE_RAW)
                valid = bool(fit.get("valid", False))
                if valid:
                    try:
                        estimates = {name: target_from_working_theta(fit, name, cfg) for name in TARGETS}
                        valid = bool(np.isfinite(list(estimates.values())).all())
                    except Exception:
                        valid = False
                for target in TARGETS:
                    truth = truths[target]
                    if valid:
                        estimate = float(estimates[target])
                        error = estimate - truth
                        squared = float(error**2)
                        natural_truth = inverse_target(truth, target)
                        natural_estimate = inverse_target(estimate, target)
                        if target.startswith("life_"):
                            natural_error = (natural_estimate - natural_truth) / natural_truth
                            natural_error_unit = "relative"
                        else:
                            natural_error = natural_estimate - natural_truth
                            natural_error_unit = "absolute_probability"
                    else:
                        estimate = np.nan; error = np.nan
                        squared = np.nan
                        natural_truth = inverse_target(truth, target)
                        natural_estimate = np.nan; natural_error = np.nan
                        natural_error_unit = "relative" if target.startswith("life_") else "absolute_probability"
                    raw.append({
                        "design_id": did, "Scenario": scenario["Scenario"],
                        "Category": scenario["Category"], "scenario_index": scenario_index,
                        "role": scenario.get("role", "primary"),
                        "scenario_weight": scenario["weight"], "rep": rep, "target": target,
                        "truth_transformed": truth, "estimate_transformed": estimate,
                        "error_transformed": error, "squared_error": squared,
                        "truth_natural": natural_truth, "estimate_natural": natural_estimate,
                        "natural_error": natural_error, "natural_error_unit": natural_error_unit,
                        "fit_valid": valid, "optimizer_success": bool(fit.get("success", False)),
                        "multistart_fallback_used": bool(fit.get("multistart_fallback_used", False)),
                        "n_increments": int(fit.get("n_increments", len(data[-1]))),
                    })
    unique_raw = pd.DataFrame(raw)
    mapping = method_designs[["target", "method", "design_id"]].copy()
    mapped = unique_raw.merge(mapping, on=["target", "design_id"], how="inner", validate="many_to_many")
    return unique_raw, mapped


def aggregate_method_validation(mapped, cfg=CFG):
    scenario_rows = []
    for (target, method, scenario_index), group in mapped.groupby(["target", "method", "scenario_index"]):
        valid = group[group["fit_valid"]]
        errors = valid["error_transformed"].to_numpy(float)
        natural_errors = valid["natural_error"].to_numpy(float)
        bias = float(np.mean(errors)) if len(errors) else np.nan
        variance = float(np.var(errors, ddof=1)) if len(errors) > 1 else (0.0 if len(errors) == 1 else np.nan)
        scenario_rows.append({
            "target": target, "method": method, "Scenario": group["Scenario"].iloc[0],
            "Category": group["Category"].iloc[0], "scenario_index": int(scenario_index),
            "role": group["role"].iloc[0],
            "scenario_weight": float(group["scenario_weight"].iloc[0]),
            "MSE_transformed": float(group["squared_error"].mean()),
            "Bias_transformed": bias, "Variance_transformed": variance,
            "Bias2_plus_variance": float(bias**2 + variance) if np.isfinite(bias + variance) else np.nan,
            "Natural_RMSE": float(np.sqrt(np.mean(natural_errors**2))) if len(natural_errors) else np.nan,
            "Fit_valid_fraction": float(group["fit_valid"].mean()),
            "Optimizer_success_fraction": float(group["optimizer_success"].mean()),
            "Multistart_fallback_fraction": float(group["multistart_fallback_used"].mean()),
        })
    scenario_frame = pd.DataFrame(scenario_rows)
    summary_rows = []
    for (target, method), group in scenario_frame.groupby(["target", "method"]):
        group = group.sort_values("scenario_index")
        primary = group[group["scenario_weight"] > 0].copy()
        weights = normalize_weights(primary["scenario_weight"])
        losses = primary["MSE_transformed"].to_numpy(float)
        mean_loss = float(np.dot(weights, losses))
        cvar = weighted_cvar(losses, weights, cfg.robust_cvar_alpha)
        conditional_phi = (1 - cfg.robust_cvar_weight) * mean_loss + cfg.robust_cvar_weight * cvar
        min_fit = float(primary["Fit_valid_fraction"].min())
        eligible = bool(
            np.isfinite(conditional_phi)
            and min_fit >= cfg.minimum_valid_fit_fraction
        )
        phi = float(conditional_phi) if eligible else np.inf
        natural = primary["Natural_RMSE"].to_numpy(float)
        natural_weighted = float(np.sqrt(np.dot(weights, natural**2)))
        summary_rows.append({
            "target": target, "method": method, "Phi_RRE_validation": float(phi),
            "Phi_RRE_conditional": float(conditional_phi),
            "Validation_eligible": eligible,
            "Weighted_mean_MSE": mean_loss, "CVaR_MSE": cvar,
            "Weighted_natural_RMSE": natural_weighted,
            "Natural_RMSE_unit": "relative" if target.startswith("life_") else "absolute_probability",
            "Minimum_fit_valid_fraction": min_fit,
            "Mean_fit_valid_fraction": float(np.dot(weights, primary["Fit_valid_fraction"])),
        })
    return pd.DataFrame(summary_rows), scenario_frame


def phi_from_losses(losses_by_scenario, weights, cfg=CFG, cvar_weight=None):
    lam = cfg.robust_cvar_weight if cvar_weight is None else float(cvar_weight)
    losses = np.asarray(losses_by_scenario, float)
    weights = normalize_weights(weights)
    return float((1 - lam) * np.dot(weights, losses) + lam * weighted_cvar(losses, weights, cfg.robust_cvar_alpha))


def paired_improvement_table(mapped, cfg=CFG):
    rows = []
    baselines = [m for m in sorted(mapped["method"].unique()) if m != "Proposed RRE-optimality"]
    rng = rng_from_seed(MASTER_SEED + 707)
    for target in TARGETS:
        target_frame = mapped[mapped["target"] == target]
        for baseline in baselines:
            comparison = target_frame[target_frame["method"].isin(["Proposed RRE-optimality", baseline])]
            pivot = comparison.pivot_table(
                index=["scenario_index", "rep"], columns="method", values="squared_error", aggfunc="first"
            )
            if "Proposed RRE-optimality" not in pivot or baseline not in pivot:
                continue
            scenario_indices = [
                i for i in sorted(pivot.index.get_level_values("scenario_index").unique())
                if float(SCENARIOS[int(i)]["weight"]) > 0
            ]
            weights = normalize_weights([SCENARIOS[i]["weight"] for i in scenario_indices])

            complete_sets = []
            for scenario_index in scenario_indices:
                group = pivot.xs(scenario_index, level="scenario_index")
                complete = group[["Proposed RRE-optimality", baseline]].dropna().index
                complete_sets.append(set(int(v) for v in complete))
            common_reps = sorted(set.intersection(*complete_sets)) if complete_sets else []
            if not common_reps:
                continue

            def aggregate(frame, method):
                losses = [
                    float(frame.xs(i, level="scenario_index").loc[common_reps, method].mean())
                    for i in scenario_indices
                ]
                return phi_from_losses(losses, weights, cfg)

            phi_p = aggregate(pivot, "Proposed RRE-optimality")
            phi_b = aggregate(pivot, baseline)
            improvement = 100.0 * (phi_b - phi_p) / max(phi_b, 1e-15)
            boot = []
            rep_values = common_reps
            for _ in range(cfg.bootstrap_reps):
                sampled = rng.choice(rep_values, size=len(rep_values), replace=True)
                p_losses, b_losses = [], []
                for scenario_index in scenario_indices:
                    group = pivot.xs(scenario_index, level="scenario_index")
                    p_losses.append(float(group.loc[sampled, "Proposed RRE-optimality"].mean()))
                    b_losses.append(float(group.loc[sampled, baseline].mean()))
                p_phi = phi_from_losses(p_losses, weights, cfg)
                b_phi = phi_from_losses(b_losses, weights, cfg)
                boot.append(100.0 * (b_phi - p_phi) / max(b_phi, 1e-15))
            rows.append({
                "target": target, "baseline": baseline,
                "Phi_proposed": phi_p, "Phi_baseline": phi_b,
                "Improvement_pct": improvement,
                "CI95_low_pct": float(np.percentile(boot, 2.5)),
                "CI95_high_pct": float(np.percentile(boot, 97.5)),
                "paired_reps_per_scenario": len(rep_values),
                "pairing_rule": "common valid replications across all primary scenarios",
            })
    return pd.DataFrame(rows)


def validation_weight_sensitivity(scenario_frame, cfg=CFG):
    category_schemes = {
        "Predeclared": {category: float(sum(s["weight"] for s in SCENARIOS if s["Category"] == category))
                         for category in sorted(set(s["Category"] for s in SCENARIOS))},
        "True-model-heavy": {
            "True model": 0.50,
            "Activation energy": 0.15,
            "Mechanism change": 0.10,
            "Time scale": 0.10,
            "Acceleration curvature": 0.05,
            "Process family": 0.10,
        },
        "Structural-heavy": {
            "True model": 0.15,
            "Activation energy": 0.10,
            "Mechanism change": 0.20,
            "Time scale": 0.20,
            "Acceleration curvature": 0.15,
            "Process family": 0.20,
        },
    }
    rows = []
    for target in TARGETS:
        for scheme, category_mass in category_schemes.items():
            for lam in (0.0, 0.25, 0.50):
                for method, group in scenario_frame[scenario_frame["target"] == target].groupby("method"):
                    group = group.sort_values("scenario_index")
                    raw_weights = []
                    for _, item in group.iterrows():
                        if scheme == "Predeclared":
                            raw_weights.append(float(SCENARIOS[int(item["scenario_index"])]["weight"]))
                        else:
                            count = sum(1 for s in SCENARIOS if s["Category"] == item["Category"])
                            raw_weights.append(category_mass[item["Category"]] / count)
                    phi = phi_from_losses(group["MSE_transformed"], raw_weights, cfg, cvar_weight=lam)
                    rows.append({"target": target, "scheme": scheme, "lambda_CVaR": lam,
                                 "method": method, "Phi_RRE": phi})
    frame = pd.DataFrame(rows)
    minima = frame.loc[frame.groupby(["target", "scheme", "lambda_CVaR"])["Phi_RRE"].idxmin(),
                       ["target", "scheme", "lambda_CVaR", "method"]].rename(columns={"method": "best_method"})
    return frame.merge(minima, on=["target", "scheme", "lambda_CVaR"], how="left")


def uncertainty_severity_diagnostic(method_designs, cfg=CFG, ctx=TRUE_CTX):
    methods = ["Proposed RRE-optimality", "V-optimality", "Cost-matched V-optimality"]
    mapping = method_designs[method_designs["method"].isin(methods)][
        ["target", "method", "design_id"]
    ].drop_duplicates()
    design_lookup = method_designs.drop_duplicates("design_id").set_index("design_id")
    rows = []
    for severity in (0.0, 0.5, 1.0, 1.5):
        scenarios = build_scenarios(severity)
        for did in sorted(mapping["design_id"].unique()):
            design = row_to_design(design_lookup.loc[did])
            result = evaluate_rre(
                design, cfg.severity_diagnostic_reps,
                f"severity_diagnostic_{severity:.2f}", cfg, ctx, scenarios,
            )
            for target in TARGETS:
                rows.append({
                    "design_id": did, "target": target,
                    "uncertainty_severity": severity,
                    "Phi_RRE": result["summary"][f"Phi_RRE_{target}"],
                    "Phi_RRE_conditional": result["summary"][f"Phi_RRE_conditional_{target}"],
                    "RRE_eligible": result["summary"][f"RRE_eligible_{target}"],
                    "minimum_fit_fraction": result["summary"][f"min_fit_fraction_{target}"],
                    "diagnostic_reps": cfg.severity_diagnostic_reps,
                    "interpretation": "fixed-plan severity stress test; no re-optimization",
                })
    frame = pd.DataFrame(rows).merge(
        mapping, on=["target", "design_id"], how="inner", validate="many_to_many"
    )
    pivot = frame.pivot_table(
        index=["target", "uncertainty_severity"], columns="method",
        values="Phi_RRE", aggfunc="first",
    ).reset_index()
    if {"Proposed RRE-optimality", "V-optimality"}.issubset(pivot.columns):
        pivot["Improvement_vs_V_pct"] = 100 * (
            pivot["V-optimality"] - pivot["Proposed RRE-optimality"]
        ) / pivot["V-optimality"].clip(lower=1e-15)
    return frame, pivot


def selection_stability_bootstrap(refine_raw, cfg=CFG):
    rng = rng_from_seed(MASTER_SEED + 1707)
    rows = []
    for target in TARGETS:
        data = refine_raw[refine_raw["target"] == target]
        design_ids = sorted(data["design_id"].unique())
        scenario_indices = [
            i for i in sorted(data["scenario_index"].unique())
            if float(SCENARIOS[int(i)]["weight"]) > 0
        ]
        rep_values = sorted(data["rep"].unique())
        weights = normalize_weights([SCENARIOS[i]["weight"] for i in scenario_indices])
        wins = {did: 0 for did in design_ids}
        rank_sums = {did: 0.0 for did in design_ids}
        boot_reps = min(cfg.bootstrap_reps, 1000 if RUN_MODE == "paper" else 300)
        for _ in range(boot_reps):
            sampled = rng.choice(rep_values, size=len(rep_values), replace=True)
            phi_values = {}
            for did in design_ids:
                design_data = data[data["design_id"] == did]
                losses = []
                fit_ok = True
                for scenario_index in scenario_indices:
                    values = design_data[design_data["scenario_index"] == scenario_index].set_index("rep")["squared_error"]
                    sampled_values = values.loc[sampled]
                    fit_fraction = float(sampled_values.notna().mean())
                    fit_ok &= fit_fraction >= cfg.minimum_valid_fit_fraction
                    losses.append(float(sampled_values.mean()))
                phi_values[did] = (
                    phi_from_losses(losses, weights, cfg)
                    if fit_ok and np.isfinite(losses).all() else np.inf
                )
            ordered = sorted(phi_values, key=phi_values.get)
            wins[ordered[0]] += 1
            for rank, did in enumerate(ordered, start=1):
                rank_sums[did] += rank
        for did in design_ids:
            rows.append({
                "target": target, "design_id": did,
                "selection_probability": wins[did] / boot_reps,
                "mean_bootstrap_rank": rank_sums[did] / boot_reps,
                "bootstrap_reps": boot_reps,
            })
    return pd.DataFrame(rows)


## 10. Figures, exports and primary analysis


In [ ]:
def plot_convergence(screen_frame, target, cfg=CFG):
    ordered = screen_frame.sort_values("evaluation_order")
    values = ordered[f"Phi_RRE_{target}"].to_numpy(float)
    running = np.minimum.accumulate(values)
    x = np.arange(1, len(values) + 1)

    fig, ax = plt.subplots(figsize=(6.15, 3.85))
    positive = values[np.isfinite(values) & (values > 0)]
    floor = max(float(np.min(positive)) * 0.8, 1e-10)
    plot_values = np.maximum(values, floor)
    plot_running = np.maximum(running, floor)

    ax.scatter(
        x, plot_values,
        s=9,
        facecolor=PALETTE["light_gray"],
        edgecolor="none",
        alpha=0.42,
        rasterized=True,
        label="Evaluated designs",
        zorder=1,
    )
    ax.plot(
        x, plot_running,
        color=PALETTE["navy"],
        linewidth=1.75,
        label="Best-so-far criterion",
        zorder=3,
    )
    ax.scatter(
        [x[-1]], [plot_running[-1]],
        s=34,
        marker="o",
        facecolor=PALETTE["red"],
        edgecolor="white",
        linewidth=0.8,
        zorder=4,
    )
    ax.annotate(
        f"{plot_running[-1]:.2g}",
        xy=(x[-1], plot_running[-1]),
        xytext=(-8, 8),
        textcoords="offset points",
        ha="right", va="bottom",
        fontsize=7.5,
        color=PALETTE["red"],
    )
    ax.set_yscale("log")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    style_axis(
        ax,
        "Screened design evaluations",
        r"Estimated $\Phi_{RRE}$",
        TARGETS.get(target, {}).get("label", str(target)),
        grid="y",
    )
    ax.legend(loc="upper right", handlelength=2.2)
    save_figure(fig, "Fig02_RRE_search_convergence")


def _heatmap_norm(values):
    finite = values[np.isfinite(values)]
    positive = finite[finite > 0]
    if positive.size and float(np.max(positive) / np.min(positive)) >= 30.0:
        return LogNorm(vmin=float(np.min(positive)), vmax=float(np.max(positive)))
    if finite.size:
        return Normalize(vmin=float(np.min(finite)), vmax=float(np.max(finite)))
    return None


def _annotated_matrix(ax, matrix, row_labels, col_labels, cmap="cividis", fmt=".3g",
                      colorbar_label=None):
    values = np.asarray(matrix, float)
    norm = _heatmap_norm(values)
    image = ax.imshow(values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.tick_params(axis="x", rotation=0)

    ax.set_xticks(np.arange(-0.5, len(col_labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.75)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    finite = values[np.isfinite(values)]
    if finite.size:
        threshold = float(np.nanmedian(finite))
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                item = values[i, j]
                if not np.isfinite(item):
                    continue
                if norm is not None:
                    scaled = float(norm(item))
                    text_color = "white" if scaled < 0.40 or scaled > 0.82 else PALETTE["black"]
                else:
                    text_color = "white" if item > threshold else PALETTE["black"]
                ax.text(j, i, format(item, fmt), ha="center", va="center",
                        fontsize=7.2, color=text_color)
    cbar = ax.figure.colorbar(image, ax=ax, fraction=0.035, pad=0.025)
    if colorbar_label:
        cbar.set_label(colorbar_label, rotation=90, labelpad=7)
    cbar.ax.tick_params(labelsize=7.3)
    return image


def heatmap(frame, value, filename, title, fmt=".3g"):
    pivot = frame.pivot(index="method", columns="target", values=value)
    method_order = ordered_methods(pivot.index)
    target_order = [t for t in TARGETS if t in pivot.columns]
    pivot = pivot.reindex(index=method_order, columns=target_order)

    row_labels = [method_style(m)["label"] for m in pivot.index]
    col_labels = [TARGETS.get(c, {}).get("label", c) for c in pivot.columns]
    fig, ax = plt.subplots(figsize=(7.6, max(3.5, 0.46 * len(pivot) + 1.15)))
    _annotated_matrix(
        ax,
        pivot.to_numpy(float),
        row_labels,
        col_labels,
        cmap="cividis",
        fmt=fmt,
        colorbar_label=r"$\Phi_{RRE}$",
    )
    ax.set_title(title, pad=7)
    ax.set_xlabel("Reliability target")
    ax.set_ylabel(None)
    save_figure(fig, filename)


def plot_selected_resources(method_designs):
    primary = method_designs[method_designs["target"] == CFG.primary_target].copy()
    primary["method"] = pd.Categorical(primary["method"], METHOD_ORDER, ordered=True)
    primary = primary.sort_values("method").dropna(subset=["method"])
    methods = [str(v) for v in primary["method"]]
    labels = [method_style(m)["label"] for m in methods]
    y = np.arange(len(primary))

    fig, axes = plt.subplots(
        1, 3, figsize=(11.2, max(4.4, 0.48 * len(primary) + 1.25)), sharey=True
    )

    ax = axes[0]
    values = primary["Tmin"].astype(float).to_numpy()
    baseline = max(0.0, float(np.min(values)) - 2.0)
    for yi, method, value in zip(y, methods, values):
        style = method_style(method)
        ax.hlines(yi, baseline, value, color=PALETTE["light_gray"], linewidth=1.25, zorder=1)
        ax.scatter(
            value, yi,
            s=58 if method == "Proposed RRE-optimality" else 42,
            marker=style["marker"],
            facecolor=style["color"],
            edgecolor="white",
            linewidth=0.75,
            zorder=3,
        )
        ax.annotate(
            f"{value:.0f}", xy=(value, yi), xytext=(5, 0),
            textcoords="offset points", va="center", ha="left",
            fontsize=7.2, color=PALETTE["black"],
        )
    ax.set_ylim(len(primary) - 0.45, -0.55)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    style_axis(ax, r"Minimum stress $T_{\min}$ ($^\circ$C)", None, None, grid="x")
    panel_label(ax, "(a)", x=-0.12, y=1.03)

    ax = axes[1]
    allocation_cols = ["n1", "n2", "n3"]
    allocation_labels = ["Low stress", "Intermediate stress", "High stress"]
    allocation_colors = [PALETTE["blue"], PALETTE["teal"], PALETTE["amber"]]
    left = np.zeros(len(primary), float)
    for column, label, color in zip(allocation_cols, allocation_labels, allocation_colors):
        vals = primary[column].astype(float).to_numpy()
        ax.barh(
            y, vals, left=left, height=0.64,
            color=color, edgecolor="white", linewidth=0.55, label=label,
        )
        for yi, lft, val in zip(y, left, vals):
            if val >= 3:
                ax.text(
                    lft + val / 2.0, yi, f"{int(val)}",
                    ha="center", va="center", fontsize=6.8,
                    color="white" if color != PALETTE["amber"] else PALETTE["black"],
                )
        left += vals
    ax.set_xlim(0, max(float(CFG.N_ub), float(left.max())) * 1.02)
    ax.tick_params(labelleft=False)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    style_axis(ax, "Sample allocation", None, None, grid="x")
    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, 1.12),
        ncol=3, fontsize=6.8, handlelength=1.4, columnspacing=0.9,
    )
    panel_label(ax, "(b)", x=-0.12, y=1.03)

    ax = axes[2]
    costs = primary["Cost"].astype(float).to_numpy()
    for yi, method, value in zip(y, methods, costs):
        style = method_style(method)
        ax.hlines(yi, 0.0, value, color=PALETTE["light_gray"], linewidth=1.25, zorder=1)
        ax.scatter(
            value, yi,
            s=58 if method == "Proposed RRE-optimality" else 42,
            marker=style["marker"],
            facecolor=style["color"],
            edgecolor="white",
            linewidth=0.75,
            zorder=3,
        )
        ax.annotate(
            f"{value:,.0f}", xy=(value, yi), xytext=(5, 0),
            textcoords="offset points", va="center", ha="left",
            fontsize=7.0, color=PALETTE["black"],
        )
    ax.axvline(
        CFG.comparison_budget, color=PALETTE["black"],
        linestyle="--", linewidth=0.9, alpha=0.75, label="Budget cap",
    )
    ax.set_xlim(0, CFG.comparison_budget * 1.09)
    ax.tick_params(labelleft=False)
    style_axis(ax, "Total test cost", None, None, grid="x")
    ax.legend(loc="upper right", fontsize=6.9)
    panel_label(ax, "(c)", x=-0.12, y=1.03)

    fig.suptitle(
        TARGETS.get(CFG.primary_target, {}).get("label", CFG.primary_target),
        y=1.015, fontsize=9.4,
    )
    save_figure(fig, "Fig03_primary_target_designs")

def plot_category_validation(scenario_frame):
    category_plot = scenario_frame.groupby(
        ["target", "method", "Category"], as_index=False
    )["MSE_transformed"].mean()
    primary = category_plot[category_plot["target"] == CFG.primary_target].copy()
    pivot = primary.pivot(index="method", columns="Category", values="MSE_transformed")
    method_order = ordered_methods(pivot.index)
    pivot = pivot.reindex(index=method_order)

    row_labels = [method_style(m)["label"] for m in pivot.index]
    col_labels = [wrapped_label(c, width=15) for c in pivot.columns]
    fig, ax = plt.subplots(figsize=(9.2, max(3.7, 0.47 * len(pivot) + 1.25)))
    _annotated_matrix(
        ax,
        pivot.to_numpy(float),
        row_labels,
        col_labels,
        cmap="magma_r",
        fmt=".2g",
        colorbar_label="Mean transformed-scale MSE",
    )
    ax.set_title(
        f"Uncertainty-category validation — {TARGETS.get(CFG.primary_target, {}).get('label', CFG.primary_target)}",
        pad=7,
    )
    ax.set_xlabel("Uncertainty category")
    save_figure(fig, "Fig06_category_validation")


def plot_bias_variance(scenario_frame):
    data = scenario_frame[
        (scenario_frame["target"] == CFG.primary_target)
        & (scenario_frame["method"].isin(["Proposed RRE-optimality", "V-optimality"]))
    ].copy()
    categories = list(dict.fromkeys(data.sort_values("scenario_index")["Category"]))
    methods = ["Proposed RRE-optimality", "V-optimality"]

    fig, axes = plt.subplots(1, 2, figsize=(10.4, 5.0), sharex=True, sharey=True)
    y = np.arange(len(categories))
    panel_values = []

    for idx, (ax, method) in enumerate(zip(axes, methods)):
        group = data[data["method"] == method].groupby("Category")
        bias2 = np.array([
            group.get_group(c)["Bias_transformed"].pow(2).mean()
            for c in categories
        ])
        variance = np.array([
            group.get_group(c)["Variance_transformed"].mean()
            for c in categories
        ])
        panel_values.append((variance, bias2))

        ax.barh(
            y, variance,
            color=PALETTE["navy"],
            edgecolor="white", linewidth=0.55,
            height=0.68,
            label="Variance",
        )
        ax.barh(
            y, bias2, left=variance,
            color=PALETTE["amber"],
            edgecolor="white", linewidth=0.55,
            hatch="////",
            height=0.68,
            label=r"Bias$^2$",
        )
        ax.set_yticks(y)
        ax.set_yticklabels([wrapped_label(c, width=19) for c in categories])
        ax.invert_yaxis()
        style_axis(
            ax,
            "Transformed-scale loss",
            None,
            method_style(method)["label"],
            grid="x",
        )
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((-2, 3))
        ax.xaxis.set_major_formatter(formatter)
        panel_label(ax, f"({chr(97 + idx)})", x=-0.12, y=1.03)

    global_max = max(float(np.max(v + b)) for v, b in panel_values)
    label_room = max(global_max * 0.30, 0.012)
    for ax, (variance, bias2) in zip(axes, panel_values):
        ax.set_xlim(0.0, global_max + label_room)
        for yi, (var_value, bias_value) in enumerate(zip(variance, bias2)):
            total = var_value + bias_value
            ax.text(
                total + 0.015 * max(global_max, 1e-12),
                yi,
                rf"$V={var_value:.3g}$" + "\n" + rf"$B^2={bias_value:.3g}$",
                ha="left", va="center", fontsize=6.6,
                color=PALETTE["black"], linespacing=0.95,
            )

    handles = [
        Patch(facecolor=PALETTE["navy"], edgecolor="white", label="Variance"),
        Patch(
            facecolor=PALETTE["amber"], edgecolor="white",
            hatch="////", label=r"Bias$^2$",
        ),
    ]
    fig.legend(
        handles=handles, loc="upper center",
        bbox_to_anchor=(0.5, 1.02), ncol=2,
    )
    fig.subplots_adjust(top=0.84, wspace=0.14)
    save_figure(fig, "Fig05_bias_variance_decomposition")

def _curves_identical(data, method_a, method_b, *, rtol=1e-10, atol=1e-12):
    required = {"method", "uncertainty_severity", "Phi_RRE"}
    if not required.issubset(data.columns):
        return False
    a = (
        data[data["method"] == method_a]
        [["uncertainty_severity", "Phi_RRE"]]
        .dropna()
        .sort_values("uncertainty_severity")
        .drop_duplicates("uncertainty_severity", keep="last")
    )
    b = (
        data[data["method"] == method_b]
        [["uncertainty_severity", "Phi_RRE"]]
        .dropna()
        .sort_values("uncertainty_severity")
        .drop_duplicates("uncertainty_severity", keep="last")
    )
    if a.empty or b.empty or len(a) != len(b):
        return False
    merged = a.merge(
        b, on="uncertainty_severity", how="inner",
        suffixes=("_a", "_b"), validate="one_to_one",
    )
    if len(merged) != len(a):
        return False
    return bool(np.allclose(
        merged["Phi_RRE_a"].to_numpy(float),
        merged["Phi_RRE_b"].to_numpy(float),
        rtol=rtol, atol=atol, equal_nan=True,
    ))


def plot_uncertainty_severity(severity_frame):
    fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.7), sharex=True)
    legend_handles = {}

    for idx, (ax, target) in enumerate(zip(axes.flat, TARGETS)):
        data = severity_frame[severity_frame["target"] == target].copy()
        duplicate_v = _curves_identical(data, "V-optimality", "Cost-matched V-optimality")
        methods = ["Proposed RRE-optimality", "V-optimality"]
        if not duplicate_v:
            methods.append("Cost-matched V-optimality")

        for method in methods:
            group = data[data["method"] == method].sort_values("uncertainty_severity")
            if group.empty:
                continue
            style = method_style(method)
            label = style["label"]
            if duplicate_v and method == "V-optimality":
                label = "V-optimality (= cost-matched V)"
            line, = ax.plot(
                group["uncertainty_severity"],
                group["Phi_RRE"],
                color=style["color"],
                linestyle=style["linestyle"],
                marker=style["marker"],
                markerfacecolor="white",
                markeredgewidth=0.9,
                markersize=4.4,
                linewidth=1.55,
                label=label,
                zorder=3 if method == "Proposed RRE-optimality" else 2,
            )
            legend_handles[label] = line

        finite = data["Phi_RRE"].to_numpy(float)
        finite = finite[np.isfinite(finite) & (finite > 0)]
        if finite.size and float(np.max(finite) / np.min(finite)) > 40:
            ax.set_yscale("log")
        else:
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_powerlimits((-2, 3))
            ax.yaxis.set_major_formatter(formatter)

        style_axis(
            ax,
            "Uncertainty severity" if idx >= 2 else None,
            r"Estimated $\Phi_{RRE}$" if idx % 2 == 0 else None,
            TARGETS[target]["label"],
            grid="y",
        )
        panel_label(ax, f"({chr(97 + idx)})", x=-0.12, y=1.03)

    fig.legend(
        handles=list(legend_handles.values()),
        labels=list(legend_handles.keys()),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.025),
        ncol=min(3, len(legend_handles)),
        columnspacing=1.3,
        handlelength=2.4,
    )
    fig.subplots_adjust(top=0.88, hspace=0.24, wspace=0.20)
    save_figure(fig, "Fig07_uncertainty_severity_diagnostic")


def build_self_checks(optimization, method_designs, validation_summary, scenario_validation, paired,
                      selection_stability, cfg=CFG):
    checks = []
    def add(name, passed, value, threshold, critical=True):
        checks.append({"Check": name, "Passed": bool(passed), "Value": value,
                       "Threshold": threshold, "Critical": bool(critical)})
    add("Scenario weights sum to one", abs(sum(SCENARIO_WEIGHTS) - 1) < 1e-12,
        float(sum(SCENARIO_WEIGHTS)), "=1")
    add("All four use-condition targets are finite", all(
        np.isfinite(target_truth(s, t, cfg, TRUE_CTX)) for s in SCENARIOS for t in TARGETS
    ), True, "True")
    add("Selected designs obey the hard budget", bool((method_designs["Cost"] <= cfg.comparison_budget + 1e-8).all()),
        float(method_designs["Cost"].max()), f"<={cfg.comparison_budget}")
    resource_ok = bool(
        (method_designs["N"] == cfg.N_lb).all()
        and (method_designs["m"] == cfg.m_lb).all()
    )
    add("All criteria use identical fixed N and m", resource_ok,
        f"N={cfg.N_lb}, m={cfg.m_lb}", "fixed and identical")
    v_gaps = []
    for target in TARGETS:
        selected_v = method_designs[
            (method_designs["target"] == target)
            & (method_designs["method"] == "V-optimality")
        ]["V_target"].iloc[0]
        v_gaps.append(float(selected_v - optimization["pool"][f"V_{target}"].min()))
    add("V baselines are exact optima on the enumerated domain",
        max(abs(v) for v in v_gaps) < 1e-12, max(abs(v) for v in v_gaps), "<1e-12")
    match_gaps = []
    for target in TARGETS:
        target_rows = method_designs[method_designs["target"] == target].set_index("method")
        match_gaps.append(abs(
            float(target_rows.loc["Cost-matched V-optimality", "Cost"])
            - float(target_rows.loc["Proposed RRE-optimality", "Cost"])
        ) / float(target_rows.loc["Proposed RRE-optimality", "Cost"]))
    add("Cost-matched V lies in the predeclared band", max(match_gaps) <= cfg.matched_cost_tolerance_ratio + 1e-12,
        max(match_gaps), f"<={cfg.matched_cost_tolerance_ratio}")
    add("Selected FIMs are positive definite", bool((method_designs["FIM_min_eig"] > 0).all()),
        float(method_designs["FIM_min_eig"].min()), ">0")
    add("Independent validation fit fraction", bool(
        validation_summary["Minimum_fit_valid_fraction"].min() >= cfg.minimum_valid_fit_fraction
    ), float(validation_summary["Minimum_fit_valid_fraction"].min()), f">={cfg.minimum_valid_fit_fraction}")
    add("Proposed designs passed independent confirmation", bool(
        method_designs[method_designs["method"] == "Proposed RRE-optimality"]["Phi_RRE_confirmation"].notna().all()
    ), int(method_designs[method_designs["method"] == "Proposed RRE-optimality"]["Phi_RRE_confirmation"].notna().sum()), "=4")
    add("Matched-cost comparison pool is nontrivial", len(optimization["pool"]) >= 20,
        len(optimization["pool"]), ">=20")
    add("Tmin search covers multiple feasible values", len(optimization["feasible_Tmin"]) >= 5,
        len(optimization["feasible_Tmin"]), ">=5")
    expected_baselines = method_designs["method"].nunique() - 1
    expected_pairs = len(TARGETS) * expected_baselines
    add("Paired comparisons complete", len(paired) == expected_pairs, len(paired), f"={expected_pairs}")
    proposed_ids = method_designs[method_designs["method"] == "Proposed RRE-optimality"][
        ["target", "design_id"]
    ]
    selected_stability = proposed_ids.merge(
        selection_stability, on=["target", "design_id"], how="left", validate="one_to_one"
    )
    minimum_selection_probability = float(selected_stability["selection_probability"].min())
    add("Refined-design stochastic selection probability", minimum_selection_probability >= 0.50,
        minimum_selection_probability, ">=0.50", critical=False)
    true_model_rows = scenario_validation[scenario_validation["Scenario"] == "True IG model"]
    decomposition_gap = np.nanmax(np.abs(
        true_model_rows["MSE_transformed"] - (
            true_model_rows["Bias_transformed"] ** 2
            + true_model_rows["Variance_transformed"] * (cfg.validation_reps - 1) / max(cfg.validation_reps, 1)
        )
    ))
    add("True-model MSE bias-variance identity", decomposition_gap < 1e-8,
        float(decomposition_gap), "<1e-8", critical=False)

    true_mission_reliability = float(
        target_truth(SCENARIOS[0], "mission_R", cfg, TRUE_CTX)
    )
    expected_mission_h = float(cfg.mission_years * cfg.hours_per_year)
    add(
        f"Mission time equals the predeclared {cfg.mission_years:g}-year horizon",
        abs(cfg.mission_time_h - expected_mission_h) < 1e-8,
        cfg.mission_time_h,
        f"={expected_mission_h}",
    )
    add(
        f"True-model reliability at {cfg.mission_years:g} years is a valid probability",
        0.0 < true_mission_reliability < 1.0,
        true_mission_reliability,
        "in (0,1)",
    )

    beta_ratio_100 = float(selected_beta_ratio(cfg.Tmax, cfg, TRUE_CTX))
    aft_100 = float(selected_time_acceleration_factor(cfg.Tmax, cfg, TRUE_CTX))
    aft_identity_gap = abs(
        aft_100 - beta_ratio_100 ** (1.0 / float(TRUE_CTX["q"]))
    )
    add(
        "Calendar-time acceleration-factor identity",
        aft_identity_gap < 1e-10,
        aft_identity_gap,
        "<1e-10",
    )

    equal_rows = method_designs[
        method_designs["method"] == "Conventional equal-allocation"
    ]
    lowest_feasible_tmin = int(optimization["pool"]["Tmin"].min())
    equal_ok = bool(
        len(equal_rows) == len(TARGETS)
        and (equal_rows["Tmin"] == lowest_feasible_tmin).all()
        and (equal_rows[["n1", "n2", "n3"]].to_numpy() == np.array([20, 20, 20])).all()
    )
    add(
        "Equal-allocation engineering baseline uses the lowest feasible Tmin",
        equal_ok,
        int(equal_ok),
        "=1",
    )

    probe_low = constraint_report(
        {"Tmin": lowest_feasible_tmin, "N": cfg.N_lb, "m": cfg.m_lb, "Ni": [20, 20, 20]},
        cfg,
        TRUE_CTX,
    )
    probe_prev = constraint_report(
        {"Tmin": lowest_feasible_tmin - 1, "N": cfg.N_lb, "m": cfg.m_lb, "Ni": [20, 20, 20]},
        cfg,
        TRUE_CTX,
    ) if lowest_feasible_tmin > cfg.Tmin_lb else None
    boundary_ok = bool(
        probe_low["feasible"]
        and probe_low["t_campaign"] <= cfg.tmax_allow
        and (probe_prev is None or probe_prev["t_campaign"] > cfg.tmax_allow)
    )
    add(
        f"Lowest feasible Tmin is generated by the {cfg.tmax_allow:g} h campaign limit",
        boundary_ok,
        {"Tmin": lowest_feasible_tmin, "campaign_h": float(probe_low["t_campaign"])},
        f"campaign<={cfg.tmax_allow} and previous stress exceeds the limit",
    )
    return pd.DataFrame(checks)


def make_conclusion(validation_summary, paired, checks, cfg=CFG):
    lines = []
    for target in TARGETS:
        subset = validation_summary[validation_summary["target"] == target]
        proposed = subset[subset["method"] == "Proposed RRE-optimality"].iloc[0]
        best_baseline = subset[subset["method"] != "Proposed RRE-optimality"].nsmallest(1, "Phi_RRE_validation").iloc[0]
        gain = 100 * (best_baseline["Phi_RRE_validation"] - proposed["Phi_RRE_validation"]) / max(best_baseline["Phi_RRE_validation"], 1e-15)
        v_pair = paired[(paired["target"] == target) & (paired["baseline"] == "V-optimality")]
        ci_text = "not available"
        if len(v_pair):
            item = v_pair.iloc[0]
            ci_text = f"{item['Improvement_pct']:.1f}% (95% paired bootstrap CI {item['CI95_low_pct']:.1f}% to {item['CI95_high_pct']:.1f}%)"
        lines.append(
            f"{target}: proposed Phi_RRE={proposed['Phi_RRE_validation']:.6g}; "
            f"best baseline={best_baseline['method']} ({best_baseline['Phi_RRE_validation']:.6g}); "
            f"relative gain vs best baseline={gain:.1f}%; gain vs V={ci_text}."
        )
    critical_pass = bool(checks.loc[checks["Critical"], "Passed"].all())
    v_rows = paired[paired["baseline"] == "V-optimality"]
    all_v_point_better = bool(len(v_rows) == len(TARGETS) and (v_rows["Improvement_pct"] > 0).all())
    all_v_ci_better = bool(len(v_rows) == len(TARGETS) and (v_rows["CI95_low_pct"] > 0).all())
    if RUN_MODE == "paper" and critical_pass and all_v_ci_better:
        status = "SUPPORTED: predeclared paper-mode evidence supports superiority over V-optimality for all targets."
    elif RUN_MODE == "paper" and critical_pass and all_v_point_better:
        status = "PARTIALLY SUPPORTED: point estimates improve on V, but at least one paired interval crosses zero."
    elif RUN_MODE == "audit":
        status = "AUDIT ONLY: logic and execution are checked; numerical superiority claims require paper mode."
    else:
        status = "NOT SUPPORTED AS A UNIVERSAL CLAIM: report target-specific results and their uncertainty."
    text = status + "\n" + "\n".join(lines)
    return status, text


def save_outputs(optimization, method_designs, validation_unique, validation_mapped,
                 validation_summary, scenario_validation, paired, sensitivity,
                 selection_stability, severity_frame, severity_comparison,
                 checks, conclusion):
    tmin_table = tmin_feasibility_table(CFG, TRUE_CTX)
    parameter_table = pd.DataFrame([{
        "alpha0": TRUE_CTX["alpha0"],
        "alpha1_data_fitted": ALPHA1_FITTED,
        "activation_energy_data_fitted_eV": EA_DATA_FITTED_EV,
        "activation_energy_minus_deviation_eV": EA_DATA_LOWER_EV,
        "activation_energy_plus_deviation_eV": EA_DATA_UPPER_EV,
        "activation_energy_relative_deviation": CFG.ea_relative_deviation,
        "alpha1_selected": TRUE_CTX["alpha1"],
        "eta": TRUE_CTX["eta"], "q": TRUE_CTX["q"],
        "AF_method": "real-data fitted Arrhenius relation",
        "Beta_ratio_Tmax": float(selected_beta_ratio(CFG.Tmax)),
        "AFt_Tmax": float(selected_time_acceleration_factor(CFG.Tmax)),
        "mission_years": float(CFG.mission_years),
        "mission_time_h": float(CFG.mission_time_h),
        "B10_true_h": float(B10_TRUE_H),
        "true_model_R_at_mission_time": float(
            target_truth(SCENARIOS[0], "mission_R", CFG, TRUE_CTX)
        ),
        "campaign_time_limit_h": float(CFG.tmax_allow),
        "NLL": REAL_FIT["nll"], "n_positive_increments": len(DY_REAL),
        **{f"truth_{target}": target_truth(SCENARIOS[0], target, CFG, TRUE_CTX) for target in TARGETS},
    }])
    scenario_catalog = pd.DataFrame(SCENARIOS)
    config_table = pd.DataFrame([asdict(CFG)])
    config_table.to_csv(OUT_DIR / "Table00_configuration.csv", index=False, encoding="utf-8-sig")
    scenario_catalog.to_csv(OUT_DIR / "Table00b_predeclared_scenarios.csv", index=False, encoding="utf-8-sig")
    tmin_table.to_csv(OUT_DIR / "Table00c_Tmin_feasibility.csv", index=False, encoding="utf-8-sig")
    parameter_table.to_csv(OUT_DIR / "Table01_parameter_and_target_estimates.csv", index=False, encoding="utf-8-sig")
    method_designs.to_csv(OUT_DIR / "Table02_selected_designs_all_targets.csv", index=False, encoding="utf-8-sig")
    validation_summary.to_csv(OUT_DIR / "Table03_independent_validation_summary.csv", index=False, encoding="utf-8-sig")
    fisher_nesting = scenario_validation[scenario_validation["Scenario"] == "True IG model"][
        ["target", "method", "MSE_transformed", "Bias_transformed", "Variance_transformed",
         "Fit_valid_fraction"]
    ].merge(
        method_designs[["target", "method", "V_target", "design_id"]],
        on=["target", "method"], how="left", validate="one_to_one",
    )
    fisher_nesting["finite_sample_MSE_to_asymptotic_V_ratio"] = (
        fisher_nesting["MSE_transformed"] / fisher_nesting["V_target"].clip(lower=1e-15)
    )
    fisher_nesting.to_csv(
        OUT_DIR / "Table03b_Fisher_nesting_diagnostic.csv", index=False, encoding="utf-8-sig"
    )
    scenario_validation.to_csv(OUT_DIR / "Table04_scenario_bias_variance.csv", index=False, encoding="utf-8-sig")
    scenario_validation[scenario_validation["role"] == "external_stress_test"].to_csv(
        OUT_DIR / "Table04b_external_process_family_stress_test.csv",
        index=False, encoding="utf-8-sig",
    )
    paired.to_csv(OUT_DIR / "Table05_paired_improvement_vs_baselines.csv", index=False, encoding="utf-8-sig")
    sensitivity.to_csv(OUT_DIR / "Table06_weight_and_CVaR_sensitivity.csv", index=False, encoding="utf-8-sig")
    selection_stability.to_csv(
        OUT_DIR / "Table06b_selection_stability_bootstrap.csv", index=False, encoding="utf-8-sig"
    )
    severity_frame.to_csv(
        OUT_DIR / "Table06c_uncertainty_severity_fixed_plan.csv",
        index=False, encoding="utf-8-sig",
    )
    severity_comparison.to_csv(
        OUT_DIR / "Table06d_uncertainty_severity_comparison.csv",
        index=False, encoding="utf-8-sig",
    )
    checks.to_csv(OUT_DIR / "Table07_self_checks.csv", index=False, encoding="utf-8-sig")
    optimization["candidates"].to_csv(OUT_DIR / "optimization_candidate_archive.csv", index=False, encoding="utf-8-sig")
    optimization["screen"].to_csv(OUT_DIR / "RRE_screen_archive.csv", index=False, encoding="utf-8-sig")
    optimization["refine"].to_csv(OUT_DIR / "RRE_refinement_archive.csv", index=False, encoding="utf-8-sig")
    optimization["refine_scenarios"].to_csv(OUT_DIR / "RRE_refinement_scenario_losses.csv", index=False, encoding="utf-8-sig")
    optimization["refine_raw"].to_csv(OUT_DIR / "RRE_refinement_raw_replications.csv", index=False, encoding="utf-8-sig")
    optimization["confirmation"].to_csv(
        OUT_DIR / "RRE_confirmation_archive.csv", index=False, encoding="utf-8-sig"
    )
    optimization["confirmation_scenarios"].to_csv(
        OUT_DIR / "RRE_confirmation_scenario_losses.csv", index=False, encoding="utf-8-sig"
    )
    optimization["confirmation_raw"].to_csv(
        OUT_DIR / "RRE_confirmation_raw_replications.csv", index=False, encoding="utf-8-sig"
    )
    validation_unique.to_csv(OUT_DIR / "independent_validation_unique_designs_raw.csv", index=False, encoding="utf-8-sig")
    validation_mapped.to_csv(OUT_DIR / "independent_validation_method_mapped_raw.csv", index=False, encoding="utf-8-sig")
    (OUT_DIR / "FINAL_CONCLUSION.txt").write_text(conclusion, encoding="utf-8")
    result_map = pd.DataFrame([
        {"Claim": "Criterion is target-specific and contains variance plus extrapolation bias",
         "Primary evidence": "Table04_scenario_bias_variance.csv; Fig05_bias_variance_decomposition.pdf"},
        {"Claim": "V-optimality is the correctly specified asymptotic special case",
         "Primary evidence": "Table03b_Fisher_nesting_diagnostic.csv"},
        {"Claim": "RRE-optimality is compared fairly under one hard budget",
         "Primary evidence": "Table02_selected_designs_all_targets.csv (fixed N,m; exact V and cost-matched V); Table07_self_checks.csv"},
        {"Claim": "Advantage is not restricted to minimum-stress selection or q=0.1 life",
         "Primary evidence": "Table03_independent_validation_summary.csv; Fig04_multi_target_validation.pdf"},
        {"Claim": "Uncertainty of superiority is quantified",
         "Primary evidence": "Table05_paired_improvement_vs_baselines.csv; Table06b_selection_stability_bootstrap.csv"},
        {"Claim": "Scenario/CVaR assumptions are transparent",
         "Primary evidence": "Table00b_predeclared_scenarios.csv; Table06_weight_and_CVaR_sensitivity.csv"},
        {"Claim": "Any RRE advantage is linked to extrapolation-uncertainty severity",
         "Primary evidence": "Table06c_uncertainty_severity_fixed_plan.csv; Table06d_uncertainty_severity_comparison.csv; Fig07_uncertainty_severity_diagnostic.pdf"},
    ])
    result_map.to_csv(OUT_DIR / "result_to_claim_map.csv", index=False, encoding="utf-8-sig")


def write_manifest(start_time, status):
    hashes = {}
    for path in sorted(OUT_DIR.glob("*")):
        if path.is_file():
            hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {
        "status": status, "run_mode": RUN_MODE, "master_seed": MASTER_SEED,
        "elapsed_seconds": time.time() - start_time,
        "python": sys.version, "platform": platform.platform(),
        "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
        "matplotlib": matplotlib.__version__, "config": asdict(CFG),
        "scenario_weights": {s["Scenario"]: s["weight"] for s in SCENARIOS},
        "output_sha256": hashes,
    }
    if "__file__" in globals():
        source_path = Path(__file__).resolve()
        if source_path.exists():
            manifest["source_file"] = str(source_path)
            manifest["source_sha256"] = hashlib.sha256(source_path.read_bytes()).hexdigest()
    (OUT_DIR / "reproducibility_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def main():
    start = time.time()
    print(f"RUN_MODE={RUN_MODE}; output={OUT_DIR.resolve()}")
    print(
        "True acceleration relation: real-data fitted Arrhenius; "
        f"Ea_fit={EA_DATA_FITTED_EV:.6g} eV; "
        f"Ea deviations=({EA_DATA_LOWER_EV:.6g}, {EA_DATA_UPPER_EV:.6g}) eV"
    )
    print(
        f"beta-ratio AF at 100C={float(selected_beta_ratio(CFG.Tmax)):.6g}; "
        f"calendar-time AF_t at 100C={float(selected_time_acceleration_factor(CFG.Tmax)):.6g}; "
        f"t_m({CFG.mission_years:g} years)={CFG.mission_time_h:.6g} h; "
        f"R_true({CFG.mission_years:g} years)="
        f"{target_truth(SCENARIOS[0], 'mission_R', CFG, TRUE_CTX):.6g}"
    )
    plot_connector_data()
    optimization = run_three_stage_optimization(CFG, TRUE_CTX)
    method_designs = select_method_designs(optimization, CFG)
    validation_unique, validation_mapped = validate_unique_designs(method_designs, CFG, TRUE_CTX)
    validation_summary, scenario_validation = aggregate_method_validation(validation_mapped, CFG)
    paired = paired_improvement_table(validation_mapped, CFG)
    sensitivity = validation_weight_sensitivity(scenario_validation, CFG)
    selection_stability = selection_stability_bootstrap(
        optimization["confirmation_raw"], CFG
    )
    severity_frame, severity_comparison = uncertainty_severity_diagnostic(
        method_designs, CFG, TRUE_CTX
    )
    checks = build_self_checks(
        optimization, method_designs, validation_summary, scenario_validation, paired,
        selection_stability, CFG
    )
    status, conclusion = make_conclusion(validation_summary, paired, checks, CFG)

    plot_convergence(optimization["screen"], CFG.primary_target, CFG)
    plot_selected_resources(method_designs)
    heatmap(validation_summary, "Phi_RRE_validation", "Fig04_multi_target_validation",
            r"Independent validation: $\Phi_{RRE}$ (lower is better)")
    plot_bias_variance(scenario_validation)
    plot_category_validation(scenario_validation)
    plot_uncertainty_severity(severity_frame)

    save_outputs(
        optimization, method_designs, validation_unique, validation_mapped,
        validation_summary, scenario_validation, paired, sensitivity,
        selection_stability, severity_frame, severity_comparison,
        checks, conclusion,
    )
    write_manifest(start, status)
    display(method_designs)
    display(validation_summary)
    display(paired)
    display(checks)
    print("\n" + conclusion)
    print(f"\nElapsed: {time.time() - start:.1f} s")
    print(f"Output directory: {OUT_DIR.resolve()}")
    return {
        "optimization": optimization, "method_designs": method_designs,
        "validation_summary": validation_summary, "scenario_validation": scenario_validation,
        "paired": paired, "severity_comparison": severity_comparison,
        "checks": checks, "status": status, "conclusion": conclusion,
    }


if __name__ == "__main__":
    RESULTS = main()


## 11. Model-adequacy diagnostics


In [ ]:
import itertools
from scipy.optimize import minimize as review_minimize
from scipy.optimize import minimize_scalar as review_minimize_scalar

REVIEW_MODE = os.environ.get("ADT_REVIEWER_MODE", "paper" if RUN_MODE == "paper" else "audit").lower()
if REVIEW_MODE not in {"audit", "paper"}:
    raise ValueError("ADT_REVIEWER_MODE must be audit or paper")
REVIEW_OUT = OUT_DIR / "reviewer_revision_addendum"
REVIEW_OUT.mkdir(parents=True, exist_ok=True)
REVIEW_SETTINGS = {
    "diagnostic_null_reps": 80 if REVIEW_MODE == "audit" else 400,
    "diagnostic_scenario_reps": 50 if REVIEW_MODE == "audit" else 300,
    "fixed_screen_count": 120 if REVIEW_MODE == "audit" else 150,
    "fixed_screen_reps": 3 if REVIEW_MODE == "audit" else 6,
    "fixed_refine_count": 12 if REVIEW_MODE == "audit" else 40,
    "fixed_refine_reps": 8 if REVIEW_MODE == "audit" else 60,
    "fixed_confirm_reps": 15 if REVIEW_MODE == "audit" else 200,
    "joint_proposals": 300 if REVIEW_MODE == "audit" else 6000,
    "joint_screen_count": 20 if REVIEW_MODE == "audit" else 120,
    "joint_screen_reps": 2 if REVIEW_MODE == "audit" else 5,
    "joint_refine_count": 5 if REVIEW_MODE == "audit" else 20,
    "joint_refine_reps": 5 if REVIEW_MODE == "audit" else 40,
    "joint_validation_reps": 10 if REVIEW_MODE == "audit" else 200,
}


def reviewer_wilson(successes, trials, z=1.959963984540054):
    if trials <= 0:
        return (float("nan"), float("nan"))
    p = successes / trials
    den = 1 + z*z/trials
    center = (p + z*z/(2*trials)) / den
    half = z * math.sqrt(p*(1-p)/trials + z*z/(4*trials*trials)) / den
    return (max(0.0, center-half), min(1.0, center+half))


def reviewer_selected_designs(results=RESULTS):
    frame = results["method_designs"]
    primary = frame[frame["target"] == CFG.primary_target]
    methods = ["Proposed RRE-optimality", "V-optimality", "Cost-matched V-optimality"]
    return {name: row_to_design(primary[primary["method"] == name].iloc[0]) for name in methods}


def reviewer_alternative_nll(x, mode, data, levels, cfg=CFG):
    T, left, right, dy = (np.asarray(v, float) for v in data)
    if mode == "free_stress_beta":
        log_beta = np.array([x[np.argmin(np.abs(levels - t))] for t in T])
        eta = np.exp(x[len(levels)])
        q = np.exp(x[len(levels)+1]) * np.ones_like(T)
    elif mode == "stress_q_trend":
        s = standardized_stress(T, cfg.T0, cfg.Tmax)
        log_beta = x[0] + x[1]*s
        eta = np.exp(x[2])
        q = np.exp(x[3]) * (1 + x[4]*s)
    elif mode == "high_T_shape":
        s = standardized_stress(T, cfg.T0, cfg.Tmax)
        log_beta = x[0] + x[1]*s
        eta = np.exp(np.where(T > 85.0, x[3], x[2]))
        q = np.exp(x[4]) * np.ones_like(T)
    else:
        raise ValueError(mode)
    if np.any(q <= 0.05) or np.any(q >= 3):
        return 1e100
    A = right**q - left**q
    if np.any(A <= 0) or not np.isfinite(A).all():
        return 1e100
    beta = np.exp(np.clip(log_beta, -20, 10))
    value = -np.sum(ig_logpdf(dy, beta*A, eta*A*A))
    return float(value) if np.isfinite(value) else 1e100


def reviewer_lr_statistics(data, design, cfg=CFG):
    null = fit_assumed_ig_with_fallback(data, cfg, init=TRUE_RAW)
    if not null.get("valid", False):
        return None
    levels = np.asarray(planned_schedule(design, cfg, TRUE_CTX)[1], float)
    x0 = np.asarray(null["raw"], float)
    starts = {
        "free_stress_beta": (
            np.r_[x0[0]+x0[1]*standardized_stress(levels, cfg.T0, cfg.Tmax), x0[2], x0[3]],
            [(-20, 10)]*len(levels)+[(-20, 10), (math.log(.05), math.log(3.0))],
        ),
        "stress_q_trend": (np.r_[x0, 0.0],
                           [(-20, 10), (-10, 15), (-20, 10), (math.log(.05), math.log(3.0)), (-.8, .8)]),
        "high_T_shape": (np.r_[x0[0], x0[1], x0[2], x0[2], x0[3]],
                         [(-20, 10), (-10, 15), (-20, 10), (-20, 10), (math.log(.05), math.log(3.0))]),
    }
    answer = {}
    for mode, (start, bounds) in starts.items():
        candidates = []
        for offset in (0.0, 0.1):
            trial = start.copy()
            if offset:
                trial[-1] = np.clip(trial[-1]+offset, bounds[-1][0]+1e-5, bounds[-1][1]-1e-5)
            fit = review_minimize(reviewer_alternative_nll, trial,
                                  args=(mode, data, levels, cfg), method="L-BFGS-B",
                                  bounds=bounds, options={"maxiter": 180, "ftol": 1e-10})
            if np.isfinite(fit.fun):
                candidates.append(fit)
        if not candidates:
            return None
        best = min(candidates, key=lambda item: item.fun)
        if best.fun > null["nll"] + 1e-3:
            return None
        answer[mode] = max(0.0, 2*(float(null["nll"])-float(best.fun)))
    return answer


def reviewer_fast_diagnostic_stats(data, design, cfg=CFG):
    T,left,right,dy=(np.asarray(x,float) for x in data)
    levels=np.asarray(planned_schedule(design,cfg,TRUE_CTX)[1],float)
    group=[]
    for temp in levels:
        mask=np.isclose(T,temp,rtol=0,atol=1e-7)
        if mask.sum()<6:return None
        l,r,y=left[mask],right[mask],dy[mask]
        unique=np.unique(np.round(r,9))
        rows=[]
        for stop in unique:
            interval=np.isclose(r,stop,rtol=0,atol=1e-7)
            if interval.any():
                rows.append((float(l[interval][0]),float(r[interval][0]),
                             float(np.mean(y[interval])),int(interval.sum())))
        if len(rows)<3:return None
        group.append((temp,np.asarray(rows,float),l,r,y))
    def fit_given_q(rows,q):
        A=np.maximum(rows[:,1]**q-rows[:,0]**q,1e-15)
        observed=np.log(np.maximum(rows[:,2],1e-15))-np.log(A)
        weights=rows[:,3]
        intercept=float(np.average(observed,weights=weights))
        return intercept,float(np.sum(weights*(observed-intercept)**2))
    def pooled_sse(q):
        return sum(fit_given_q(item[1],q)[1] for item in group)
    fit=review_minimize_scalar(pooled_sse,bounds=(0.08,2.5),method="bounded",
                               options={"xatol":0.002})
    q_pool=float(fit.x)
    pooled=float(fit.fun)
    separate=0.0
    logbeta=[]
    dispersion=[]
    for temp,rows,l,r,y in group:
        qi=review_minimize_scalar(lambda q:fit_given_q(rows,q)[1],
                                  bounds=(0.08,2.5),method="bounded",
                                  options={"xatol":0.002})
        separate+=float(qi.fun)
        intercept,_=fit_given_q(rows,q_pool)
        logbeta.append(intercept)
        beta=math.exp(np.clip(intercept,-20,10))
        A=np.maximum(r**q_pool-l**q_pool,1e-15)
        mu=beta*A
        dispersion.append(float(np.mean((y-mu)**2/np.maximum(beta**3*A,1e-15))))
    logbeta=np.asarray(logbeta,float)
    s=standardized_stress(levels,cfg.T0,cfg.Tmax)
    units=np.asarray(design["Ni"],float)
    X=np.c_[np.ones(len(levels)),s]
    coef=np.linalg.lstsq(X*np.sqrt(units[:,None]),logbeta*np.sqrt(units),rcond=None)[0]
    lof=float(np.sum(units*(logbeta-X@coef)**2))
    q_heterogeneity=float(max(pooled-separate,0)/max(pooled,1e-12))
    dispersion=np.asarray(dispersion,float)
    high_dispersion=float(abs(np.log(max(dispersion[-1],1e-15)/
                                  max(np.average(dispersion[:-1],weights=units[:-1]),1e-15))))
    answer={"acceleration_LOF":lof,"q_heterogeneity":q_heterogeneity,
            "high_T_dispersion":high_dispersion}
    return answer if np.isfinite(list(answer.values())).all() else None


def reviewer_diagnostic_power(designs=None, cfg=CFG):
    designs = reviewer_selected_designs() if designs is None else designs
    modes = ("acceleration_LOF", "q_heterogeneity", "high_T_dispersion")
    alpha = 0.05
    rows, calibration_rows = [], []
    for method, design in designs.items():
        print(f"R1 diagnostic: {method}", flush=True)
        null_stats = []
        for b in range(REVIEW_SETTINGS["diagnostic_null_reps"]):
            seed = stable_seed(MASTER_SEED, "reviewer_R1_null", method, b)
            data = simulate_design_data(design, SCENARIOS[0], seed, cfg, TRUE_CTX)
            stat = reviewer_fast_diagnostic_stats(data, design, cfg)
            if stat is not None:
                null_stats.append(stat)
        if len(null_stats) < .90*REVIEW_SETTINGS["diagnostic_null_reps"]:
            raise RuntimeError(f"Diagnostic null fit-success below 90% for {method}")
        critical = {mode: float(np.quantile([s[mode] for s in null_stats], 1-alpha/len(modes)))
                    for mode in modes}
        calibration_rows.append({"method": method, "n_null": len(null_stats),
                                 "null_reps": REVIEW_SETTINGS["diagnostic_null_reps"],
                                 **{f"critical_{mode}": critical[mode] for mode in modes}})
        for scenario_index, scenario in enumerate(SCENARIOS[:9]):
            counts = {mode: 0 for mode in modes}
            counts["any"] = 0
            valid = 0
            for b in range(REVIEW_SETTINGS["diagnostic_scenario_reps"]):
                seed = stable_seed(MASTER_SEED, "reviewer_R1_power", method, scenario_index, b)
                data = simulate_design_data(design, scenario, seed, cfg, TRUE_CTX)
                stat = reviewer_fast_diagnostic_stats(data, design, cfg)
                if stat is None:
                    continue
                valid += 1
                flags = {mode: stat[mode] > critical[mode] for mode in modes}
                for mode, flag in flags.items():
                    counts[mode] += int(flag)
                counts["any"] += int(any(flags.values()))
            for mode in (*modes, "any"):
                lo, hi = reviewer_wilson(counts[mode], valid)
                rows.append({"method": method, "scenario_index": scenario_index,
                             "Scenario": scenario["Scenario"], "Category": scenario["Category"],
                             "diagnostic": mode, "detections": counts[mode], "valid": valid,
                             "attempted": REVIEW_SETTINGS["diagnostic_scenario_reps"],
                             "invalid_statistic_fraction": 1-valid/REVIEW_SETTINGS["diagnostic_scenario_reps"],
                             "detection_probability": counts[mode]/valid if valid else np.nan,
                             "wilson_lo": lo, "wilson_hi": hi,
                             "power_lower_if_fail_not_detected": counts[mode]/REVIEW_SETTINGS["diagnostic_scenario_reps"],
                             "power_upper_if_fail_detected":
                             (counts[mode]+REVIEW_SETTINGS["diagnostic_scenario_reps"]-valid)/REVIEW_SETTINGS["diagnostic_scenario_reps"]})
    return pd.DataFrame(rows), pd.DataFrame(calibration_rows)


## 12. Risk sensitivity and robust benchmarks


In [ ]:
def reviewer_primary_scenarios():
    return [s for s in SCENARIOS if float(s.get("weight", 0)) > 0]


def reviewer_weights(power):
    scenarios = reviewer_primary_scenarios()
    base = np.asarray([s["weight"] for s in scenarios], float)
    base /= base.sum()
    equal_departures = np.r_[0.2, np.repeat(0.8/(len(base)-1), len(base)-1)]
    ref = power[(power["method"] == "Neutral reference") & (power["diagnostic"] == "any")]
    detect = {int(r.scenario_index): float(r.detection_probability) for r in ref.itertuples()}
    if len(detect) != len(base) or not np.isfinite(list(detect.values())).all():
        raise RuntimeError("Reference-design detection probabilities incomplete")
    nd = np.zeros_like(base)
    nd[0] = base[0]
    departure_mass = base[1:] * np.array([1-detect[i] for i in range(1,len(base))])
    if departure_mass.sum() <= 0:
        raise RuntimeError("No nondiagnostic departure mass remains")
    nd[1:] = (1-base[0]) * departure_mass/departure_mass.sum()
    if not (np.isclose(nd.sum(),1.0) and np.isclose(nd[0],base[0])):
        raise AssertionError("Nondiagnostic weights must sum to one and preserve reference mass")
    return {"original": base, "equal_departures": equal_departures, "nondiagnostic": nd}


def reviewer_score(evaluation, weights, alpha=0.8, lam=0.25, maximin=False,
                   target=None):
    target = CFG.primary_target if target is None else target
    sub = evaluation["scenario"]
    sub = sub[(sub["target"] == target) & (sub["scenario_weight"] > 0)].sort_values("scenario_index")
    losses = sub["scenario_MSE"].to_numpy(float)
    valid = sub["valid_fit_fraction"].to_numpy(float)
    if len(losses) != len(weights) or np.any(~np.isfinite(losses)) or np.min(valid) < CFG.minimum_valid_fit_fraction:
        return np.inf
    return float(np.max(losses) if maximin else
                 (1-lam)*np.dot(weights, losses) + lam*weighted_cvar(losses, weights, alpha))


def reviewer_specs():
    specs = [{"name": f"lambda={lam:g};alpha={alpha:g};original", "weights": "original",
              "lambda": lam, "alpha": alpha, "maximin": False}
             for lam in (0.0, 0.25, 1.0) for alpha in (0.50, 0.80, 0.95)]
    specs += [{"name": f"lambda=0.25;alpha=0.80;{w}", "weights": w,
               "lambda": 0.25, "alpha": 0.80, "maximin": False}
              for w in ("equal_departures", "nondiagnostic")]
    specs += [{"name": "maximin;original", "weights": "original",
               "lambda": np.nan, "alpha": np.nan, "maximin": True}]
    return specs


def reviewer_rank(evaluations, specs, weights, stage):
    return pd.DataFrame([{"stage": stage, "design_id": key, "spec": spec["name"],
                          "score": reviewer_score(ev, weights[spec["weights"]],
                                                  spec["alpha"], spec["lambda"], spec["maximin"])}
                         for key, ev in evaluations.items() for spec in specs])


def reviewer_fixed_reoptimization(power, results=RESULTS):
    weights, specs = reviewer_weights(power), reviewer_specs()
    wf = pd.DataFrame({"Scenario": [s["Scenario"] for s in reviewer_primary_scenarios()],
                       "scenario_index": range(len(reviewer_primary_scenarios())), **weights})
    pool = results["optimization"]["candidates"]
    rng = rng_from_seed(MASTER_SEED + 2071)
    indices = rng.choice(len(pool), size=min(len(pool), REVIEW_SETTINGS["fixed_screen_count"]), replace=False)
    designs = {design_id(d): d for d in (row_to_design(pool.iloc[int(i)]) for i in indices)}
    designs.update({design_id(d): d for d in reviewer_selected_designs(results).values()})
    print(f"R1/R3/R5 fixed screen: {len(designs)} designs", flush=True)
    screen = {k: evaluate_rre(d, REVIEW_SETTINGS["fixed_screen_reps"], "review_screen") for k,d in designs.items()}
    sr = reviewer_rank(screen, specs, weights, "screen")
    leaders=set()
    ranked={sp["name"]:sr[sr["spec"]==sp["name"]].sort_values("score")["design_id"].tolist()
            for sp in specs}
    leader_cap=max(6,REVIEW_SETTINGS["fixed_refine_count"]//2)
    for rank in range(6):
        for sp in specs:
            if rank<len(ranked[sp["name"]]):leaders.add(ranked[sp["name"]][rank])
            if len(leaders)>=leader_cap:break
        if len(leaders)>=leader_cap:break
    leaders.update(design_id(d) for d in reviewer_selected_designs(results).values())
    refine_designs={k:designs[k] for k in leaders}
    for key in sorted(leaders):
        for d in local_neighbors(designs[key],rng,count=4 if REVIEW_MODE=="audit" else 8):
            refine_designs[design_id(d)]=d
            if len(refine_designs)>=REVIEW_SETTINGS["fixed_refine_count"]:break
        if len(refine_designs)>=REVIEW_SETTINGS["fixed_refine_count"]:break
    print(f"R1/R3/R5 fixed refine: {len(refine_designs)} designs", flush=True)
    refine = {k: evaluate_rre(d, REVIEW_SETTINGS["fixed_refine_reps"], "review_refine")
              for k,d in refine_designs.items()}
    rr = reviewer_rank(refine, specs, weights, "refine")
    winners = {sp["name"]: str(rr[rr["spec"] == sp["name"]].loc[
        rr[rr["spec"] == sp["name"]]["score"].idxmin(), "design_id"]) for sp in specs}
    confirm_keys = set(winners.values()) | {design_id(d) for d in reviewer_selected_designs(results).values()}
    print(f"R1/R3/R5 independent confirmation: {len(confirm_keys)} designs", flush=True)
    confirm = {k: evaluate_rre(refine_designs[k], REVIEW_SETTINGS["fixed_confirm_reps"], "review_confirm")
               for k in confirm_keys}
    cr = reviewer_rank(confirm, specs, weights, "confirm")
    rows = []
    for sp in specs:
        key = winners[sp["name"]]; d = refine_designs[key]
        rows.append({"spec": sp["name"], "weights": sp["weights"], "lambda": sp["lambda"],
                     "alpha": sp["alpha"], "maximin": sp["maximin"], "design_id": key,
                     "Tmin": d["Tmin"], "N": d["N"], "m": d["m"],
                     "Ni": str(tuple(map(int,d["Ni"]))), "Cost": compute_total_cost(d)["C_total"],
                     "selection_score": float(rr[(rr["spec"] == sp["name"]) & (rr["design_id"] == key)]["score"].iloc[0]),
                     "independent_confirmation_score": float(cr[(cr["spec"] == sp["name"]) & (cr["design_id"] == key)]["score"].iloc[0])})
    selected = pd.DataFrame(rows)
    for name, frame in (("R1_R3_R5_reoptimized_designs", selected),
                        ("R1_R3_scenario_weights", wf),
                        ("R1_R3_R5_screen_scores", sr),
                        ("R1_R3_R5_refine_scores", rr),
                        ("R1_R3_R5_confirm_scores", cr)):
        frame.to_csv(REVIEW_OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    pd.concat([v["scenario"].assign(stage="confirm") for v in confirm.values()]).to_csv(
        REVIEW_OUT / "R1_R3_R5_confirmation_scenario_losses.csv", index=False, encoding="utf-8-sig")
    methods=reviewer_selected_designs(results)
    method_ids={name:design_id(d) for name,d in methods.items()}
    compare=[]
    for sp in specs:
        keys={"newly selected":winners[sp["name"]],**method_ids}
        for target in TARGETS:
            baseline=reviewer_score(confirm[method_ids["V-optimality"]],weights[sp["weights"]],
                                    sp["alpha"],sp["lambda"],sp["maximin"],target)
            for method,key in keys.items():
                score=reviewer_score(confirm[key],weights[sp["weights"]],sp["alpha"],
                                     sp["lambda"],sp["maximin"],target)
                compare.append({"spec":sp["name"],"target":target,"method":method,
                                "design_id":key,"criterion":score,
                                "V_baseline_criterion":baseline,
                                "improvement_vs_V_percent":100*(1-score/baseline) if np.isfinite(score) and np.isfinite(baseline) and baseline>0 else np.nan})
    pd.DataFrame(compare).to_csv(REVIEW_OUT/"R1_R3_R5_confirmed_comparison_vs_V.csv",
                                 index=False,encoding="utf-8-sig")
    return selected, wf, confirm


## 13. Joint-budget design search


In [ ]:
_REVIEW_USE_MEAN_CROSSING_H = ((CFG.YD / math.exp(TRUE_CTX["alpha0"])) **
                               (1.0 / float(TRUE_CTX["q"])))


def reviewer_joint_min_campaign_h(Tmin):
    return float(CFG.observe_ratio * _REVIEW_USE_MEAN_CROSSING_H /
                 selected_time_acceleration_factor(float(Tmin)))


JOINT_DOMAIN = {
    "N_values": tuple(range(30, 81)), "m_values": tuple(range(5, 21)),
    "r_values": (3, 4, 5),
    "Tmin_values_C": tuple(T for T in range(CFG.Tmin_lb, CFG.Tmin_ub + 1)
                            if reviewer_joint_min_campaign_h(T) <= CFG.tmax_allow),
    "interior_grid_C": tuple(range(CFG.Tmin_lb + 1, int(CFG.Tmax))),
    "campaign_fraction_values": (0.0, 0.10, 0.25, 0.50, 0.75, 1.0),
    "minimum_observe_ratio": CFG.observe_ratio,
    "min_units_per_level": CFG.min_units_per_stress,
    "minimum_stress_gap_C": CFG.delta_T_min,
    "operating_rate_rule": "original c_stress_base * 2**stress_rank, extended to r=3..5",
    "budget": CFG.comparison_budget,
    "maximum_campaign_h": CFG.tmax_allow,
}


def reviewer_joint_schedule(design):
    levels = np.asarray(design["levels"], float)
    af = np.asarray(selected_time_acceleration_factor(levels), float)
    duration = float(design["campaign_h"]) * af[0] / af
    schedules = [np.linspace(0, x, int(design["m"])+1) for x in duration]
    return levels, duration, schedules


def reviewer_joint_cost(design):
    levels, duration, _ = reviewer_joint_schedule(design)
    N, m, Ni = int(design["N"]), int(design["m"]), np.asarray(design["Ni"], int)
    if Ni.sum() != N or len(Ni) != len(levels):
        return np.inf
    ranks=np.argsort(np.argsort(levels))
    stress_rates=CFG.c_stress_base*(2.0**ranks)
    return float(CFG.c_sample*N + CFG.c_measure*N*m +
                 np.dot(Ni*stress_rates, duration))


def reviewer_joint_feasible(design):
    levels = np.asarray(design["levels"], float)
    Ni = np.asarray(design["Ni"], int)
    return bool(len(levels) in JOINT_DOMAIN["r_values"] and
                levels[0] in JOINT_DOMAIN["Tmin_values_C"] and levels[-1] == CFG.Tmax and
                np.min(np.diff(levels)) >= JOINT_DOMAIN["minimum_stress_gap_C"] and
                int(design["N"]) in JOINT_DOMAIN["N_values"] and
                int(design["m"]) in JOINT_DOMAIN["m_values"] and
                len(Ni) == len(levels) and Ni.sum() == design["N"] and
                np.min(Ni) >= JOINT_DOMAIN["min_units_per_level"] and
                design["campaign_h"] + 1e-8 >= reviewer_joint_min_campaign_h(levels[0]) and
                design["campaign_h"] <= min(CFG.tmax_allow, JOINT_DOMAIN["maximum_campaign_h"]) and
                reviewer_joint_cost(design) <= CFG.comparison_budget + CFG.budget_tolerance)


def reviewer_joint_id(d):
    return ("J_T" + "-".join(f"{x:g}" for x in d["levels"]) + f"_N{d['N']}_m{d['m']}" +
            "_a" + "-".join(str(int(x)) for x in d["Ni"]) + f"_h{d['campaign_h']:g}")


def reviewer_joint_candidates():
    rng = rng_from_seed(MASTER_SEED + 2111)
    proposals = {}
    for _ in range(REVIEW_SETTINGS["joint_proposals"]):
        Tmin = float(rng.choice(JOINT_DOMAIN["Tmin_values_C"]))
        r = int(rng.choice(JOINT_DOMAIN["r_values"]))
        gap=JOINT_DOMAIN["minimum_stress_gap_C"]
        interior = [x for x in JOINT_DOMAIN["interior_grid_C"] if Tmin+gap <= x <= CFG.Tmax-gap]
        if len(interior) < r-2:
            continue
        mids = sorted(rng.choice(interior, size=r-2, replace=False).tolist()) if r>2 else []
        N = int(rng.choice(JOINT_DOMAIN["N_values"]))
        m = int(rng.choice(JOINT_DOMAIN["m_values"]))
        u = JOINT_DOMAIN["min_units_per_level"]
        Ni = u + rng.multinomial(N-r*u, rng.dirichlet(np.ones(r)))
        minimum_h=reviewer_joint_min_campaign_h(Tmin)
        fraction=float(rng.choice(JOINT_DOMAIN["campaign_fraction_values"]))
        campaign_h=minimum_h+fraction*(CFG.tmax_allow-minimum_h)
        d = {"levels": tuple([Tmin]+mids+[float(CFG.Tmax)]), "N": N, "m": m,
             "Ni": tuple(int(x) for x in Ni),
             "campaign_h": float(campaign_h)}
        if reviewer_joint_feasible(d):
            proposals[reviewer_joint_id(d)] = d
    for d0 in reviewer_selected_designs().values():
        _, levels, duration, _ = planned_schedule(d0)
        d = {"levels": tuple(float(x) for x in levels), "N": d0["N"], "m": d0["m"],
             "Ni": tuple(int(x) for x in d0["Ni"]), "campaign_h": float(duration[0])}
        if reviewer_joint_feasible(d):
            proposals[reviewer_joint_id(d)] = d
    proposals.update(reviewer_joint_resource_candidates())
    return proposals


def reviewer_joint_resource_candidates():
    proposals={}
    reference_levels=tuple(float(x) for x in stress_levels(54))
    alternatives=(reference_levels,(54.0,80.0,float(CFG.Tmax)))
    fractions=np.array([.60,1.0/3.0,1.0/15.0])
    for N in tuple(range(30,81,5))+(69,):
        for m in range(5,21):
            allocation=np.maximum(CFG.min_units_per_stress,np.floor(N*fractions).astype(int))
            while allocation.sum()<N:
                fractional=N*fractions-allocation
                allocation[int(np.argmax(fractional))]+=1
            while allocation.sum()>N:
                candidates=np.where(allocation>CFG.min_units_per_stress)[0]
                allocation[int(candidates[np.argmax(allocation[candidates]-N*fractions[candidates])])]-=1
            for levels in alternatives:
                d={"levels":levels,"N":int(N),"m":int(m),
                   "Ni":tuple(int(x) for x in allocation),
                   "campaign_h":reviewer_joint_min_campaign_h(levels[0])}
                if reviewer_joint_feasible(d):
                    proposals[reviewer_joint_id(d)]=d
    return proposals


def reviewer_joint_simulate(d, scenario, seed):
    levels, _, schedules = reviewer_joint_schedule(d)
    rng = rng_from_seed(seed)
    columns = [[], [], [], []]
    for T, n, times in zip(levels, d["Ni"], schedules):
        for _ in range(int(n)):
            cumulative = 0.0
            for left,right in zip(times[:-1], times[1:]):
                mean,var = scenario_increment_moments(T, left, right, scenario)
                if scenario.get("process") == "gamma":
                    y = rng.gamma(max(mean*mean/var,1e-12), max(var/mean,1e-12))
                else:
                    y = rng.wald(max(mean,1e-12), max(mean**3/var,1e-12))
                for col,val in zip(columns,(T,left,right,max(y,1e-15))):
                    col.append(float(val))
                cumulative += y
                if CFG.censor_at_failure and cumulative >= CFG.YD:
                    break
    return tuple(np.asarray(x,float) for x in columns)


def reviewer_joint_fim(d):
    levels, _, schedules = reviewer_joint_schedule(d)
    a0,a1,eta,q = (TRUE_CTX[k] for k in ("alpha0","alpha1","eta","q"))
    info = np.zeros((4,4),float)
    for T,n,times in zip(levels,d["Ni"],schedules):
        s = float(standardized_stress(T,CFG.T0,CFG.Tmax)); beta=float(beta_model(T,a0,a1))
        left,right=times[:-1],times[1:]
        A=right**q-left**q; dA=power_log_term(right,q)-power_log_term(left,q)
        survival=np.array([1-failure_cdf_ig(x,beta,eta,q,CFG.YD) for x in left])
        w=float(n)*np.maximum(survival,1e-6)
        info[0,0]+=np.sum(w*eta/beta*A);info[0,1]+=np.sum(w*eta*s/beta*A)
        info[0,3]+=np.sum(w*eta/beta*dA);info[1,1]+=np.sum(w*eta*s*s/beta*A)
        info[1,3]+=np.sum(w*eta*s/beta*dA);info[2,2]+=np.sum(w/(2*eta*eta))
        info[2,3]+=np.sum(w*dA/(eta*A))
        info[3,3]+=np.sum(w*dA*dA*(2/A**2+eta/(beta*A)))
    info[1,0]=info[0,1];info[3,0]=info[0,3];info[3,1]=info[1,3];info[3,2]=info[2,3]
    return info


def reviewer_joint_v(d):
    info=reviewer_joint_fim(d)
    if np.linalg.eigvalsh(info).min() <= 0:
        return np.inf
    g=cached_target_gradient(TRUE_CTX, CFG.primary_target)
    return float(g @ np.linalg.solve(info,g))


def reviewer_joint_regression_check():
    baseline=reviewer_selected_designs()["Proposed RRE-optimality"]
    _,levels,duration,_=planned_schedule(baseline)
    d={"levels":tuple(float(x) for x in levels),"N":baseline["N"],"m":baseline["m"],
       "Ni":tuple(int(x) for x in baseline["Ni"]),"campaign_h":float(duration[0])}
    new_levels,new_duration,_=reviewer_joint_schedule(d)
    checks={"stress_levels_equal":bool(np.allclose(levels,new_levels,rtol=0,atol=1e-10)),
            "duration_equal":bool(np.allclose(duration,new_duration,rtol=1e-10,atol=1e-8)),
            "cost_equal":bool(np.isclose(compute_total_cost(baseline)["C_total"],reviewer_joint_cost(d),rtol=1e-10)),
            "FIM_equal":bool(np.allclose(expected_fisher_information(baseline),reviewer_joint_fim(d),rtol=1e-9,atol=1e-9))}
    pd.DataFrame([checks]).to_csv(REVIEW_OUT/"R2_joint_baseline_regression_checks.csv",index=False,encoding="utf-8-sig")
    if not all(checks.values()):
        raise AssertionError(f"Joint extension does not reproduce original design: {checks}")
    return checks


def reviewer_joint_evaluate(d,reps,stage):
    rows=[]
    for i,s in enumerate(reviewer_primary_scenarios()):
        truth=target_truth(s,CFG.primary_target)
        sq=[];valid=0
        for rep in range(reps):
            data=reviewer_joint_simulate(d,s,stable_seed(MASTER_SEED,stage,i,rep))
            fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
            if fit.get("valid",False):
                try:
                    est=target_from_working_theta(fit,CFG.primary_target)
                    if np.isfinite(est):
                        sq.append((est-truth)**2);valid+=1
                except Exception:
                    pass
        rows.append({"scenario_index":i,"Scenario":s["Scenario"],"MSE":np.mean(sq) if sq else np.inf,
                     "valid_fit_fraction":valid/reps})
    return pd.DataFrame(rows)


def reviewer_joint_score(frame,weights):
    if frame["valid_fit_fraction"].min() < CFG.minimum_valid_fit_fraction:
        return np.inf
    loss=frame.sort_values("scenario_index")["MSE"].to_numpy(float)
    return float((1-CFG.robust_cvar_weight)*np.dot(weights,loss)+
                 CFG.robust_cvar_weight*weighted_cvar(loss,weights,CFG.robust_cvar_alpha))


def reviewer_joint_search(weights):
    reviewer_joint_regression_check()
    candidates=reviewer_joint_candidates()
    print(f"R2 feasible engineering-domain proposals: {len(candidates)}", flush=True)
    if not candidates:raise RuntimeError("Engineering joint domain yielded no feasible candidates")
    rows=[]
    for k,d in candidates.items():
        rows.append({"design_id":k,"V":reviewer_joint_v(d),"Cost":reviewer_joint_cost(d)})
    vf=pd.DataFrame(rows).sort_values("V")
    chosen=set(vf.head(3 if REVIEW_MODE=="audit" else 20)["design_id"])
    rng=rng_from_seed(MASTER_SEED+2112)
    chosen.update(rng.choice(list(candidates),size=min(len(candidates),REVIEW_SETTINGS["joint_screen_count"]),replace=False).tolist())
    chosen.update(reviewer_joint_resource_candidates())
    baseline_keys=set()
    for d0 in reviewer_selected_designs().values():
        _,lev,duration,_=planned_schedule(d0)
        bd={"levels":tuple(float(x) for x in lev),"N":d0["N"],"m":d0["m"],
            "Ni":tuple(int(x) for x in d0["Ni"]),"campaign_h":float(duration[0])}
        key=reviewer_joint_id(bd)
        if key not in candidates:
            raise AssertionError(f"Original fixed design absent from joint domain: {key}")
        baseline_keys.add(key)
    chosen.update(baseline_keys)
    screens={k:reviewer_joint_evaluate(candidates[k],REVIEW_SETTINGS["joint_screen_reps"],"joint_screen") for k in chosen}
    print(f"R2 engineering-domain screen: {len(screens)} designs", flush=True)
    sf=pd.DataFrame([{"design_id":k,"score":reviewer_joint_score(v,weights)} for k,v in screens.items()])
    finalists=set(sf.nsmallest(REVIEW_SETTINGS["joint_refine_count"],"score")["design_id"])
    finalists.add(str(vf.iloc[0]["design_id"]))
    finalists.update(baseline_keys)
    refined={k:reviewer_joint_evaluate(candidates[k],REVIEW_SETTINGS["joint_refine_reps"],"joint_refine") for k in finalists}
    print(f"R2 engineering-domain refine: {len(refined)} designs", flush=True)
    rf=pd.DataFrame([{"design_id":k,"score":reviewer_joint_score(v,weights)} for k,v in refined.items()])
    winners={"joint RRE":str(rf.loc[rf["score"].idxmin(),"design_id"]),
             "joint V":str(vf.iloc[0]["design_id"])}
    old=reviewer_selected_designs()
    winners["original fixed RRE"]=next(k for k in baseline_keys if tuple(candidates[k]["Ni"])==tuple(old["Proposed RRE-optimality"]["Ni"]))
    winners["original fixed V"]=next(k for k in baseline_keys if tuple(candidates[k]["Ni"])==tuple(old["V-optimality"]["Ni"]))
    results=[]
    for method,k in winners.items():
        d=candidates[k]
        frame=reviewer_joint_evaluate(d,REVIEW_SETTINGS["joint_validation_reps"],"joint_validation")
        frame.assign(method=method,design_id=k).to_csv(REVIEW_OUT/f"R2_{method.replace(' ','_')}_scenario_validation.csv",index=False,encoding="utf-8-sig")
        results.append({"method":method,"design_id":k,"N":d["N"],"m":d["m"],"r":len(d["levels"]),
                        "levels_C":str(d["levels"]),"Ni":str(d["Ni"]),"campaign_h":d["campaign_h"],
                        "AFt_Tmin":float(selected_time_acceleration_factor(d["levels"][0])),
                        "effective_observe_ratio":float(d["campaign_h"]*selected_time_acceleration_factor(d["levels"][0])/
                          ((CFG.YD/math.exp(TRUE_CTX["alpha0"]))**(1/TRUE_CTX["q"]))),
                        "Cost":reviewer_joint_cost(d),"V":reviewer_joint_v(d),
                        "independent_Phi_RRE":reviewer_joint_score(frame,weights)})
    vf.to_csv(REVIEW_OUT/"R2_joint_candidate_V_scores.csv",index=False,encoding="utf-8-sig")
    sf.to_csv(REVIEW_OUT/"R2_joint_screen_scores.csv",index=False,encoding="utf-8-sig")
    rf.to_csv(REVIEW_OUT/"R2_joint_refine_scores.csv",index=False,encoding="utf-8-sig")
    table=pd.DataFrame(results)
    table.to_csv(REVIEW_OUT/"R2_joint_exploratory_selected.csv",index=False,encoding="utf-8-sig")
    (REVIEW_OUT/"R2_joint_exploratory_domain.json").write_text(json.dumps(JOINT_DOMAIN,indent=2,ensure_ascii=False),encoding="utf-8")
    return table


## 14. Scenario provenance and summaries


In [ ]:
def reviewer_scenario_provenance():
    rows=[]
    reference=reviewer_primary_scenarios()[0]
    for i,s in enumerate(reviewer_primary_scenarios()):
        effects=[]
        for T in (CFG.T0,54.0,85.0,CFG.Tmax):
            effects.append({"T_C":T,
                            "beta_ratio_to_reference":float(scenario_beta(T,s)/scenario_beta(T,reference)),
                            "eta_ratio_to_reference":float(scenario_eta(T,s)/scenario_eta(T,reference)),
                            "q_ratio_to_reference":float(scenario_q(T,s)/scenario_q(T,reference))})
        rows.append({"scenario_index":i,"Scenario":s["Scenario"],"kind":s["kind"],
                     "declared_weight":s["weight"],"parameterization":json.dumps(s,ensure_ascii=False,sort_keys=True),
                     "effects_at_stresses":json.dumps(effects,ensure_ascii=False),
                     "mechanism_evidence":"Mechanism class motivated by accelerated-degradation literature",
                     "severity_evidence":"Declared connector-case sensitivity level; calibrate with product measurements when available",
                     "weight_evidence":"Prespecified planning weight; alternatives tested in R3",
                     "next_required_evidence":"Connector mechanism measurements or elicitation for product-level calibration"})
    frame=pd.DataFrame(rows)
    frame.to_csv(REVIEW_OUT/"R4_uncertainty_set_provenance.csv",index=False,encoding="utf-8-sig")
    procedure={
        "step_1":"Fit and freeze the reference use-condition and acceleration model from source data; record units, censoring, time horizon and data provenance.",
        "step_2":"Enumerate mechanism-specific departures (activation-energy variation, high-temperature mechanism change, stress-dependent time scale, smooth acceleration curvature) and record the source supporting each mechanism class.",
        "step_3":"Choose parameter bounds from connector-specific physics, prior measurements or expert elicitation when available; otherwise declare the sensitivity levels explicitly.",
        "step_4":"Tabulate the implied beta, eta and q effects at use and test stresses, check monotonicity and physical plausibility, and predeclare scenario weights plus at least one alternative weighting.",
        "step_5":"Run design-specific adequacy diagnostics and sensitivity re-optimization; publish the seed, code, scenario table, budget, selected designs and uncertainty intervals.",
        "step_6":"Report the result conditional on the declared product and scenarios; repeat the protocol with mechanism and cost information for another application.",
        "parameter_status":"The ±10% activation-energy and ±0.60 curvature values are declared sensitivity settings; product measurements can replace them."
    }
    (REVIEW_OUT/"R4_reproducible_construction_procedure.json").write_text(
        json.dumps(procedure,ensure_ascii=False,indent=2),encoding="utf-8")
    return frame


def reviewer_write_readout(power,fixed,joint):
    comparison=pd.read_csv(REVIEW_OUT/"R1_R3_R5_confirmed_comparison_vs_V.csv")
    lines=["# Reviewer 1 supplementary results / 审稿人1补充分析结果", "",
           f"Mode / 运行模式: **{REVIEW_MODE}**. Audit mode is a code check only; use paper mode for manuscript numbers. / audit 模式仅用于代码核查，投稿数值须使用 paper 模式。", "",
           "## R1: model diagnostics and hard-to-diagnose weighting / 模型诊断与难诊断情形权重", "",
           "Three group-moment diagnostics are calibrated against Monte Carlo null distributions with Bonferroni nominal familywise level 0.05. Detection is conditional on valid statistics; failures and Wilson 95% intervals are in R1_empirical_diagnostic_power.csv.", "",
           "3项应力组矩统计量由工作模型下的蒙特卡洛零分布校准，并以Bonferroni方法将名义家族误报率控制在0.05；检出率以统计量有效的重复试验为分母，另列失败比例与Wilson 95%区间。", ""]
    selected=power[(power["diagnostic"]=="any") &
                   (power["method"].isin(("Proposed RRE-optimality","V-optimality","Neutral reference")))]
    lines += ["| Scenario / 情形 | RRE detection | V detection | Neutral detection |",
              "|---|---:|---:|---:|"]
    for sc in reviewer_primary_scenarios():
        vals=[]
        for method in ("Proposed RRE-optimality","V-optimality","Neutral reference"):
            v=selected[(selected["Scenario"]==sc["Scenario"]) & (selected["method"]==method)]["detection_probability"]
            vals.append(f"{float(v.iloc[0]):.3f}" if len(v) else "NA")
        lines.append(f"| {sc['Scenario']} | {' | '.join(vals)} |")
    lines += ["", "The reference-model weight remains 0.20. The remaining 0.80 is redistributed among departures in proportion to original weight × (1 − detection probability at the fixed neutral design). The working-model fitting pipeline is held fixed, isolating the effect of diagnostic-based weighting.",
              "正确模型权重保持0.20不变；其余0.80在偏离情形之间按原权重×（1−固定中性方案的检出率）重新分配。工作模型拟合流程保持一致，以单独考察诊断重权的影响。", ""]
    for spec in ("lambda=0.25;alpha=0.8;original","lambda=0.25;alpha=0.80;nondiagnostic","maximin;original"):
        sub=comparison[comparison["spec"]==spec]
        lines += [f"### {spec}", "", "| Target / 目标 | Method / 方法 | Plan / 方案 | Phi or max loss | Improvement vs V / 相对V改善 |",
                  "|---|---|---|---:|---:|"]
        for row in sub.itertuples():
            lines.append(f"| {row.target} | {row.method} | {row.design_id} | {row.criterion:.6g} | {row.improvement_vs_V_percent:.1f}% |")
        lines.append("")
    n_feasible=len(pd.read_csv(REVIEW_OUT/"R2_joint_candidate_V_scores.csv"))
    n_screen=len(pd.read_csv(REVIEW_OUT/"R2_joint_screen_scores.csv"))
    n_refine=len(pd.read_csv(REVIEW_OUT/"R2_joint_refine_scores.csv"))
    lines += ["## R2: engineering-bounded joint search / 工程边界下的共同预算搜索", "",
              "The author's resource range is N = 30–80, m = 5–20, and r = 3–5. The original one-degree stress gap, 0.80 minimum observation ratio, 40,000-hour campaign cap, and 12,000 budget apply. The feasible minimum stress begins at 54°C, calculated from the original observation and time rules. Running costs follow the original rank-based formula at each r.",
              "作者提供资源范围N=30–80、m=5–20、r=3–5；沿用原文最小1°C温差、0.80最低观测比例、40,000 h时长上限和12,000预算。最低可行应力由原观测与时长规则计算，起点54°C。各应力水平的运行费率沿用按排序加倍的公式。", "",
              f"The finite search generated {n_feasible} feasible candidates, screened {n_screen}, and refined {n_refine}; the screened set included the systematic N-by-m resource grid and the original plans.",
              f"有限搜索产生{n_feasible}个可行候选、筛选{n_screen}个、精细评价{n_refine}个；筛选集包含N×m系统网格和原方案。", "",
              "| Method | N | m | r | Stresses (°C) | Allocation | Campaign h | AF_t(min) | Effective rho | Cost | Independent Phi |",
              "|---|---:|---:|---:|---|---|---:|---:|---:|---:|---:|"]
    for row in joint.itertuples():
        lines.append(f"| {row.method} | {row.N} | {row.m} | {row.r} | {row.levels_C} | {row.Ni} | {row.campaign_h:.0f} | {row.AFt_Tmin:.2f} | {row.effective_observe_ratio:.3f} | {row.Cost:.1f} | {row.independent_Phi_RRE:.6g} |")
    lines += ["", "## R3: aggregation and weights / 聚合参数及权重", "",
              "R1_R3_R5_reoptimized_designs.csv lists the reselected plan and independent confirmation score for each lambda, alpha, and weighting specification. The common selected allocation across settings shows design stability; alpha invariance when lambda = 0 follows from the mean-only objective.",
              "R1_R3_R5_reoptimized_designs.csv逐项列出重新选择的方案和独立确认准则值。各设定选出相同分配，说明方案稳定；lambda=0时alpha不影响纯均值目标。", "",
              "| Specification / 设定 | Selected plan / 入选方案 | Independent criterion / 独立准则值 |",
              "|---|---|---:|"]
    for row in fixed.itertuples():
        lines.append(f"| {row.spec} | {row.design_id} | {row.independent_confirmation_score:.6g} |")
    lines += ["",
              "## R4: uncertainty-set construction / 不确定集构建", "",
              "R4_uncertainty_set_provenance.csv and R4_reproducible_construction_procedure.json record each parameterization, its implied changes at four temperatures, and the evidence status of its numerical level. The ten-percent Ea and ±0.60 curvature values are declared sensitivity levels; the six-step protocol specifies how another product can calibrate its own bounds.",
              "R4文件逐项记录参数化、四个温度下的变化及数值水平的证据状态。活化能±10%与曲率±0.60明确作为敏感性水平；六步程序说明其他产品如何校准自身边界。", "",
              "## R5: robust maximin comparator / 稳健maximin对照", "",
              "The maximin search targets the maximum primary-scenario MSE in the same fixed design domain and evaluates the found plan on exactly the same scenarios as RRE and V. Read its independent comparison above and the complete target table in R1_R3_R5_confirmed_comparison_vs_V.csv.",
              "maximin搜索以同一固定设计域内的最大逐情形MSE为目标，入选方案与RRE/V使用完全相同的情形评价。独立比较见上表，所有目标见比较CSV。", ""]
    ci_path=REVIEW_OUT/"R1_paired_bootstrap_improvements.csv"
    if ci_path.exists():
        ci=pd.read_csv(ci_path)
        lines += ["", "## R1 paired Monte Carlo intervals / 配对蒙特卡洛区间", "",
                  "These intervals condition on estimated weights and selected plans; they omit power-estimation and selection uncertainty.",
                  "以下区间条件于已估计的权重和入选方案，未计入检出率估计与方案选择的不确定性。", "",
                  "| Target | Metric | Point | 95% lower | 95% upper |", "|---|---|---:|---:|---:|"]
        for row in ci.itertuples():
            lines.append(f"| {row.target} | {row.metric} | {row.point:.2f} | {row.bootstrap_lo:.2f} | {row.bootstrap_hi:.2f} |")
        lines.append("")
    joint_ci=REVIEW_OUT/"R2_joint_paired_bootstrap.csv"
    if joint_ci.exists():
        ci=pd.read_csv(joint_ci)
        lines += ["## R2 paired Monte Carlo intervals / 联合设计配对区间", "",
                  "These paired intervals quantify simulation variability conditional on the declared engineering domain and selected plans.",
                  "以下配对区间量化在已声明工程设计域和入选方案条件下的模拟波动。", "",
                  "| First | Comparator | Improvement % | 95% lower | 95% upper |", "|---|---|---:|---:|---:|"]
        for row in ci.itertuples():
            lines.append(f"| {row.first} | {row.second} | {row.improvement_percent:.2f} | {row.bootstrap_lo:.2f} | {row.bootstrap_hi:.2f} |")
        lines.append("")
    path=REVIEW_OUT/"R1_R5_bilingual_results_readout.md"
    path.write_text("\n".join(lines),encoding="utf-8")
    return path


## 15. Run diagnostic and budget analyses


In [ ]:
def reviewer_run_addendum():
    began=time.time()
    designs=reviewer_selected_designs()
    designs["Neutral reference"]=canonicalize_design({"Tmin":54,"N":60,"m":12,"Ni":np.array([20,20,20])})
    power,calibration=reviewer_diagnostic_power(designs)
    power.to_csv(REVIEW_OUT/"R1_empirical_diagnostic_power.csv",index=False,encoding="utf-8-sig")
    calibration.to_csv(REVIEW_OUT/"R1_null_calibration.csv",index=False,encoding="utf-8-sig")
    fixed,weights,confirmed=reviewer_fixed_reoptimization(power)
    joint=reviewer_joint_search(reviewer_weights(power)["original"])
    provenance=reviewer_scenario_provenance()
    reviewer_write_readout(power,fixed,joint)
    report={"mode":REVIEW_MODE,"elapsed_seconds":time.time()-began,
            "master_seed":MASTER_SEED,"settings":REVIEW_SETTINGS,
            "joint_domain_note":"Author resource limits N 30..80, m 5..20, r 3..5; original Tmin, gap, observation ratio, campaign cap, budget and ranked cost rule retained",
            "diagnostic_familywise_alpha":0.05,
            "diagnostic_calibration":"Monte Carlo null with exact censoring schedule, Bonferroni across three group-moment checks",
            "nondiagnostic_weight_reference":"Neutral 20/20/20 at 54 C; reference mass stays 0.20 and departure mass 0.80 is redistributed",
            "nondiagnostic_interpretation":"Reweighting sensitivity, not a full detect-then-refit policy risk",
            "selection_scope":"Three-stage stochastic search, not exhaustive global optimum",
            "fixed_domain_candidates_scanned":REVIEW_SETTINGS["fixed_screen_count"],
            "joint_feasible_candidates":len(reviewer_joint_candidates()),
            "result_files":sorted(p.name for p in REVIEW_OUT.iterdir() if p.is_file())}
    (REVIEW_OUT/"reviewer_addendum_manifest.json").write_text(json.dumps(report,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    display(fixed)
    display(joint)
    print(f"Reviewer addendum complete: {REVIEW_OUT.resolve()} ({report['elapsed_seconds']:.1f} s)")
    return {"diagnostic_power":power,"calibration":calibration,"selected_fixed":fixed,
            "weights":weights,"confirmed_fixed":confirmed,"joint":joint,"provenance":provenance,
            "manifest":report}


if os.environ.get("ADT_REVIEWER_AUTORUN","1") == "1":
    REVIEW_RESULTS=reviewer_run_addendum()


## 16. Paired improvement intervals


In [ ]:
def reviewer_paired_bootstrap_ci(confirmed, selected, weight_frame, reps=None, boot=1000):
    method_ids={name:design_id(d) for name,d in reviewer_selected_designs().items()}
    v_id=method_ids['V-optimality']
    names={'original':'lambda=0.25;alpha=0.8;original',
           'nondiagnostic':'lambda=0.25;alpha=0.80;nondiagnostic'}
    winner={label:str(selected[selected.spec==spec].iloc[0].design_id)
            for label,spec in names.items()}
    needed=set(winner.values())|{v_id}
    if not needed.issubset(confirmed):
        raise RuntimeError('Missing confirmed design for paired bootstrap')
    reps=int(reps or REVIEW_SETTINGS['fixed_confirm_reps'])
    rng=rng_from_seed(MASTER_SEED+2481)
    arrays={}
    for key in needed:
        raw=confirmed[key]['raw']
        for target in TARGETS:
            sub=raw[(raw.target==target)&(raw.scenario_index<9)]
            arr=np.full((9,reps),np.nan)
            for i in range(9):
                group=sub[sub.scenario_index==i].sort_values('rep')
                if len(group)!=reps or not np.array_equal(group.rep.to_numpy(int),np.arange(reps)):
                    raise RuntimeError(f'Incomplete confirmation raw array: {key}, {target}, scenario {i}')
                arr[i]=group.squared_error.to_numpy(float)
            if not np.isfinite(arr).all():
                raise RuntimeError('Paired bootstrap requires fully valid confirmation fits; inspect raw fits')
            arrays[key,target]=arr
    weights={name:weight_frame[name].to_numpy(float) for name in names}
    def criterion(loss,w):
        return float(.75*np.dot(w,loss)+.25*weighted_cvar(loss,w,.8))
    rows=[]
    for target in TARGETS:
        point={}; samples={name:[] for name in names}
        for name in names:
            w=weights[name]
            new=arrays[winner[name],target].mean(axis=1)
            v=arrays[v_id,target].mean(axis=1)
            point[name]=100*(1-criterion(new,w)/criterion(v,w))
        for _ in range(boot):
            pick=rng.integers(0,reps,size=(9,reps))
            for name in names:
                w=weights[name]
                new=np.take_along_axis(arrays[winner[name],target],pick,axis=1).mean(axis=1)
                v=np.take_along_axis(arrays[v_id,target],pick,axis=1).mean(axis=1)
                samples[name].append(100*(1-criterion(new,w)/criterion(v,w)))
        for name in names:
            lo,hi=np.percentile(samples[name],[2.5,97.5])
            rows.append({'target':target,'metric':f'{name}_improvement_vs_V_percent',
                         'point':point[name],'bootstrap_lo':lo,'bootstrap_hi':hi,
                         'reps_per_scenario':reps,'bootstrap_reps':boot})
        difference=np.asarray(samples['original'])-np.asarray(samples['nondiagnostic'])
        lo,hi=np.percentile(difference,[2.5,97.5])
        rows.append({'target':target,'metric':'original_minus_nondiagnostic_gain_percentage_points',
                     'point':point['original']-point['nondiagnostic'],
                     'bootstrap_lo':lo,'bootstrap_hi':hi,
                     'reps_per_scenario':reps,'bootstrap_reps':boot})
    frame=pd.DataFrame(rows)
    frame.to_csv(REVIEW_OUT/'R1_paired_bootstrap_improvements.csv',index=False,encoding='utf-8-sig')
    return frame


if 'REVIEW_RESULTS' in globals():
    REVIEW_BOOTSTRAP=reviewer_paired_bootstrap_ci(
        REVIEW_RESULTS['confirmed_fixed'], REVIEW_RESULTS['selected_fixed'],
        REVIEW_RESULTS['weights'], boot=200 if REVIEW_MODE=='audit' else 1000)
    reviewer_write_readout(REVIEW_RESULTS['diagnostic_power'],
                           REVIEW_RESULTS['selected_fixed'], REVIEW_RESULTS['joint'])
    manifest_path=REVIEW_OUT/'reviewer_addendum_manifest.json'
    if manifest_path.exists():
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        manifest['result_files']=sorted(p.name for p in REVIEW_OUT.iterdir() if p.is_file())
        manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    display(REVIEW_BOOTSTRAP)


## 17. Joint-budget comparison intervals


In [ ]:
import ast as review_ast


def reviewer_joint_paired_bootstrap(joint_table, weight_frame, reps=None, boot=1000):
    reps=int(reps or REVIEW_SETTINGS['joint_validation_reps'])
    weights=weight_frame['original'].to_numpy(float)
    designs={}
    labels={}
    for row in joint_table.itertuples():
        d={'levels':tuple(float(x) for x in review_ast.literal_eval(row.levels_C)),
           'N':int(row.N),'m':int(row.m),
           'Ni':tuple(int(x) for x in review_ast.literal_eval(row.Ni)),
           'campaign_h':float(row.campaign_h)}
        key=reviewer_joint_id(d)
        designs[key]=d
        labels[row.method]=key
    arrays={}
    for key,d in designs.items():
        arr=np.full((9,reps),np.nan)
        for i,scenario in enumerate(reviewer_primary_scenarios()):
            truth=target_truth(scenario,CFG.primary_target)
            for rep in range(reps):
                seed=stable_seed(MASTER_SEED,'joint_validation',i,rep)
                data=reviewer_joint_simulate(d,scenario,seed)
                fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
                if fit.get('valid',False):
                    estimate=target_from_working_theta(fit,CFG.primary_target)
                    arr[i,rep]=(estimate-truth)**2
        if np.isfinite(arr).mean(axis=1).min() < CFG.minimum_valid_fit_fraction:
            raise RuntimeError(f'Joint bootstrap fit fraction below threshold: {key}')
        arrays[key]=arr
    def criterion(loss):
        return float((1-CFG.robust_cvar_weight)*np.dot(weights,loss)+
                     CFG.robust_cvar_weight*weighted_cvar(loss,weights,CFG.robust_cvar_alpha))
    points={method:criterion(np.nanmean(arrays[key],axis=1)) for method,key in labels.items()}
    for row in joint_table.itertuples():
        if abs(points[row.method]-float(row.independent_Phi_RRE))>1e-8:
            raise AssertionError(f'Joint result reconstruction mismatch for {row.method}')
    comparisons=[('joint RRE','joint V'),('original fixed RRE','original fixed V'),
                 ('joint RRE','original fixed RRE'),('joint V','original fixed V')]
    rows=[];rng=rng_from_seed(MASTER_SEED+2482)
    samples={pair:[] for pair in comparisons}
    for _ in range(boot):
        pick=rng.integers(0,reps,size=(9,reps))
        criteria={method:criterion(np.nanmean(np.take_along_axis(arrays[key],pick,axis=1),axis=1))
                  for method,key in labels.items()}
        for first,second in comparisons:
            samples[first,second].append(100*(1-criteria[first]/criteria[second]))
    for first,second in comparisons:
        lo,hi=np.percentile(samples[first,second],[2.5,97.5])
        rows.append({'first':first,'second':second,
                     'improvement_percent':100*(1-points[first]/points[second]),
                     'bootstrap_lo':lo,'bootstrap_hi':hi,
                     'reps_per_scenario':reps,'bootstrap_reps':boot,
                     'interpretation':'conditional on scenario weights, candidate search and selected plans'})
    frame=pd.DataFrame(rows)
    frame.to_csv(REVIEW_OUT/'R2_joint_paired_bootstrap.csv',index=False,encoding='utf-8-sig')
    return frame


if 'REVIEW_RESULTS' in globals():
    REVIEW_JOINT_BOOTSTRAP=reviewer_joint_paired_bootstrap(
        REVIEW_RESULTS['joint'],REVIEW_RESULTS['weights'],
        boot=200 if REVIEW_MODE=='audit' else 1000)
    reviewer_write_readout(REVIEW_RESULTS['diagnostic_power'],
                           REVIEW_RESULTS['selected_fixed'], REVIEW_RESULTS['joint'])
    manifest_path=REVIEW_OUT/'reviewer_addendum_manifest.json'
    if manifest_path.exists():
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        manifest['result_files']=sorted(p.name for p in REVIEW_OUT.iterdir() if p.is_file())
        manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    display(REVIEW_JOINT_BOOTSTRAP)


## 18. Data audits, scenario truths and figure labels


In [ ]:
MINOR_OUT = OUT_DIR / 'minor_revision_evidence'
MINOR_OUT.mkdir(parents=True, exist_ok=True)
CATEGORY_DISPLAY = {
    'True model': 'Reference model',
    'Mechanism change': 'High-temperature acceleration behavior',
    'Time scale': 'Stress-dependent time scale',
    'Process family': 'Process-family stress test',
}

def minor_data_and_truth_audit():
    observation_rows, summary_rows, truth_rows = [], [], []
    for temperature, item in sorted(REAL_DATA.items()):
        times = np.asarray(item['times'], float)
        trajectories = np.asarray(item['Y'], float)
        increments = np.diff(np.column_stack([np.zeros(len(trajectories)), trajectories]), axis=1)
        summary_rows.append({'temperature_C': temperature, 'units': len(trajectories),
                             'inspection_count_per_unit': len(times),
                             'inspection_times_h': ';'.join(f'{t:g}' for t in times),
                             'all_increments': increments.size,
                             'retained_positive': int((increments > 0).sum()),
                             'discarded_nonpositive': int((increments <= 0).sum())})
        for unit, trajectory in enumerate(trajectories, 1):
            for j, (t, y) in enumerate(zip(times, trajectory)):
                observation_rows.append({'temperature_C': temperature, 'unit_id': unit,
                                         'time_h': t, 'relaxation_percent': y,
                                         'left_time_h': 0.0 if j == 0 else times[j-1],
                                         'increment_percent': increments[unit-1, j],
                                         'included_in_fit': bool(increments[unit-1, j] > 0)})
    for index, scenario in enumerate(SCENARIOS):
        truth_rows.append({'scenario_index': index, 'Scenario': scenario['Scenario'],
                           'Category': CATEGORY_DISPLAY.get(scenario['Category'], scenario['Category']),
                           'role': scenario.get('role', 'primary'), 'weight': scenario['weight'],
                           'beta_at_T0': float(scenario_beta(CFG.T0, scenario)),
                           'eta_at_T0': float(scenario_eta(CFG.T0, scenario)),
                           'q_at_T0': float(scenario_q(CFG.T0, scenario)),
                           **{'truth_'+name: target_truth(scenario, name) for name in TARGETS}})
    summaries = pd.DataFrame(summary_rows)
    truths = pd.DataFrame(truth_rows)
    assert summaries['retained_positive'].sum() == 186
    assert summaries['discarded_nonpositive'].sum() == 0
    primary = truths[truths['weight'] > 0]
    assert len(primary) == 9
    for name in TARGETS:
        assert np.allclose(primary['truth_'+name], primary['truth_'+name].iloc[0], rtol=0, atol=1e-12)
    summaries.to_csv(MINOR_OUT/'observed_data_schedule_audit.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(observation_rows).to_csv(MINOR_OUT/'connector_observations_and_increments.csv', index=False, encoding='utf-8-sig')
    truths.to_csv(MINOR_OUT/'scenario_truths_all_targets.csv', index=False, encoding='utf-8-sig')
    return summaries, truths

def contrast_color(rgba):
    rgb = np.asarray(rgba[:3], float)
    linear = np.where(rgb <= 0.04045, rgb/12.92, ((rgb+0.055)/1.055)**2.4)
    luminance = float(linear @ np.array([0.2126, 0.7152, 0.0722]))
    return 'black' if (luminance+0.05)/0.05 >= 1.05/(luminance+0.05) else 'white'

def _annotated_matrix(ax, matrix, row_labels, col_labels, cmap='cividis', fmt='.3g',
                      colorbar_label=None):
    values = np.asarray(matrix, float)
    image = ax.imshow(values, aspect='auto', cmap=cmap, norm=_heatmap_norm(values), interpolation='nearest')
    ax.set_xticks(np.arange(len(col_labels)), labels=col_labels)
    ax.set_yticks(np.arange(len(row_labels)), labels=row_labels)
    ax.tick_params(axis='both', labelsize=8, length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_xticks(np.arange(-0.5, len(col_labels), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(row_labels), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.6)
    ax.tick_params(which='minor', bottom=False, left=False)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if np.isfinite(values[i, j]):
                color = contrast_color(image.cmap(image.norm(values[i, j])))
                ax.text(j, i, format(values[i, j], fmt), ha='center', va='center', fontsize=8, color=color)
    bar = ax.figure.colorbar(image, ax=ax, fraction=0.027, pad=0.025)
    if colorbar_label: bar.set_label(colorbar_label, fontsize=8)
    bar.ax.tick_params(labelsize=9)
    return image

def plot_category_validation(scenario_frame):
    frame = scenario_frame.copy()
    frame['Category'] = frame['Category'].replace(CATEGORY_DISPLAY)
    table = frame.groupby(['target', 'method', 'Category'], as_index=False)['MSE_transformed'].mean()
    pivot = table[table['target'] == CFG.primary_target].pivot(index='method', columns='Category', values='MSE_transformed')
    columns = ['Reference model', 'Activation energy', 'High-temperature acceleration behavior',
               'Stress-dependent time scale', 'Acceleration curvature', 'Process-family stress test']
    pivot = pivot.reindex(index=ordered_methods(pivot.index), columns=columns)
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    _annotated_matrix(ax, pivot.to_numpy(float), [method_style(m)['label'] for m in pivot.index],
                      [wrapped_label(c, width=17) for c in pivot.columns], cmap='magma_r', fmt='.2g',
                      colorbar_label='Mean transformed-scale MSE')
    ax.set_xlabel('Uncertainty category', fontsize=9)
    fig.subplots_adjust(left=0.20, bottom=0.28, right=0.93, top=0.98)
    save_figure(fig, 'Fig06_category_validation_revised')

def plot_bias_variance(scenario_frame):
    data = scenario_frame[(scenario_frame['target'] == CFG.primary_target)
                          & scenario_frame['method'].isin(['Proposed RRE-optimality', 'V-optimality'])].copy()
    data['Category'] = data['Category'].replace(CATEGORY_DISPLAY)
    categories = list(dict.fromkeys(data.sort_values('scenario_index')['Category']))
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 4.8), sharey=True)
    y = np.arange(len(categories))
    values = []
    for ax, method in zip(axes, ['Proposed RRE-optimality', 'V-optimality']):
        group = data[data['method'] == method].groupby('Category')
        variance = np.array([group.get_group(c)['Variance_transformed'].mean() for c in categories])
        bias2 = np.array([group.get_group(c)['Bias_transformed'].pow(2).mean() for c in categories])
        values.append((variance, bias2))
        ax.barh(y, variance, height=.62, color=PALETTE['navy'], edgecolor='white', linewidth=.5)
        ax.barh(y, bias2, left=variance, height=.62, color=PALETTE['amber'], edgecolor='white', linewidth=.5, hatch='///')
        ax.set_yticks(y, labels=[wrapped_label(c, width=23) for c in categories])
        ax.set_ylim(len(categories)-.5, -.5)
        ax.tick_params(axis='both', labelsize=8)
        ax.set_xlabel('Transformed-scale loss', fontsize=9)
        ax.set_title(method_style(method)['label'], fontsize=10, pad=12)
        ax.grid(False)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    global_max = max(float(np.max(v+b)) for v,b in values)
    label_x = 1.04 * global_max
    for ax, (variance, bias2) in zip(axes, values):
        ax.set_xlim(0, global_max*1.78)
        for yi, (v,b) in enumerate(zip(variance,bias2)):
            ax.text(label_x, yi, f'V={v:.3g}\nB²={b:.3g}', ha='left', va='center', fontsize=8, linespacing=1.15)
    fig.subplots_adjust(left=.22, right=.99, bottom=.19, top=.93, wspace=.12)
    from matplotlib.transforms import ScaledTranslation
    for ax, letter in zip(axes, ['a', 'b']):
        ax.text(0, 1, letter, transform=ax.transAxes + ScaledTranslation(-12/72, 12/72, fig.dpi_scale_trans),
                weight='bold', fontsize=10, ha='left', va='bottom')
    fig.legend(handles=[Patch(facecolor=PALETTE['navy'], label='Variance'),
                        Patch(facecolor=PALETTE['amber'], edgecolor='white', hatch='///', label='Bias²')],
               loc='lower center', bbox_to_anchor=(.6, -.09), ncol=2, fontsize=9)
    save_figure(fig, 'Fig05_bias_variance_decomposition_revised')

MINOR_STAGE_TIMINGS = {}
def record_stage_wall_time(function, stage):
    def timed(*args, **kwargs):
        began = time.perf_counter()
        result = function(*args, **kwargs)
        MINOR_STAGE_TIMINGS[stage] = time.perf_counter()-began
        (MINOR_OUT/'measured_stage_wall_times.json').write_text(
            json.dumps(MINOR_STAGE_TIMINGS, indent=2), encoding='utf-8')
        return result
    return timed

def minor_reproduction_export():
    summaries, truths = minor_data_and_truth_audit()
    path = OUT_DIR/'Table04_scenario_bias_variance.csv'
    if not path.exists():
        raise FileNotFoundError('Run the original paper cells first so that Table04_scenario_bias_variance.csv exists.')
    source = pd.read_csv(path)
    revised = source.copy()
    revised['Category'] = revised['Category'].replace(CATEGORY_DISPLAY)
    numeric = source.select_dtypes(include=[np.number, 'bool']).columns
    assert source[numeric].equals(revised[numeric])
    revised.to_csv(MINOR_OUT/'figure_source_data_display_labels.csv', index=False, encoding='utf-8-sig')
    plot_category_validation(source)
    plot_bias_variance(source)
    (MINOR_OUT/'minor_revision_numeric_preservation.json').write_text(json.dumps({
        'input_rows':len(source), 'output_rows':len(revised), 'all_numeric_columns_identical':True,
        'observation_count':186, 'discarded_nonpositive':0,
        'categories':'Reference model plus four departure classes plus one zero-weight process-family stress test',
        'figure5_source_file':'Fig06_category_validation_revised.pdf',
        'figure6_source_file':'Fig05_bias_variance_decomposition_revised.pdf'},indent=2),encoding='utf-8')
    return summaries, truths


## 19. Export observation and truth tables


In [ ]:
MINOR_DATA_SUMMARY, MINOR_SCENARIO_TRUTHS = minor_reproduction_export()
display(MINOR_DATA_SUMMARY)
display(MINOR_SCENARIO_TRUTHS)


## 20. Search timing wrapper


In [ ]:
if not getattr(run_three_stage_optimization, '_minor_timed', False):
    run_three_stage_optimization = record_stage_wall_time(run_three_stage_optimization, 'original_design_search_seconds')
    run_three_stage_optimization._minor_timed = True


## 21. Risk and screening audit configuration


In [ ]:
import hashlib as r2_hashlib
import json as r2_json
import time as r2_time
from pathlib import Path as R2Path
from concurrent.futures import ThreadPoolExecutor as R2Executor

R2_OUT = OUT_DIR / 'reviewer2_addendum'
R2_OUT.mkdir(parents=True, exist_ok=True)
R2_SETTINGS = {'audit_reps':96, 'confirmation_reps':400, 'workers':4,
               'new_domain_candidates':64, 'bootstrap_screens':500,
               'bootstrap_confirmation':2000, 'screen_shortlist':8,
               'empirical_near_best_relative_band':0.05,
               'stage_audit':'reviewer2_independent_screen_audit_v1',
               'stage_confirmation':'reviewer2_independent_confirmation_v1'}
R2_BASELINE_PATH = R2Path(os.environ.get('ADT_R2_BASELINE_DIR', str(OUT_DIR)))
R2_PREVIOUS_EVIDENCE = R2Path(os.environ.get('ADT_R2_PREVIOUS_EVIDENCE',str(globals().get('REVIEW_OUT',OUT_DIR/'reviewer_revision_addendum'))))
R2_PRIMARY = [s for s in SCENARIOS if float(s.get('weight',0)) > 0]
R2_WEIGHTS = normalize_weights([s['weight'] for s in R2_PRIMARY])
R2_INPUT_SIGNATURE = r2_hashlib.sha256(r2_json.dumps({
    'scenarios':R2_PRIMARY, 'config':CFG.__dict__, 'truth':TRUE_CTX,
    'fit':'original fit_assumed_ig_with_fallback; original life_q10 transform'},
    sort_keys=True, default=str).encode('utf-8')).hexdigest()


def r2_frame_designs(frame):
    return {design_id(row_to_design(row)):row_to_design(row) for _,row in frame.iterrows()}


def r2_build_audit_domain():
    pool=pd.read_csv(R2_BASELINE_PATH/'optimization_candidate_archive.csv')
    screen=pd.read_csv(R2_BASELINE_PATH/'RRE_screen_archive.csv')
    designs=r2_frame_designs(screen)
    provenance={k:{'original_screen':True,'prior_reselection':False,
                   'deterministic_neighbor':False,'new_domain_sample':False} for k in designs}
    prior=pd.read_csv(R2_PREVIOUS_EVIDENCE/'R1_R3_R5_refine_scores.csv')
    for key in prior.design_id.unique():
        match=pool[pool.design_id==key]
        if len(match):d=row_to_design(match.iloc[0])
        else:
            T,N,m,allocation=key[1:].split('_')
            d=canonicalize_design({'Tmin':int(T),'N':int(N[1:]),'m':int(m[1:]),
                                    'Ni':[int(x) for x in allocation[1:].split('-')]})
        designs[key]=d
        provenance.setdefault(key,{'original_screen':False,'prior_reselection':False,
                                   'deterministic_neighbor':False,'new_domain_sample':False})['prior_reselection']=True
    for Tmin in (54,55,56):
        for allocation in ((36,20,4),(34,22,4)):
            for a in range(-2,3):
                for b in range(-2,3):
                    shift=np.array([a,b,-a-b])
                    if np.abs(shift).sum()>4:continue
                    Ni=np.asarray(allocation)+shift
                    if np.min(Ni)<CFG.min_units_per_stress:continue
                    d=canonicalize_design({'Tmin':Tmin,'N':60,'m':12,'Ni':Ni})
                    if not is_feasible(d):continue
                    key=design_id(d);designs[key]=d
                    provenance.setdefault(key,{'original_screen':False,'prior_reselection':False,
                                               'deterministic_neighbor':False,'new_domain_sample':False})['deterministic_neighbor']=True
    remaining=pool[~pool.design_id.isin(designs)]
    rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer2_declared_domain_sample'))
    indices=rng.choice(len(remaining),size=min(R2_SETTINGS['new_domain_candidates'],len(remaining)),replace=False)
    for i in indices:
        d=row_to_design(remaining.iloc[int(i)]);key=design_id(d);designs[key]=d
        provenance[key]={'original_screen':False,'prior_reselection':False,
                         'deterministic_neighbor':False,'new_domain_sample':True}
    rows=[{'design_id':k,'Tmin':d['Tmin'],'N':d['N'],'m':d['m'],
           'n1':int(d['Ni'][0]),'n2':int(d['Ni'][1]),'n3':int(d['Ni'][2]),
           'Cost':compute_total_cost(d)['C_total'],**provenance[k]} for k,d in sorted(designs.items())]
    domain=pd.DataFrame(rows)
    assert set(screen.design_id).issubset(designs)
    assert len(pool)==37058
    domain.to_csv(R2_OUT/'R2_audited_candidate_domain.csv',index=False,encoding='utf-8-sig')
    return designs,domain


## 22. Candidate audit and loss simulation


In [ ]:
def r2_primary_errors(design,reps,stage):
    did=design_id(design)
    cache=R2_OUT/f'raw_{stage}_{did}_{reps}.npz'
    cache_meta=cache.with_suffix('.json')
    if cache.exists() and cache_meta.exists():
        metadata=r2_json.loads(cache_meta.read_text(encoding='utf-8'))
        if metadata['input_signature']==R2_INPUT_SIGNATURE:
            with np.load(cache) as z:return did,z['squared_errors'],z['valid']
    errors=np.full((len(R2_PRIMARY),reps),np.nan);valid=np.zeros_like(errors,dtype=bool)
    for si,scenario in enumerate(R2_PRIMARY):
        truth=target_truth(scenario,CFG.primary_target)
        for b in range(reps):
            data=simulate_design_data(design,scenario,stable_seed(MASTER_SEED,stage,si,b))
            fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
            if not fit.get('valid',False):continue
            try:
                estimate=target_from_working_theta(fit,CFG.primary_target,CFG)
                if not np.isfinite(estimate):continue
                errors[si,b]=(estimate-truth)**2;valid[si,b]=True
            except (ValueError,RuntimeError,FloatingPointError):pass
    np.savez_compressed(cache,squared_errors=errors,valid=valid)
    cache_meta.write_text(r2_json.dumps({'input_signature':R2_INPUT_SIGNATURE,'stage':stage,
        'replications_per_scenario':reps,'design_id':did,'target':CFG.primary_target},indent=2),encoding='utf-8')
    return did,errors,valid


def r2_evaluate_domain(designs,reps,stage):
    ordered=[d for _,d in sorted(designs.items())]
    results=[]
    with R2Executor(max_workers=R2_SETTINGS['workers']) as executor:
        for number,result in enumerate(executor.map(lambda d:r2_primary_errors(d,reps,stage),ordered),1):
            results.append(result)
            if number%10==0 or number==len(ordered):
                print(f'{stage}: {number}/{len(ordered)} designs evaluated.',flush=True)
    return {did:{'errors':errors,'valid':valid} for did,errors,valid in results}


def r2_loss_matrix(evaluations):
    keys=sorted(evaluations)
    matrix=np.array([np.nanmean(evaluations[k]['errors'],axis=1) for k in keys])
    success=np.array([evaluations[k]['valid'].mean(axis=1) for k in keys])
    eligible=(success.min(axis=1)>=CFG.minimum_valid_fit_fraction)&np.isfinite(matrix).all(axis=1)
    return keys,matrix,success,eligible


def r2_phi(matrix,weights=None,alpha=.8,lam=.25):
    weights=R2_WEIGHTS if weights is None else np.asarray(weights,float)
    return (1-lam)*np.dot(matrix,weights)+lam*np.array([weighted_cvar(x,weights,alpha)for x in matrix])


def r2_tail_accounting(losses,weights,alpha):
    weights=normalize_weights(weights);remaining=1-alpha;rows=[]
    for rank,i in enumerate(np.argsort(-np.asarray(losses)),1):
        take=max(0.,min(weights[i],remaining));remaining=max(0.,remaining-take)
        rows.append({'scenario_index':int(i),'Scenario':R2_PRIMARY[i]['Scenario'],'rank':rank,
                     'loss':float(losses[i]),'weight':weights[i],'included_tail_mass':take,
                     'normalized_tail_weight':take/(1-alpha),'CVaR_contribution':take*losses[i]/(1-alpha)})
    return pd.DataFrame(rows)


## 23. Alternative risk criteria


In [ ]:
def r2_criterion_comparison(keys,matrix,eligible):
    mean=np.dot(matrix,R2_WEIGHTS);cvar=np.array([weighted_cvar(x,R2_WEIGHTS,.8)for x in matrix])
    worst=matrix.max(axis=1)
    oracle=matrix[eligible].min(axis=0)
    regret=(matrix-oracle).max(axis=1)
    pd.DataFrame({'scenario_index':range(len(oracle)),'Scenario':[s['Scenario']for s in R2_PRIMARY],
                  'finite_audit_field_oracle_loss':oracle}).to_csv(R2_OUT/'R2_minimax_regret_oracles.csv',index=False,encoding='utf-8-sig')
    scores={'Mean-CVaR':.75*mean+.25*cvar,'Weighted scenario MSE':mean,
            'Discrete-prior expected loss (same weights)':mean,'Worst-case MSE':worst,'Minimax regret':regret}
    rows=[];leaders={}
    for name,score in scores.items():
        score=score.copy();score[~eligible]=np.inf;i=int(np.argmin(score));leaders[name]=keys[i]
        rows.append({'criterion':name,'design_id':keys[i],'selection_score':score[i],
                     'mean_loss':mean[i],'CVaR_0_8':cvar[i],'worst_loss':worst[i],
                     'max_regret_in_audit_field':regret[i]})
    pd.DataFrame(rows).to_csv(R2_OUT/'R2_risk_criterion_selected_plans.csv',index=False,encoding='utf-8-sig')
    assert np.array_equal(scores['Weighted scenario MSE'],scores['Discrete-prior expected loss (same weights)'])
    return leaders


def r2_risk_preference_illustration():
    weights=np.array([.8,.1,.1]);losses=np.array([[0.,0.,10.],[2.,2.,2.],[.8,.8,3.]])
    mean=losses@weights;cvar=np.array([weighted_cvar(x,weights,.8)for x in losses])
    frame=pd.DataFrame({'illustrative_plan':['A','B','C'],'loss_s0':losses[:,0],
        'loss_s1':losses[:,1],'loss_s2':losses[:,2],'weighted_mean':mean,
        'CVaR_0_8':cvar,'max_loss':losses.max(axis=1),'mean_CVaR_0_25':.75*mean+.25*cvar})
    assert frame.weighted_mean.idxmin()==0 and frame.max_loss.idxmin()==1 and frame.mean_CVaR_0_25.idxmin()==2
    frame.to_csv(R2_OUT/'R2_constructed_risk_preference_illustration.csv',index=False,encoding='utf-8-sig')
    return frame


def r2_parameter_reaggregation(keys,matrix,eligible):
    weights=pd.read_csv(R2_PREVIOUS_EVIDENCE/'R1_R3_scenario_weights.csv').sort_values('scenario_index')
    specs=[('original',lam,alpha)for lam in (0.,.25,1.)for alpha in (.5,.8,.95)]
    specs += [('equal_departures',.25,.8),('nondiagnostic',.25,.8)]
    rows=[]
    for name,lam,alpha in specs:
        score=r2_phi(matrix,weights[name].to_numpy(),alpha,lam);score[~eligible]=np.inf
        i=int(np.argmin(score));rows.append({'weights':name,'lambda':lam,'alpha':alpha,
            'design_id':keys[i],'audit_selection_score':float(score[i]),'audit_reps':R2_SETTINGS['audit_reps']})
    result=pd.DataFrame(rows)
    result.to_csv(R2_OUT/'R2_risk_parameter_reaggregation.csv',index=False,encoding='utf-8-sig')
    return result


## 24. Discrete scenarios and tail accounting


In [ ]:
def r2_atom_and_weight_checks(keys,matrix,eligible):
    reference='T54_N60_m12_a34-22-4'
    assert reference in keys
    tail=[]
    for alpha in (.5,.8,.95):
        tail.append(r2_tail_accounting(matrix[keys.index(reference)],R2_WEIGHTS,alpha).assign(alpha=alpha))
    pd.concat(tail).to_csv(R2_OUT/'R2_tail_mass_accounting.csv',index=False,encoding='utf-8-sig')
    prior_losses=pd.read_csv(R2_PREVIOUS_EVIDENCE/'R1_R3_R5_confirmation_scenario_losses.csv')
    prior_losses=prior_losses[(prior_losses.design_id==reference)&(prior_losses.target==CFG.primary_target)&
                              (prior_losses.scenario_weight>0)].sort_values('scenario_index')
    assert len(prior_losses)==len(R2_PRIMARY)
    pd.concat([r2_tail_accounting(prior_losses.scenario_MSE.to_numpy(),R2_WEIGHTS,a).assign(alpha=a,replications=200)
               for a in (.5,.8,.95)]).to_csv(R2_OUT/'R2_tail_mass_accounting_prior200.csv',index=False,encoding='utf-8-sig')
    specs=[]
    for omitted in range(1,len(R2_PRIMARY)):
        w=R2_WEIGHTS.copy();w[omitted]=0.;w[1:]*=(1-w[0])/w[1:].sum()
        specs.append((f'leave_departure_{omitted}',w,.8,.25,omitted))
    for scale in (.5,.75,1.25,1.5):
        for si in (7,8):
            w=R2_WEIGHTS.copy();w[si]*=scale;w[1:]*=(1-w[0])/w[1:].sum()
            specs.append((f'curvature_atom_{si}_mass_scale_{scale:g}',w,.8,.25,-1))
    rows=[]
    for name,w,alpha,lam,omitted in specs:
        scores=r2_phi(matrix,w,alpha,lam);scores[~eligible]=np.inf;i=int(np.argmin(scores))
        rows.append({'spec':name,'design_id':keys[i],'score':scores[i],
                     'omitted_scenario':omitted,'weight_reference':w[0],
                     'weight_concave':w[7],'weight_convex':w[8],
                     **{f'weight_s{j}':float(value)for j,value in enumerate(w)}})
    pd.DataFrame(rows).to_csv(R2_OUT/'R2_discrete_scenario_reselection.csv',index=False,encoding='utf-8-sig')
    checks=[]
    for alpha in (.5,.8,.95):
        for lam in (0.,.25,1.):
            original=r2_phi(matrix,R2_WEIGHTS,alpha,lam)
            split=np.repeat(matrix,2,axis=1);splitw=np.repeat(R2_WEIGHTS/2,2)
            after=r2_phi(split,splitw,alpha,lam)
            discrepancy=float(np.max(np.abs(original-after)))
            assert discrepancy<1e-12
            checks.append({'alpha':alpha,'lambda':lam,'original_atoms':9,'split_atoms':18,
                           'max_absolute_criterion_difference':discrepancy})
    pd.DataFrame(checks).to_csv(R2_OUT/'R2_identical_atom_splitting_checks.csv',index=False,encoding='utf-8-sig')
    return pd.DataFrame(rows)


## 25. Screening audit and confirmation


In [ ]:
def r2_screening_audit(evaluations,keys,matrix,eligible,domain):
    cube=np.array([evaluations[k]['errors']for k in keys])
    reference=r2_phi(matrix);reference[~eligible]=np.inf
    best=float(reference.min());near=reference<=best*(1+R2_SETTINGS['empirical_near_best_relative_band'])
    bestindex=int(np.argmin(reference))
    rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer2_screen_resampling'))
    rows=[]
    for size in (12,24,48):
        for b in range(R2_SETTINGS['bootstrap_screens']):
            indices=rng.choice(cube.shape[2],size=size,replace=False)
            sampled=np.nanmean(cube[:,:,indices],axis=2)
            scores=r2_phi(sampled);scores[~eligible]=np.inf
            selected=np.argsort(scores)[:R2_SETTINGS['screen_shortlist']]
            rows.append({'screen_reps':size,'resample':b,'best_reference_retained':int(bestindex in selected),
                         'near_best_any_retained':int(near[selected].any()),
                         'near_best_fraction_retained':float(near[selected].sum()/max(1,near.sum())),
                         'shortlist_reference_gap_percent':100*(reference[selected].min()/best-1)})
    pd.DataFrame(rows).to_csv(R2_OUT/'R2_screening_resample_details.csv',index=False,encoding='utf-8-sig')
    summary=pd.DataFrame(rows).groupby('screen_reps',as_index=False).agg(
        resamples=('resample','size'),best_reference_retention=('best_reference_retained','mean'),
        near_best_any_retention=('near_best_any_retained','mean'),
        near_best_fraction_retained=('near_best_fraction_retained','mean'),
        mean_shortlist_reference_gap_percent=('shortlist_reference_gap_percent','mean'))
    summary.to_csv(R2_OUT/'R2_screening_retention_summary.csv',index=False,encoding='utf-8-sig')
    half=[]
    for name,train,test in [('first48_to_last48',slice(0,48),slice(48,96)),('last48_to_first48',slice(48,96),slice(0,48))]:
        scores=r2_phi(np.nanmean(cube[:,:,train],axis=2));scores[~eligible]=np.inf
        i=int(np.argmin(scores));heldout=r2_phi(np.nanmean(cube[:,:,test],axis=2))
        half.append({'split':name,'selected_design':keys[i],'heldout_score':heldout[i],
                     'original36_20_4_score':heldout[keys.index('T54_N60_m12_a36-20-4')],
                     'previous34_22_4_score':heldout[keys.index('T54_N60_m12_a34-22-4')]})
    pd.DataFrame(half).to_csv(R2_OUT/'R2_independent_split_selection.csv',index=False,encoding='utf-8-sig')
    rank=pd.DataFrame({'design_id':keys,'audit_score':reference,'eligible':eligible,
                       'rank':pd.Series(reference).rank(method='min').to_numpy()}).merge(domain,on='design_id')
    rank.to_csv(R2_OUT/'R2_full_audit_rankings.csv',index=False,encoding='utf-8-sig')
    return keys[bestindex]


def r2_confirmation_summary(confirm,leaders,audit_best,domain):
    keys,matrix,success,eligible=r2_loss_matrix(confirm)
    phi=r2_phi(matrix)
    rows=[{'design_id':key,'Phi_RRE':phi[i],'mean_loss':np.dot(matrix[i],R2_WEIGHTS),
           'CVaR_0_8':weighted_cvar(matrix[i],R2_WEIGHTS,.8),'worst_loss':matrix[i].max(),
           'minimum_valid_fit_fraction':success[i].min(),'eligible':eligible[i]}for i,key in enumerate(keys)]
    summary=pd.DataFrame(rows).merge(domain,on='design_id')
    summary.to_csv(R2_OUT/'R2_independent_400_confirmation.csv',index=False,encoding='utf-8-sig')
    raw=[]
    for key in keys:
        for si,s in enumerate(R2_PRIMARY):
            for rep,value in enumerate(confirm[key]['errors'][si]):
                raw.append({'design_id':key,'scenario_index':si,'Scenario':s['Scenario'],
                            'rep':rep,'squared_error':value,'fit_valid':bool(confirm[key]['valid'][si,rep])})
    pd.DataFrame(raw).to_csv(R2_OUT/'R2_independent_confirmation_raw.csv',index=False,encoding='utf-8-sig')
    rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer2_confirmation_bootstrap'))
    boot=[]
    for b in range(R2_SETTINGS['bootstrap_confirmation']):
        ix=np.vstack([rng.integers(0,R2_SETTINGS['confirmation_reps'],R2_SETTINGS['confirmation_reps'])for _ in R2_PRIMARY])
        loss=np.array([np.nanmean(np.take_along_axis(confirm[k]['errors'],ix,axis=1),axis=1)for k in keys])
        scores=r2_phi(loss);boot.append(scores)
    boot=np.asarray(boot);comparisons=[]
    references=['T54_N60_m12_a36-20-4','T54_N60_m12_a34-22-4','T54_N60_m12_a30-3-27']
    for key in sorted(set(leaders.values())|{audit_best}):
        for ref in references:
            if key==ref:continue
            a,c=keys.index(key),keys.index(ref);values=100*(1-boot[:,a]/boot[:,c])
            comparisons.append({'selected_design':key,'reference_design':ref,
                'improvement_percent':100*(1-phi[a]/phi[c]),
                'paired_mc_95_lo':float(np.quantile(values,.025)),
                'paired_mc_95_hi':float(np.quantile(values,.975))})
    pd.DataFrame(comparisons).to_csv(R2_OUT/'R2_paired_400_confirmation_intervals.csv',index=False,encoding='utf-8-sig')
    return summary


## 26. Run risk and screening audits


In [ ]:
def run_reviewer2_addendum():
    start=r2_time.perf_counter()
    r2_risk_preference_illustration()
    designs,domain=r2_build_audit_domain()
    print(f'Reviewer 2 audit: {len(designs)} candidates, {R2_SETTINGS["audit_reps"]} replications, all 184 original screened plans retained.',flush=True)
    ev=r2_evaluate_domain(designs,R2_SETTINGS['audit_reps'],R2_SETTINGS['stage_audit'])
    keys,matrix,success,eligible=r2_loss_matrix(ev)
    lossrows=[{'design_id':key,'scenario_index':si,'Scenario':s['Scenario'],
               'scenario_MSE':matrix[i,si],'valid_fit_fraction':success[i,si]}
              for i,key in enumerate(keys) for si,s in enumerate(R2_PRIMARY)]
    pd.DataFrame(lossrows).to_csv(R2_OUT/'R2_audit_scenario_losses.csv',index=False,encoding='utf-8-sig')
    discrete=r2_atom_and_weight_checks(keys,matrix,eligible)
    leaders=r2_criterion_comparison(keys,matrix,eligible)
    parameter_specs=r2_parameter_reaggregation(keys,matrix,eligible)
    best=r2_screening_audit(ev,keys,matrix,eligible,domain)
    confirmation=set(leaders.values())|set(discrete.design_id)|set(parameter_specs.design_id)|{best,'T54_N60_m12_a36-20-4','T54_N60_m12_a34-22-4','T54_N60_m12_a30-3-27'}
    print(f'Independent confirmation: {len(confirmation)} designs, 400 replications per scenario.',flush=True)
    confirmed=r2_evaluate_domain({k:designs[k]for k in confirmation},R2_SETTINGS['confirmation_reps'],R2_SETTINGS['stage_confirmation'])
    summary=r2_confirmation_summary(confirmed,leaders,best,domain)
    manifest={'settings':R2_SETTINGS,'target':CFG.primary_target,'input_signature':R2_INPUT_SIGNATURE,
              'audit_candidate_count':len(domain),'full_fixed_resource_feasible_domain':37058,
              'original_screen_candidates':int(domain.original_screen.sum()),
              'previously_rejected_original_screen_candidates':int(domain.original_screen.sum())-16,
              'minimum_audit_valid_fit_fraction':float(success.min()),
              'eligible_audit_candidates':int(eligible.sum()),'leaders':leaders,'audit_best':best,
              'confirmation_designs':sorted(confirmation),
              'elapsed_seconds':r2_time.perf_counter()-start,
              'scope':'finite audit field, primary q10 target, fresh audit and independent confirmation streams; no unrestricted global optimality claim',
              'retention_scope':'conditional empirical resampling from the 96-replication audit field, not a population guarantee',
              'regret_scope':'scenario oracles minimized over the identical finite audit field'}
    (R2_OUT/'R2_addendum_manifest.json').write_text(r2_json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print(r2_json.dumps(manifest,ensure_ascii=False,indent=2),flush=True)
    return {'manifest':manifest,'confirmation':summary,'discrete_reselection':discrete}


R2_RESULTS = None
if os.environ.get('ADT_REVIEWER2_AUTORUN','1') == '1':
    R2_RESULTS = run_reviewer2_addendum()


## 27. Risk and screening result tables


In [ ]:
reused = pd.read_csv(R2_PREVIOUS_EVIDENCE/'R1_R3_R5_reoptimized_designs.csv')
display(reused)
for filename in ['R2_risk_criterion_selected_plans.csv','R2_risk_parameter_reaggregation.csv','R2_tail_mass_accounting_prior200.csv','R2_tail_mass_accounting.csv',
                 'R2_discrete_scenario_reselection.csv','R2_screening_retention_summary.csv',
                 'R2_independent_split_selection.csv','R2_independent_400_confirmation.csv',
                 'R2_paired_400_confirmation_intervals.csv']:
    path=R2_OUT/filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path))


## 28. Alternative process specifications


In [ ]:
import json as r3_json
import hashlib as r3_hashlib
import time as r3_time
from pathlib import Path as R3Path
from concurrent.futures import ThreadPoolExecutor as R3Executor
from scipy.integrate import quad as r3_quad
from numpy.polynomial.hermite import hermgauss as r3_hermgauss

R3_OUT=OUT_DIR/'reviewer3_addendum'
R3_OUT.mkdir(parents=True,exist_ok=True)
R3_BASELINE=R3Path(os.environ.get('ADT_R3_BASELINE_DIR',str(R2_BASELINE_PATH)))
R3_R2_RESULTS=R3Path(os.environ.get('ADT_R3_R2_RESULTS',str(R2_OUT)))
R3_SETTINGS={'parameter_reps':200,'gamma_audit_reps':96,'gamma_confirm_reps':400,
             'bootstrap_reps':2000,'workers':4,'gamma_masses':[.05,.10,.20],
             'q_factors':[.85,1.,1.15],'eta_factors':[.5,1.,2.],
             'beta_factors':[.8,1.2],'convolution_ig_share':.5,'gamma_unit_cv':.15}
R3_REFERENCE_CONTEXT=dict(TRUE_CTX)
R3_SIGNATURE=r3_hashlib.sha256(r3_json.dumps({'context':TRUE_CTX,'cfg':CFG.__dict__,
    'settings':R3_SETTINGS,'protocol':'fixed nominal schedules; unchanged IG refit; full positive increments'},
    sort_keys=True,default=str).encode()).hexdigest()


def r3_context(spec):
    ctx=dict(R3_REFERENCE_CONTEXT)
    q0=ctx['q'];anchor=(CFG.YD/np.exp(ctx['alpha0']))**(1/q0)
    ctx['q']=q0*float(spec.get('q_factor',1.))
    ctx['alpha0']+=(q0-ctx['q'])*np.log(anchor)+np.log(spec.get('beta_factor',1.))
    ctx['eta']*=float(spec.get('eta_factor',1.))
    return ctx


def r3_specification_catalog():
    specs=[]
    families=['ig','gamma','ig_gamma_convolution','gamma_unit_effect']
    for family in families:
        for qf in R3_SETTINGS['q_factors']:
            for ef in R3_SETTINGS['eta_factors']:
                specs.append({'family':family,'q_factor':qf,'eta_factor':ef,'beta_factor':1.})
        for bf in R3_SETTINGS['beta_factors']:
            specs.append({'family':family,'q_factor':1.,'eta_factor':1.,'beta_factor':bf})
    for i,s in enumerate(specs):s['spec_id']=f'R3P{i:02d}'
    assert len(specs)==44
    return specs


def r3_increment(rng,mean,variance,family,unit_multiplier=1.):
    if family=='ig':return float(rng.wald(mean,mean**3/variance))
    if family in ['gamma','gamma_unit_effect']:
        return float(rng.gamma(mean**2/variance,variance/mean))*unit_multiplier
    if family=='ig_gamma_convolution':
        a=R3_SETTINGS['convolution_ig_share']
        return float(rng.wald(a*mean,(a*mean)**3/(a*variance))+
                     rng.gamma((1-a)*mean**2/variance,variance/mean))
    raise ValueError(family)


def r3_simulate_fixed_schedule(design,spec,seed):
    d,levels,_,schedules=planned_schedule(design,CFG,R3_REFERENCE_CONTEXT)
    ctx=r3_context(spec);rng=rng_from_seed(seed);rows=[[],[],[],[]]
    logvar=np.log1p(R3_SETTINGS['gamma_unit_cv']**2)
    scenario={'kind':'true_model'}
    for T,units,times in zip(levels,d['Ni'],schedules):
        for _ in range(int(units)):
            effect=float(rng.lognormal(-logvar/2,np.sqrt(logvar)))if spec['family']=='gamma_unit_effect'else 1.
            total=0.
            for left,right in zip(times[:-1],times[1:]):
                mean,variance=scenario_increment_moments(T,left,right,scenario,CFG,ctx)
                increment=r3_increment(rng,mean,variance,spec['family'],effect)
                if not np.isfinite(increment)or increment<=0:raise ValueError('Nonpositive increment; no deletion or censoring is permitted.')
                for column,value in zip(rows,(T,left,right,increment)):column.append(float(value))
                total+=increment
                if CFG.censor_at_failure and total>=CFG.YD:break
    return tuple(np.asarray(column)for column in rows)


def r3_fixed_plans():
    frame=pd.read_csv(R3_R2_RESULTS/'R2_audited_candidate_domain.csv')
    ids=['T54_N60_m12_a36-20-4','T54_N60_m12_a34-22-4','T54_N60_m12_a21-36-3','T54_N60_m12_a30-3-27']
    return {key:row_to_design(frame[frame.design_id==key].iloc[0])for key in ids}


def r3_validate_primary_archive():
    manifest=r3_json.loads((R3_R2_RESULTS/'R2_addendum_manifest.json').read_text(encoding='utf-8'))
    if manifest['input_signature']!=R2_INPUT_SIGNATURE:raise ValueError('Primary archive signature differs from current scenario/context/configuration.')
    base=pd.read_csv(R3_R2_RESULTS/'R2_audit_scenario_losses.csv')
    if len(base)!=339*9 or base[['design_id','scenario_index']].duplicated().any():raise ValueError('Incomplete or duplicate primary audit matrix.')
    return base


## 29. Process truths and simulation


In [ ]:
def r3_failure_probability(t,spec,quadrature_nodes=80):
    ctx=r3_context(spec);beta=np.exp(ctx['alpha0']);eta=ctx['eta'];q=ctx['q'];D=CFG.YD
    if spec['family']=='ig':return failure_cdf_ig(t,beta,eta,q,D)
    if spec['family']=='gamma':return gamma_failure_cdf(t,beta,eta,q,D)
    u=max(t,1e-12)**q;m=beta*u;v=beta**3*u/eta
    if spec['family']=='gamma_unit_effect':
        lv=np.log1p(R3_SETTINGS['gamma_unit_cv']**2)
        tolerance=1e-10 if quadrature_nodes==80 else 1e-12
        density=lambda z:np.exp(-z*z/2)/np.sqrt(2*np.pi)
        integrand=lambda z:gamma_dist.sf(D/np.exp(np.sqrt(lv)*z-lv/2),a=m*m/v,scale=v/m)*density(z)
        probability,error=r3_quad(integrand,-12.,12.,epsabs=tolerance,epsrel=tolerance,limit=200)
        if error>1e-8:raise ValueError('Random-effect mixture quadrature precision insufficient.')
        return float(probability)
    a=R3_SETTINGS['convolution_ig_share'];gamshape=(1-a)*m*m/v;scale=v/m
    igmean=a*m;igshape=(a*m)**3/(a*v)
    distribution=invgauss(mu=igmean/igshape,scale=igshape)
    upper=float(gamma_dist.cdf(D,a=gamshape,scale=scale))
    integrand=lambda z:distribution.cdf(D-gamma_dist.ppf(z,a=gamshape,scale=scale))
    cdf,error=r3_quad(integrand,0.,upper,epsabs=2e-10,epsrel=2e-9,limit=200)
    if error>2e-7:raise ValueError('Convolution quadrature precision insufficient.')
    return float(np.clip(1-cdf,0,1))


def r3_log_truth(spec,quadrature_nodes=80):
    ctx=r3_context(spec);central=(CFG.YD/np.exp(ctx['alpha0']))**(1/ctx['q'])
    lo=central*.02;hi=central*4
    root=lambda t:r3_failure_probability(t,spec,quadrature_nodes)-.1
    while root(lo)>0:lo*=.2
    while root(hi)<0:hi*=2
    return float(np.log(brentq(root,lo,hi,xtol=1e-7,rtol=1e-10)))


def r3_parameter_task(key,spec,reps=None):
    reps=R3_SETTINGS['parameter_reps']if reps is None else reps
    cache=R3_OUT/f'raw_{spec["spec_id"]}_{key}_{reps}.npz';meta=cache.with_suffix('.json')
    if cache.exists()and meta.exists():
        metadata=r3_json.loads(meta.read_text())
        if metadata['signature']==R3_SIGNATURE and (spec['family']!='gamma_unit_effect' or metadata.get('mixture_truth_integration')=='adaptive_normal_quadrature'):return
    truth=r3_log_truth(spec);errors=np.full(reps,np.nan);valid=np.zeros(reps,bool);counts=np.zeros(reps,int)
    design=r3_fixed_plans()[key]
    for b in range(reps):
        data=r3_simulate_fixed_schedule(design,spec,stable_seed(MASTER_SEED,'reviewer3_parameter_confirmation',spec['spec_id'],b))
        counts[b]=len(data[3]);fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
        if fit.get('valid',False):
            estimate=target_from_working_theta(fit,CFG.primary_target,CFG)
            if np.isfinite(estimate):errors[b]=(estimate-truth)**2;valid[b]=True
    np.savez_compressed(cache,squared_errors=errors,valid=valid,n_observed_increments=counts)
    meta.write_text(r3_json.dumps({'signature':R3_SIGNATURE,'spec':spec,'log_lifetime_truth':truth,
        'replications':reps,'valid_fraction':float(valid.mean()),'no_nonpositive_increment_deletion':True,
        'mixture_truth_integration':'adaptive_normal_quadrature'if spec['family']=='gamma_unit_effect'else 'not_applicable'},indent=2),encoding='utf-8')


## 30. Process-parameter comparisons


In [ ]:
def r3_parameter_summary():
    rows=[];paired=[];truths=[];keys=list(r3_fixed_plans());reference=keys[-1]
    for spec in r3_specification_catalog():
        ctx=r3_context(spec);truth=r3_log_truth(spec);truth160=r3_log_truth(spec,160)
        if abs(truth-truth160)>1e-8:raise ValueError('Truth quadrature failed convergence check.')
        truths.append({**spec,**{k:ctx[k]for k in ['alpha0','alpha1','eta','q']},'log_lifetime_truth':truth,
                       'tightened_quadrature_log_difference':abs(truth-truth160)})
        raw={}
        for key in keys:
            cache=R3_OUT/f'raw_{spec["spec_id"]}_{key}_200.npz'
            with np.load(cache)as z:raw[key]=z['squared_errors'].copy();vf=float(z['valid'].mean());obs=float(z['n_observed_increments'].mean())
            finite=raw[key][np.isfinite(raw[key])];se=float(np.std(finite,ddof=1)/np.sqrt(len(finite)))
            rows.append({**spec,'design_id':key,'MSE':float(np.mean(finite)),'MSE_mc_se':se,'eligible':vf>=.95,
                         'valid_fraction':vf,'mean_observed_increments':obs,'log_lifetime_truth':truth})
        rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer3_parameter_bootstrap',spec['spec_id']))
        indices=rng.integers(0,200,size=(R3_SETTINGS['bootstrap_reps'],200))
        for key in keys[:-1]:
            eligible=(np.isfinite(raw[key]).mean()>=.95 and np.isfinite(raw[reference]).mean()>=.95)
            values=100*(1-np.nanmean(raw[key][indices],axis=1)/np.nanmean(raw[reference][indices],axis=1))if eligible else np.array([np.nan])
            paired.append({**spec,'design_id':key,'reference_design':reference,
                'eligible_comparison':eligible,'improvement_percent':100*(1-np.nanmean(raw[key])/np.nanmean(raw[reference]))if eligible else np.nan,
                'paired_mc95_lo':float(np.quantile(values,.025)),'paired_mc95_hi':float(np.quantile(values,.975))})
    pd.DataFrame(rows).to_csv(R3_OUT/'R3_process_parameter_losses.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(paired).to_csv(R3_OUT/'R3_process_parameter_comparisons.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(truths).to_csv(R3_OUT/'R3_process_parameter_truths.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(rows).groupby('family').agg(MSE_min=('MSE','min'),MSE_max=('MSE','max'),minimum_valid_fraction=('valid_fraction','min')).to_csv(R3_OUT/'R3_process_family_summary.csv',encoding='utf-8-sig')
    return pd.DataFrame(rows),pd.DataFrame(paired)


## 31. Positive-Gamma-weight design search


In [ ]:
def r3_gamma_audit_task(key,reps=96,stage='reviewer3_gamma_audit'):
    cache=R3_OUT/f'raw_{stage}_{key}_{reps}.npz';meta=cache.with_suffix('.json')
    if cache.exists()and meta.exists()and r3_json.loads(meta.read_text())['signature']==R3_SIGNATURE:return
    frame=pd.read_csv(R3_R2_RESULTS/'R2_audited_candidate_domain.csv');d=row_to_design(frame[frame.design_id==key].iloc[0])
    scenario=next(s for s in SCENARIOS if s.get('process')=='gamma');truth=target_truth(scenario,CFG.primary_target)
    errors=np.full(reps,np.nan);valid=np.zeros(reps,bool)
    for b in range(reps):
        data=simulate_design_data(d,scenario,stable_seed(MASTER_SEED,stage,'gamma',b))
        fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
        if fit.get('valid',False):
            estimate=target_from_working_theta(fit,CFG.primary_target)
            if np.isfinite(estimate):errors[b]=(estimate-truth)**2;valid[b]=True
    np.savez_compressed(cache,squared_errors=errors,valid=valid)
    meta.write_text(r3_json.dumps({'signature':R3_SIGNATURE,'stage':stage,'replications':reps,
        'truth':truth,'valid_fraction':float(valid.mean())},indent=2),encoding='utf-8')


def r3_gamma_reselection():
    frame=pd.read_csv(R3_R2_RESULTS/'R2_audited_candidate_domain.csv').sort_values('design_id')
    keys=list(frame.design_id);base=r3_validate_primary_archive()
    matrix=base.pivot(index='design_id',columns='scenario_index',values='scenario_MSE').loc[keys].to_numpy()
    gamma_losses=[];eligible=[]
    for key in keys:
        with np.load(R3_OUT/f'raw_reviewer3_gamma_audit_{key}_96.npz')as z:
            gamma_losses.append(float(np.nanmean(z['squared_errors'])));eligible.append(float(z['valid'].mean())>=.95)
    matrix=np.column_stack([matrix,gamma_losses]);rows=[]
    for mass in R3_SETTINGS['gamma_masses']:
        weights=np.r_[(1-mass)*R2_WEIGHTS,mass];scores=r2_phi(matrix,weights,.8,.25)
        primary_eligible=base.groupby('design_id').valid_fit_fraction.min().loc[keys].to_numpy()>=.95
        scores[~(np.asarray(eligible)&primary_eligible)]=np.inf;i=int(np.argmin(scores))
        rows.append({'gamma_mass':mass,'design_id':keys[i],'selection_score':scores[i],
                     **{f'weight_s{k}':float(v)for k,v in enumerate(weights)}})
    pd.DataFrame({'design_id':keys,'gamma_MSE':gamma_losses,'eligible':eligible}).to_csv(R3_OUT/'R3_gamma_audit_losses.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(rows).to_csv(R3_OUT/'R3_positive_gamma_reselection.csv',index=False,encoding='utf-8-sig')
    return pd.DataFrame(rows)


def r3_gamma_confirmation(selected=None):
    selected=r3_gamma_reselection()if selected is None else selected
    keys=sorted(set(selected.design_id)|set(r3_fixed_plans()))
    domain=pd.read_csv(R3_R2_RESULTS/'R2_audited_candidate_domain.csv')
    errors={};valid={}
    for key in keys:
        design=row_to_design(domain[domain.design_id==key].iloc[0])
        prior=R3_R2_RESULTS/f'raw_{R2_SETTINGS["stage_confirmation"]}_{key}_400.npz'
        prior_meta=prior.with_suffix('.json')
        if prior.exists()and prior_meta.exists()and r3_json.loads(prior_meta.read_text())['input_signature']==R2_INPUT_SIGNATURE:
            with np.load(prior)as z:e=z['squared_errors'].copy();v=z['valid'].copy()
        else:_,e,v=r2_primary_errors(design,400,R2_SETTINGS['stage_confirmation'])
        r3_gamma_audit_task(key,400,'reviewer3_gamma_confirmation')
        with np.load(R3_OUT/f'raw_reviewer3_gamma_confirmation_{key}_400.npz')as z:
            errors[key]=np.vstack([e,z['squared_errors']]);valid[key]=np.vstack([v,z['valid']])
    rows=[];comparisons=[]
    rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer3_gamma_confirm_bootstrap'))
    draws=rng.integers(0,400,size=(R3_SETTINGS['bootstrap_reps'],10,400))
    loss={k:np.nanmean(e,axis=1)for k,e in errors.items()}
    bootstrap={k:np.array([np.nanmean(np.take_along_axis(e,ix,axis=1),axis=1)for ix in draws])for k,e in errors.items()}
    for mass in R3_SETTINGS['gamma_masses']:
        weights=np.r_[(1-mass)*R2_WEIGHTS,mass]
        leader=selected[selected.gamma_mass==mass].design_id.iloc[0]
        scores={k:float(r2_phi(loss[k][None,:],weights,.8,.25)[0])for k in keys}
        boot={k:r2_phi(values,weights,.8,.25)for k,values in bootstrap.items()}
        for key in keys:rows.append({'gamma_mass':mass,'design_id':key,'selected_under_mass':key==leader,
            'confirmed_Phi':scores[key],'gamma_MSE':loss[key][-1],'minimum_valid_fraction':float(valid[key].mean(axis=1).min())})
        for reference in ['T54_N60_m12_a36-20-4','T54_N60_m12_a30-3-27']:
            value=100*(1-boot[leader]/boot[reference])
            comparisons.append({'gamma_mass':mass,'design_id':leader,'reference_design':reference,
                'improvement_percent':100*(1-scores[leader]/scores[reference]),
                'paired_mc95_lo':float(np.quantile(value,.025)),'paired_mc95_hi':float(np.quantile(value,.975))})
    pd.DataFrame(rows).to_csv(R3_OUT/'R3_positive_gamma_confirmation.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(comparisons).to_csv(R3_OUT/'R3_positive_gamma_confirmation_intervals.csv',index=False,encoding='utf-8-sig')
    return pd.DataFrame(rows),pd.DataFrame(comparisons)


## 32. Seed and cost checks


In [ ]:
def r3_seed_and_cost_audit():
    rows=[]
    for stage,reps in [('screen',12),('refine',100),('confirmation',250),('independent_validation',500),
                       ('reviewer2_independent_screen_audit_v1',96),('reviewer2_independent_confirmation_v1',400)]:
        for si in range(10):
            for b in range(reps):rows.append({'stage':stage,'scenario_index':si,'replication_index':b,'seed':stable_seed(MASTER_SEED,stage,si,b)})
    seeds=pd.DataFrame(rows);seeds.to_csv(R3_OUT/'R3_seed_index_audit.csv',index=False,encoding='utf-8-sig')
    pool=pd.read_csv(R3_BASELINE/'optimization_candidate_archive.csv');d=r3_fixed_plans()['T54_N60_m12_a36-20-4'];center=compute_total_cost(d)['C_total'];ratio=CFG.matched_cost_tolerance_ratio
    eligible=pool[(pool.Cost-center).abs()<=ratio*center];selected=eligible.nsmallest(1,'V_life_q10')
    selected.assign(relative_cost_difference=(selected.Cost-center)/center).to_csv(R3_OUT/'R3_cost_matched_V_audit.csv',index=False,encoding='utf-8-sig')
    record={'seed_formula':'integer from first8 hex characters of SHA256("|".join(parts))',
            'master_seed':MASTER_SEED,'stage_scenario_replication_keys':len(seeds),'unique_seed_values':int(seeds.seed.nunique()),
            'cost_reference':center,'relative_cost_tolerance':ratio,'lower_cost':center*(1-ratio),'upper_cost':center*(1+ratio),
            'plans_in_cost_band':len(eligible),'selected_cost_matched_V':selected.design_id.iloc[0],
            'algorithm_steps_original_14_16':['screening retention','refinement of retained union','independent confirmation'],
            'algorithm_steps_original_17_18':['eligible confirmed-loss minimization','cost matched V and final validation']}
    (R3_OUT/'R3_seed_cost_protocol.json').write_text(r3_json.dumps(record,indent=2),encoding='utf-8')
    return record


## 33. Run process-family analyses


In [ ]:
def run_reviewer3_addendum():
    start=r3_time.perf_counter();specs=r3_specification_catalog();plans=r3_fixed_plans()
    with R3Executor(max_workers=R3_SETTINGS['workers'])as pool:
        list(pool.map(lambda item:r3_parameter_task(*item),[(key,spec)for spec in specs for key in plans]))
    domain=pd.read_csv(R3_R2_RESULTS/'R2_audited_candidate_domain.csv')
    with R3Executor(max_workers=R3_SETTINGS['workers'])as pool:list(pool.map(r3_gamma_audit_task,domain.design_id))
    losses,paired=r3_parameter_summary();selected=r3_gamma_reselection();confirmed,intervals=r3_gamma_confirmation(selected);protocol=r3_seed_and_cost_audit()
    record={'settings':R3_SETTINGS,'signature':R3_SIGNATURE,'parameter_specifications':44,'fixed_comparison_plans':4,
            'gamma_audited_candidates':339,'selected_positive_gamma_plans':selected.to_dict('records'),'gamma_confirmations':confirmed.to_dict('records'),
            'minimum_parameter_fit_valid_fraction':float(losses.valid_fraction.min()),
            'elapsed_seconds':r3_time.perf_counter()-start,'timing_scope':'current invocation; valid caches may be reused, not necessarily a cold full search','seed_cost_protocol':protocol,
            'parameter_scope':'external fixed-plan validation over four monotone process families; no new parameter-specific design optimization'}
    (R3_OUT/'R3_addendum_manifest.json').write_text(r3_json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
    print(r3_json.dumps(record,ensure_ascii=False,indent=2),flush=True)
    return record


R3_RESULTS=None
if os.environ.get('ADT_REVIEWER3_AUTORUN','1')=='1':
    R3_RESULTS=run_reviewer3_addendum()


## 34. Process-family result tables


In [ ]:
for filename in ['R3_process_parameter_losses.csv','R3_process_parameter_comparisons.csv',
                 'R3_process_parameter_truths.csv','R3_positive_gamma_reselection.csv',
                 'R3_positive_gamma_confirmation.csv','R3_positive_gamma_confirmation_intervals.csv',
                 'R3_cost_matched_V_audit.csv']:
    path=R3_OUT/filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path))


## 35. High-temperature breakpoint configuration


In [ ]:
import hashlib as r4_hashlib
import json as r4_json
import time as r4_time
from pathlib import Path as R4Path
from concurrent.futures import ThreadPoolExecutor as R4Executor

R4_OUT=OUT_DIR/'reviewer4_addendum'
R4_OUT.mkdir(parents=True,exist_ok=True)
R4_R2_RESULTS=R4Path(os.environ.get('ADT_R4_R2_RESULTS',
                                  str(OUT_DIR/'reviewer2_addendum')))
R4_BREAKPOINTS=(75.,85.,95.)
R4_REPS_AUDIT=96
R4_REPS_CONFIRM=400
R4_BOOTSTRAP=2000
R4_WORKERS=4
R4_CATALOG=None
R4_HI_INDICES=(3,4)
R4_PRIMARY_STAGE=R2_SETTINGS['stage_confirmation']
R4_NOMINAL=R2_SETTINGS['stage_audit']
R4_SIGNATURE=r4_hashlib.sha256(r4_json.dumps({
    'config':CFG.__dict__,'truth':TRUE_CTX,'scenarios':R2_PRIMARY,
    'breakpoints':R4_BREAKPOINTS,'target':CFG.primary_target,
    'method':'original finite-sample IG generator and unchanged original IG refit',
    'loss':'conditional MSE of log xi_0.1, 95-percent fit eligibility'},
    sort_keys=True,default=str).encode('utf-8')).hexdigest()


def r4_weight_sets():
    original=R2_WEIGHTS.copy()
    equal=np.r_[.2,np.repeat(.1,8)]
    high=original.copy()
    high[list(R4_HI_INDICES)]=.2
    others=[i for i in range(1,9)if i not in R4_HI_INDICES]
    high[others]=original[others]*(0.4/original[others].sum())
    sets={'original':original,'equal_departures':equal,
          'high_temperature_emphasis':high}
    assert all(np.isclose(w.sum(),1)and np.all(w>=0)for w in sets.values())
    return sets

def r4_portfolio():
    global R4_CATALOG
    if R4_CATALOG is not None:return R4_CATALOG
    domain=pd.read_csv(R4_R2_RESULTS/'R2_audited_candidate_domain.csv')
    losses=pd.read_csv(R4_R2_RESULTS/'R2_audit_scenario_losses.csv')
    scores=losses.pivot(index='design_id',columns='scenario_index',values='scenario_MSE')
    rates=losses.pivot(index='design_id',columns='scenario_index',values='valid_fit_fraction')
    assert scores.shape[1]==9 and len(scores)==len(domain)==339
    rows=domain.copy().sort_values('design_id')
    rows['in_portfolio']=True
    rows.to_csv(R4_OUT/'R4_portfolio.csv',index=False,encoding='utf-8-sig')
    designs={row.design_id:row_to_design(row)for _,row in rows.iterrows()}
    R4_CATALOG=(designs,rows)
    return R4_CATALOG


## 36. Breakpoint simulation and caches


In [ ]:
def r4_original_errors(key,reps):
    stage=R4_NOMINAL if reps==R4_REPS_AUDIT else R4_PRIMARY_STAGE
    source=R4_R2_RESULTS/f'raw_{stage}_{key}_{reps}.npz'
    meta=source.with_suffix('.json')
    if source.is_file()and meta.is_file():
        m=r4_json.loads(meta.read_text(encoding='utf-8'))
        assert m['input_signature']==R2_INPUT_SIGNATURE,(source,'source signature drift')
        with np.load(source)as z:return z['squared_errors'],z['valid']
    assert reps==R4_REPS_CONFIRM,'Every portfolio plan must have an archived 96-replication audit.'
    design=r4_portfolio()[0][key]
    cache=R4_OUT/f'raw_R4_original_confirmation_{key}_{reps}.npz'
    cfg=cache.with_suffix('.json')
    if cache.is_file()and cfg.is_file()and r4_json.loads(cfg.read_text(encoding='utf-8'))['signature']==R4_SIGNATURE:
        with np.load(cache)as z:return z['squared_errors'],z['valid']
    errors=np.full((len(R2_PRIMARY),reps),np.nan)
    valid=np.zeros_like(errors,dtype=bool)
    for si,scenario in enumerate(R2_PRIMARY):
        truth=target_truth(scenario,CFG.primary_target)
        for b in range(reps):
            data=simulate_design_data(design,scenario,stable_seed(MASTER_SEED,stage,si,b))
            fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
            if not fit.get('valid',False):continue
            try:
                estimate=target_from_working_theta(fit,CFG.primary_target,CFG)
                if np.isfinite(estimate):errors[si,b]=(estimate-truth)**2;valid[si,b]=True
            except(ValueError,RuntimeError,FloatingPointError):pass
    np.savez_compressed(cache,squared_errors=errors,valid=valid)
    cfg.write_text(r4_json.dumps({'signature':R4_SIGNATURE,'stage':stage,'reps':reps},indent=2),encoding='utf-8')
    return errors,valid

def r4_changed_errors(key,bp,reps):
    if bp==85.:
        errors,valid=r4_original_errors(key,reps)
        return errors[list(R4_HI_INDICES)],valid[list(R4_HI_INDICES)]
    stem=f'raw_R4_breakpoint_{int(bp)}_{key}_{reps}'
    cache=R4_OUT/(stem+'.npz');meta=R4_OUT/(stem+'.json')
    if cache.is_file()and meta.is_file():
        assert r4_json.loads(meta.read_text(encoding='utf-8'))['signature']==R4_SIGNATURE
        with np.load(cache)as z:return z['squared_errors'],z['valid']
    design=r4_portfolio()[0][key]
    errors=np.full((len(R4_HI_INDICES),reps),np.nan)
    valid=np.zeros_like(errors,dtype=bool)
    stage=R4_NOMINAL if reps==R4_REPS_AUDIT else R4_PRIMARY_STAGE
    for j,si in enumerate(R4_HI_INDICES):
        scenario={**R2_PRIMARY[si],'switch_T':bp}
        truth=target_truth(scenario,CFG.primary_target)
        assert np.isclose(truth,target_truth(R2_PRIMARY[si],CFG.primary_target))
        for b in range(reps):
            data=simulate_design_data(design,scenario,stable_seed(MASTER_SEED,stage,si,b))
            fit=fit_assumed_ig_with_fallback(data,CFG,init=TRUE_RAW)
            if not fit.get('valid',False):continue
            try:
                estimate=target_from_working_theta(fit,CFG.primary_target,CFG)
                if np.isfinite(estimate):errors[j,b]=(estimate-truth)**2;valid[j,b]=True
            except(ValueError,RuntimeError,FloatingPointError):pass
    np.savez_compressed(cache,squared_errors=errors,valid=valid)
    meta.write_text(r4_json.dumps({'signature':R4_SIGNATURE,'breakpoint_C':bp,
       'scenario_indices':list(R4_HI_INDICES),'stage':stage,'reps':reps,
       'truth':'unchanged because breakpoint is above use temperature'},indent=2),encoding='utf-8')
    return errors,valid

def r4_cube(key,bp,reps):
    errors,valid=r4_original_errors(key,reps)
    errors=errors.copy();valid=valid.copy()
    changed,ok=r4_changed_errors(key,bp,reps)
    errors[list(R4_HI_INDICES)]=changed;valid[list(R4_HI_INDICES)]=ok
    return errors,valid

def r4_score(errors,valid,weights):
    losses=np.nanmean(errors,axis=1)
    rates=valid.mean(axis=1)
    eligible=bool(np.all(rates>=CFG.minimum_valid_fit_fraction)and np.all(np.isfinite(losses)))
    phi=float(r2_phi(losses.reshape(1,-1),weights)[0])if eligible else np.nan
    return losses,rates,eligible,phi


## 37. Breakpoint design selection and confirmation


In [ ]:
def r4_run_portfolio():
    start=r4_time.perf_counter();designs,domain=r4_portfolio();weights=r4_weight_sets()
    pd.DataFrame([{'weight_set':name,'scenario_index':i,'scenario':R2_PRIMARY[i]['Scenario'],
                   'weight':float(value)}for name,w in weights.items()for i,value in enumerate(w)]).to_csv(
                   R4_OUT/'R4_weight_vectors.csv',index=False,encoding='utf-8-sig')
    jobs=[(key,bp)for bp in R4_BREAKPOINTS for key in designs]
    def task(pair):
        key,bp=pair;return key,bp,r4_cube(key,bp,R4_REPS_AUDIT)
    with R4Executor(max_workers=R4_WORKERS)as pool:cases=list(pool.map(task,jobs))
    allrows=[];selection=[];selected=set()
    cubes={(key,bp):(errors,valid)for key,bp,(errors,valid)in cases}
    for key,bp,(errors,valid)in cases:
        losses,rates,eligible,_=r4_score(errors,valid,R2_WEIGHTS)
        for si in range(9):allrows.append({'breakpoint_C':bp,'design_id':key,'scenario_index':si,
            'scenario':R2_PRIMARY[si]['Scenario'],'scenario_MSE':losses[si],
            'valid_fit_fraction':rates[si],'eligible':eligible,'replications':R4_REPS_AUDIT})
    for bp in R4_BREAKPOINTS:
        for name,w in weights.items():
            scored=[]
            for key in designs:
                errors,valid=cubes[key,bp];losses,rates,eligible,phi=r4_score(errors,valid,w)
                if eligible:scored.append((phi,key))
            assert scored,(bp,name)
            score,key=min(scored)
            selection.append({'breakpoint_C':bp,'weight_set':name,'design_id':key,
                'selection_Phi':score,'eligible_portfolio':len(scored),'portfolio_size':len(designs),
                'audit_reps':R4_REPS_AUDIT})
            selected.add(key)
    pd.DataFrame(allrows).to_csv(R4_OUT/'R4_breakpoint_scenario_losses.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(selection).to_csv(R4_OUT/'R4_breakpoint_reselection.csv',index=False,encoding='utf-8-sig')
    mandatory={'T54_N60_m12_a21-36-3','T54_N60_m12_a36-20-4',
               'T54_N60_m12_a34-22-4','T54_N60_m12_a30-3-27'}
    selected|=mandatory
    for key in sorted(selected):r4_original_errors(key,R4_REPS_CONFIRM)
    confirmation_jobs=[(key,bp)for bp in R4_BREAKPOINTS for key in sorted(selected)]
    with R4Executor(max_workers=R4_WORKERS)as pool:confirmed=list(pool.map(
        lambda pair:(pair[0],pair[1],r4_cube(pair[0],pair[1],R4_REPS_CONFIRM)),confirmation_jobs))
    confirmations={(key,bp):case for key,bp,case in confirmed}
    crow=[]
    for key,bp,(errors,valid)in confirmed:
        for name,w in weights.items():
            losses,rates,eligible,phi=r4_score(errors,valid,w)
            crow.append({'breakpoint_C':bp,'weight_set':name,'design_id':key,
                'confirmed_Phi':phi,'minimum_fit_fraction':rates.min(),
                'eligible':eligible,'mean_loss':float(losses@w),
                'CVaR_0_8':float(weighted_cvar(losses,w,.8)),
                'confirmation_reps':R4_REPS_CONFIRM})
    pd.DataFrame(crow).to_csv(R4_OUT/'R4_independent_confirmation.csv',index=False,encoding='utf-8-sig')
    rng=rng_from_seed(stable_seed(MASTER_SEED,'reviewer4_breakpoint_paired_bootstrap'))
    pairs=[]
    for item in selection:
        bp=item['breakpoint_C'];name=item['weight_set'];key=item['design_id'];w=weights[name]
        leader=confirmations[key,bp];v=confirmations['T54_N60_m12_a30-3-27',bp]
        score=r4_score(*leader,w)[3];vscore=r4_score(*v,w)[3]
        boot=[]
        if np.isfinite(score)and np.isfinite(vscore):
            for _ in range(R4_BOOTSTRAP):
                indices=np.vstack([rng.integers(0,R4_REPS_CONFIRM,R4_REPS_CONFIRM)for _ in range(9)])
                a=np.nanmean(np.take_along_axis(leader[0],indices,axis=1),axis=1)
                b=np.nanmean(np.take_along_axis(v[0],indices,axis=1),axis=1)
                boot.append(100*(1-r2_phi(a.reshape(1,-1),w)[0]/r2_phi(b.reshape(1,-1),w)[0]))
            lo,hi=np.quantile(boot,[.025,.975])
        else:lo=hi=np.nan
        pairs.append({'breakpoint_C':bp,'weight_set':name,'selected_design':key,
                      'V_design':'T54_N60_m12_a30-3-27','selected_confirmed_Phi':score,
                      'V_confirmed_Phi':vscore,'improvement_vs_V_percent':100*(1-score/vscore),
                      'paired_mc_95_lo':lo,'paired_mc_95_hi':hi,
                      'selected_minimum_fit_fraction':leader[1].mean(axis=1).min(),
                      'V_minimum_fit_fraction':v[1].mean(axis=1).min()})
    pd.DataFrame(pairs).to_csv(R4_OUT/'R4_selected_vs_V_confirmed.csv',index=False,encoding='utf-8-sig')
    result={'breakpoints_C':list(R4_BREAKPOINTS),'original_breakpoint_C':85,
       'portfolio_size':len(designs),'positive_scenarios':9,'weight_sets':list(weights),
       'audit_replications_per_scenario':R4_REPS_AUDIT,'confirmation_replications_per_scenario':R4_REPS_CONFIRM,
       'reference_use_stress_C':CFG.T0,'high_scenario_indices':list(R4_HI_INDICES),
       'definition_signature':R4_SIGNATURE,'selected_designs':selection,
       'confirmed_designs':sorted(selected),'elapsed_seconds_this_invocation':r4_time.perf_counter()-start,
       'timing_scope':'Actual invocation; previously archived primary scenario caches reused by signature.'}
    (R4_OUT/'R4_addendum_manifest.json').write_text(r4_json.dumps(result,indent=2),encoding='utf-8')
    return result


## 38. Run breakpoint analyses


In [ ]:
R4_RESULTS=None
if os.environ.get('ADT_REVIEWER4_AUTORUN','1')=='1':
    R4_RESULTS=r4_run_portfolio()


## 39. Breakpoint result tables


In [ ]:
for name in ['R4_weight_vectors.csv','R4_portfolio.csv','R4_breakpoint_reselection.csv',
             'R4_breakpoint_scenario_losses.csv','R4_independent_confirmation.csv',
             'R4_selected_vs_V_confirmed.csv']:
    path=R4_OUT/name
    if path.is_file():
        print(name)
        display(pd.read_csv(path))
